# Notebook 07: Model Dataset Preparation

## Objective

Prepare the engineered COPD feature dataset and validated seven-day exacerbation labels for supervised machine-learning development.

This notebook will:

1. Load the final engineered feature dataset.
2. Load the validated seven-day exacerbation label dataset.
3. Inspect and validate both datasets.
4. Combine features and labels using `Participant_ID` and `Date`.
5. Separate known-target observations from unknown-target observations.
6. Exclude identifiers and future-event audit information from model predictors.
7. Save the resulting modelling and audit datasets.

## Input datasets

| Dataset | Location |
|---|---|
| Engineered features | `data/processed/features/final_feature_dataset.parquet` |
| Seven-day labels | `data/processed/labels/exacerbation_labels_7d.parquet` |

## Target variable

`exacerbation_within_7d`

- `1`: A documented moderate or severe exacerbation occurred within the next seven days.
- `0`: No documented event occurred within the next seven days, with a complete operational follow-up window.
- `NaN`: The target is unknown because event-file information or complete follow-up is unavailable.

## Important modelling considerations

- Join datasets using `Participant_ID` and `Date`.
- Exclude unknown targets from supervised model training.
- Retain unknown observations separately for auditing.
- Exclude matched event dates, event severity, participant identifiers, and label-status information from model predictors.
- Use participant-level data splitting to reduce patient-level leakage.
- Do not use future observations when constructing predictors.

**Research limitation:** The dataset contains 74 participants and 39 documented exacerbation events. Positive feature rows are not independent events, and the dataset is intended for research prototyping rather than clinical diagnosis.

In [ ]:

from pathlib import Path

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Define project directory
# --------------------------------------------------

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# --------------------------------------------------
# 2. Define input paths
# --------------------------------------------------

FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "final_feature_dataset.parquet"
)

LABELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "labels"
    / "exacerbation_labels_7d.parquet"
)

# --------------------------------------------------
# 3. Check file existence
# --------------------------------------------------

print("INPUT FILE CHECK")
print("=" * 60)

print("Project root:", PROJECT_ROOT.resolve())
print("Feature file exists:", FEATURES_PATH.exists())
print("Label file exists:", LABELS_PATH.exists())

assert FEATURES_PATH.exists(), "Feature dataset not found."
assert LABELS_PATH.exists(), "Label dataset not found."


In [ ]:
from pathlib import Path
import os

print("CURRENT WORKING DIRECTORY:")
print(os.getcwd())

print("\nFEATURES_PATH:")
print(FEATURES_PATH)
print("Exists:", Path(FEATURES_PATH).exists())

print("\nLABELS_PATH:")
print(LABELS_PATH)
print("Exists:", Path(LABELS_PATH).exists())

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

files_to_check = {
    "FEATURES": FEATURES_PATH,
    "LABELS": LABELS_PATH
}

for name, file_path in files_to_check.items():

    print(f"\n{'=' * 60}")
    print(f"CHECKING: {name}")
    print("=" * 60)

    path = Path(file_path)

    print("File exists:", path.exists())
    print("File size (MB):", round(path.stat().st_size / (1024 ** 2), 2))

    try:
        parquet_file = pq.ParquetFile(file_path)

        print("Metadata readable: YES")
        print("Number of rows:", parquet_file.metadata.num_rows)
        print("Number of columns:", parquet_file.metadata.num_columns)
        print("Number of row groups:", parquet_file.metadata.num_row_groups)

        test_data = parquet_file.read_row_group(0)

        print("First row group readable: YES")

    except Exception as e:
        print("PARQUET CHECK FAILED")
        print("Error type:", type(e).__name__)
        print("Error message:", str(e))
        

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import importlib.util

print("LIBRARY VERSIONS")
print("=" * 50)

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

print("\nALTERNATIVE PARQUET ENGINES")
print("=" * 50)

print("fastparquet installed:",
      importlib.util.find_spec("fastparquet") is not None)

print("duckdb installed:",
      importlib.util.find_spec("duckdb") is not None)

In [ ]:
# %pip install duckdb

In [ ]:
import duckdb

print("DuckDB version:", duckdb.__version__)

for name, path in {
    "FEATURES": FEATURES_PATH,
    "LABELS": LABELS_PATH
}.items():

    print("\n" + "=" * 60)
    print("CHECKING:", name)
    print("=" * 60)

    try:
        result = duckdb.execute(
            "SELECT COUNT(*) FROM read_parquet(?)",
            [str(path)]
        ).fetchone()

        print("Read successful!")
        print("Rows:", result[0])

    except Exception as e:
        print("Read failed!")
        print(type(e).__name__, str(e))

In [ ]:
# --------------------------------------------------
# Load feature and label datasets using DuckDB
# --------------------------------------------------

import duckdb

features = duckdb.execute(
    "SELECT * FROM read_parquet(?)",
    [str(FEATURES_PATH)]
).fetchdf()

labels = duckdb.execute(
    "SELECT * FROM read_parquet(?)",
    [str(LABELS_PATH)]
).fetchdf()

print("DATASETS LOADED")
print("=" * 50)

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

print("\nUnique participants in features:",
      features["Participant_ID"].nunique())

print("Unique participants in labels:",
      labels["Participant_ID"].nunique())

print("\nFeature date range:")
print(features["Date"].min(), "to", features["Date"].max())

In [ ]:
# Verify recovered feature and label datasets

print("FEATURE DATASET")
print("=" * 50)

print("Duplicate participant-date keys:",
      features.duplicated(["Participant_ID", "Date"]).sum())

print("\nLABEL DATASET")
print("=" * 50)

print("Duplicate participant-date keys:",
      labels.duplicated(["Participant_ID", "Date"]).sum())

print("\nTARGET DISTRIBUTION")
print("=" * 50)

print(labels["exacerbation_within_7d"].value_counts(dropna=False))

print("\nMERGE KEY VALIDATION")
print("=" * 50)

feature_keys = set(zip(features["Participant_ID"], features["Date"]))
label_keys = set(zip(labels["Participant_ID"], labels["Date"]))

print("Feature keys:", len(feature_keys))
print("Label keys:", len(label_keys))
print("Matching keys:", len(feature_keys & label_keys))
print("Feature-only keys:", len(feature_keys - label_keys))
print("Label-only keys:", len(label_keys - feature_keys))

### Step 2: Participant-Date Alignment Validation

**Objective**

Verify that the engineered feature dataset and the seven-day exacerbation label dataset contain exactly matching participant-date observations.

**Validation checks**

1. Confirm that both datasets contain the required key columns: `Participant_ID` and `Date`.
2. Confirm that participant identifiers and dates have compatible data types.
3. Check for duplicate participant-date combinations in each dataset.
4. Check whether any feature records are missing corresponding labels.
5. Check whether any label records are missing corresponding features.
6. Confirm that the two datasets contain exactly the same participant-date keys.

**Why this matters**

The feature and target must refer to the same participant on the same observation date. Incorrect alignment could associate a participant's sensor measurements with another date's exacerbation outcome, producing invalid training data.

**Expected result**

Both datasets should have zero duplicate keys, zero unmatched records, and exactly matching participant-date combinations.

In [ ]:


# --------------------------------------------------
# 1. Check required key columns
# --------------------------------------------------

KEY_COLUMNS = ["Participant_ID", "Date"]

print("KEY COLUMN VALIDATION")
print("=" * 60)

for column in KEY_COLUMNS:
    print(
        f"{column}:",
        column in features.columns,
        column in labels.columns
    )

# --------------------------------------------------
# 2. Standardize key data types
# --------------------------------------------------

features["Participant_ID"] = (
    features["Participant_ID"].astype("string")
)

labels["Participant_ID"] = (
    labels["Participant_ID"].astype("string")
)

features["Date"] = pd.to_datetime(
    features["Date"], utc=True
)

labels["Date"] = pd.to_datetime(
    labels["Date"], utc=True
)

# --------------------------------------------------
# 3. Check duplicate participant-date keys
# --------------------------------------------------

feature_duplicates = features.duplicated(
    KEY_COLUMNS
).sum()

label_duplicates = labels.duplicated(
    KEY_COLUMNS
).sum()

print("\nDUPLICATE KEY VALIDATION")
print("=" * 60)

print("Feature duplicate keys:", feature_duplicates)
print("Label duplicate keys:", label_duplicates)

# --------------------------------------------------
# 4. Compare participant-date combinations
# --------------------------------------------------

feature_keys = features[KEY_COLUMNS].drop_duplicates()
label_keys = labels[KEY_COLUMNS].drop_duplicates()

comparison = feature_keys.merge(
    label_keys,
    on=KEY_COLUMNS,
    how="outer",
    indicator=True
)

print("\nPARTICIPANT-DATE ALIGNMENT")
print("=" * 60)

print(
    comparison["_merge"].value_counts()
)

print(
    "Feature records without labels:",
    (comparison["_merge"] == "left_only").sum()
)

print(
    "Label records without features:",
    (comparison["_merge"] == "right_only").sum()
)

print(
    "Matching participant-date keys:",
    (comparison["_merge"] == "both").sum()
)

# --------------------------------------------------
# 5. Participant-level comparison
# --------------------------------------------------

feature_participants = set(features["Participant_ID"].dropna())
label_participants = set(labels["Participant_ID"].dropna())

print("\nPARTICIPANT VALIDATION")
print("=" * 60)

print("Participants in features:", len(feature_participants))
print("Participants in labels:", len(label_participants))

print(
    "Participants only in features:",
    feature_participants - label_participants
)

print(
    "Participants only in labels:",
    label_participants - feature_participants
)

# --------------------------------------------------
# 6. Overall result
# --------------------------------------------------

print("\nOVERALL ALIGNMENT RESULT")
print("=" * 60)

if (
    feature_duplicates == 0
    and label_duplicates == 0
    and (comparison["_merge"] != "both").sum() == 0
    and feature_participants == label_participants
):
    print("PASS: Feature and label datasets align perfectly.")
else:
    print("REVIEW REQUIRED: Alignment issues detected.")


### Step 3: Merging Engineered Features and Exacerbation Labels

**Objective**

Combine the engineered daily sensor feature dataset with the validated seven-day exacerbation label dataset.

**Join keys**

- `Participant_ID`
- `Date`

**Join strategy**

A one-to-one inner join is used because the preceding validation established that both datasets contain exactly the same 13,085 participant-date combinations, without duplicate keys or unmatched records.

**Expected output**

| Dataset | Columns |
|---|---:|
| Feature dataset | 1,918 |
| Label dataset | 6 |
| Expected merged dataset | 1,922 |

The merged dataset will retain positive, negative, and unknown target records.

**Important**

The event date, event severity, and label status are retained temporarily for auditing. They must not be included as model predictors because they contain target-related or future-event information.

No feature imputation, scaling, or row removal is performed during this step.

In [ ]:

# --------------------------------------------------
# 1. Merge features and labels
# --------------------------------------------------

model_dataset = features.merge(
    labels,
    on=["Participant_ID", "Date"],
    how="inner",
    validate="one_to_one"
)

# --------------------------------------------------
# 2. Inspect merged dataset
# --------------------------------------------------

print("MERGE RESULTS")
print("=" * 60)

print("Feature rows:", len(features))
print("Label rows:", len(labels))
print("Merged rows:", len(model_dataset))

print("Feature columns:", len(features.columns))
print("Label columns:", len(labels.columns))
print("Merged columns:", len(model_dataset.columns))

# --------------------------------------------------
# 3. Check target distribution
# --------------------------------------------------

print("\nTARGET DISTRIBUTION")
print("=" * 60)

print(
    model_dataset["exacerbation_within_7d"]
    .value_counts(dropna=False)
)

# --------------------------------------------------
# 4. Check label status
# --------------------------------------------------

print("\nLABEL STATUS DISTRIBUTION")
print("=" * 60)

print(
    model_dataset["label_status"]
    .value_counts(dropna=False)
)

# --------------------------------------------------
# 5. Verify merge integrity
# --------------------------------------------------

print("\nMERGE INTEGRITY")
print("=" * 60)

print(
    "Duplicate participant-date rows:",
    model_dataset.duplicated(
        ["Participant_ID", "Date"]
    ).sum()
)

print(
    "Missing participant IDs:",
    model_dataset["Participant_ID"].isna().sum()
)

print(
    "Missing dates:",
    model_dataset["Date"].isna().sum()
)

assert len(model_dataset) == len(features)
assert len(model_dataset) == len(labels)
assert not model_dataset.duplicated(
    ["Participant_ID", "Date"]
).any()

print("\nPASS: Feature and label datasets merged successfully.")


### Step 4: Separating Known and Unknown Target Observations

**Objective**

Separate the merged dataset into observations with known seven-day exacerbation outcomes and observations whose outcomes cannot be reliably classified.

**Known-target observations**

Rows where `exacerbation_within_7d` is either:

- `1`: A documented moderate or severe exacerbation occurred within seven days.
- `0`: No documented event occurred within seven days, and the full operational follow-up window was available.

These observations will form the initial supervised modelling dataset.

**Unknown-target observations**

Rows where `exacerbation_within_7d` is missing (`NaN`).

These include:

- `unknown_no_event_file`: The participant has no available exacerbation event file.
- `unknown_incomplete_followup`: The participant's seven-day follow-up window extends beyond the last available feature date, with no known positive event.

Unknown observations will be retained in a separate audit dataset.

**Expected results**

| Dataset | Expected records |
|---|---:|
| Known-target observations | 4,387 |
| Unknown-target observations | 8,698 |
| Total | 13,085 |

**Important limitation**

The known negative class represents no documented event in the available event file, not proof of clinical absence of exacerbation. Event ascertainment completeness remains a dataset limitation.

In [ ]:

# --------------------------------------------------
# 1. Separate known and unknown targets
# --------------------------------------------------

known_target_dataset = model_dataset[
    model_dataset["exacerbation_within_7d"].notna()
].copy()

unknown_target_dataset = model_dataset[
    model_dataset["exacerbation_within_7d"].isna()
].copy()

# --------------------------------------------------
# 2. Inspect dataset sizes
# --------------------------------------------------

print("DATASET SEPARATION")
print("=" * 60)

print("Original merged rows:", len(model_dataset))
print("Known-target rows:", len(known_target_dataset))
print("Unknown-target rows:", len(unknown_target_dataset))

# --------------------------------------------------
# 3. Check known-target distribution
# --------------------------------------------------

print("\nKNOWN-TARGET DISTRIBUTION")
print("=" * 60)

print(
    known_target_dataset["exacerbation_within_7d"]
    .value_counts()
    .sort_index()
)

# --------------------------------------------------
# 4. Check unknown-target reasons
# --------------------------------------------------

print("\nUNKNOWN-TARGET REASONS")
print("=" * 60)

print(
    unknown_target_dataset["label_status"]
    .value_counts()
)

# --------------------------------------------------
# 5. Validate separation
# --------------------------------------------------

print("\nSEPARATION VALIDATION")
print("=" * 60)

print(
    "Missing targets in known dataset:",
    known_target_dataset["exacerbation_within_7d"].isna().sum()
)

print(
    "Non-missing targets in unknown dataset:",
    unknown_target_dataset["exacerbation_within_7d"].notna().sum()
)

print(
    "Total records after separation:",
    len(known_target_dataset) + len(unknown_target_dataset)
)

assert len(known_target_dataset) == 4387
assert len(unknown_target_dataset) == 8698

assert known_target_dataset[
    "exacerbation_within_7d"
].notna().all()

assert unknown_target_dataset[
    "exacerbation_within_7d"
].isna().all()

assert (
    len(known_target_dataset) + len(unknown_target_dataset)
    == len(model_dataset)
)

print("\nPASS: Known and unknown target datasets separated correctly.")


### Step 5: Feature Leakage Audit

**Objective**

Identify columns that must not be used as model predictors and investigate potential temporal leakage in the engineered feature dataset.

**Known non-predictor columns**

- `Participant_ID`: Retained for participant grouping and patient-level data splitting.
- `Date`: Retained for chronological analysis and temporal validation.
- `exacerbation_within_7d`: The prediction target.
- `label_status`: Label-generation audit information.
- `matched_event_date`: Future event information.
- `matched_event_severity`: Future event information.

**Temporal leakage investigation**

The engineered feature dataset contains daily sensor summaries, rolling-window features, historical personal baselines, data-quality features, and cross-sensor features.

These must be reviewed to establish that they use only information available at the prediction date.

Particular attention will be given to:

- Rolling-window features and their time direction.
- Historical personal baselines.
- Cross-sensor calculations.
- Any feature containing future, event, outcome, or target-related information.

**Important distinction**

A keyword-based column scan identifies suspicious names but cannot prove that a feature is leakage-free. Feature definitions and the feature-engineering implementation must also be inspected.

No final model predictor list will be approved until this audit is complete.

In [ ]:

# --------------------------------------------------
# 1. Define columns that must never be predictors
# --------------------------------------------------

NON_PREDICTOR_COLUMNS = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

print("NON-PREDICTOR COLUMN CHECK")
print("=" * 60)

for column in NON_PREDICTOR_COLUMNS:
    print(
        f"{column}:",
        column in known_target_dataset.columns
    )

# --------------------------------------------------
# 2. Identify candidate feature columns
# --------------------------------------------------

candidate_feature_columns = [
    column
    for column in known_target_dataset.columns
    if column not in NON_PREDICTOR_COLUMNS
]

print("\nCOLUMN COUNTS")
print("=" * 60)

print("Total merged columns:", len(model_dataset.columns))
print("Known-target columns:", len(known_target_dataset.columns))
print("Excluded non-predictor columns:", len(NON_PREDICTOR_COLUMNS))
print("Candidate feature columns:", len(candidate_feature_columns))

# --------------------------------------------------
# 3. Search for suspicious column names
# --------------------------------------------------

suspicious_keywords = [
    "future",
    "lead",
    "next_",
    "next7",
    "next_7",
    "outcome",
    "target",
    "exacerbation",
    "event_date",
    "event_severity",
    "severity",
    "label"
]

suspicious_columns = [
    column
    for column in candidate_feature_columns
    if any(
        keyword in column.lower()
        for keyword in suspicious_keywords
    )
]

print("\nSUSPICIOUS FEATURE NAMES")
print("=" * 60)

if suspicious_columns:
    for column in suspicious_columns:
        print(column)
else:
    print("No suspicious feature names detected.")

# --------------------------------------------------
# 4. Inspect temporal feature names
# --------------------------------------------------

temporal_keywords = [
    "rolling",
    "baseline",
    "historical",
    "trend",
    "lag",
    "change",
    "_3d",
    "_7d",
    "_14d",
    "_3day",
    "_7day",
    "_14day"
]

temporal_columns = [
    column
    for column in candidate_feature_columns
    if any(
        keyword in column.lower()
        for keyword in temporal_keywords
    )
]

print("\nTEMPORAL FEATURE CANDIDATES")
print("=" * 60)

print("Number of temporal candidates:", len(temporal_columns))

for column in temporal_columns[:100]:
    print(column)

if len(temporal_columns) > 100:
    print("... additional columns not displayed")

# --------------------------------------------------
# 5. Inspect the feature dictionary
# --------------------------------------------------

FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "feature_dictionary.csv"
)

print("\nFEATURE DICTIONARY")
print("=" * 60)

print("File exists:", FEATURE_DICTIONARY_PATH.exists())

if FEATURE_DICTIONARY_PATH.exists():

    feature_dictionary = pd.read_csv(
        FEATURE_DICTIONARY_PATH
    )

    print("Dictionary shape:", feature_dictionary.shape)
    print("Dictionary columns:")
    print(feature_dictionary.columns.tolist())

    display(feature_dictionary.head(15))


#### Step 5A: Investigating Temporal Feature Leakage

**Objective:** Verify that rolling and personal-baseline features use only information that would have been available at the prediction date.

**Observed results so far:**
- The merged dataset contains 1,922 columns.
- Six columns were excluded from predictors: `Participant_ID`, `Date`, `exacerbation_within_7d`, `label_status`, `matched_event_date`, and `matched_event_severity`.
- This leaves 1,916 candidate predictor features.
- The feature dictionary contains 1,916 entries, matching the candidate feature count.
- No suspicious feature names were detected.
- 1,744 candidate features were identified as temporal features by their names.

**Why further investigation is necessary:** A feature name containing `rolling`, `baseline`, or `historical` does not prove that it was calculated using past data only. The feature-engineering implementation must be inspected to confirm the time direction and avoid future information entering the predictors.

**Same-day availability:** Features described as using the current calendar day require special attention. Their suitability depends on when a prediction is intended to be made and whether the measurements used in that feature would already be available at that time.

**Interpretation:** The initial column-level checks are satisfactory, but temporal leakage has not yet been ruled out. No final predictor dataset should be created until the rolling, baseline, and same-day feature calculations have been reviewed.

In [ ]:


# STEP 5A: Inspect temporal feature definitions and implementation

import json
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# PART 1: Inspect the feature dictionary
# --------------------------------------------------

print("=" * 70)
print("FEATURE GROUP DISTRIBUTION")
print("=" * 70)

print(feature_dictionary["Feature_Group"].value_counts(dropna=False))

print("\n" + "=" * 70)
print("TIME WINDOW DISTRIBUTION")
print("=" * 70)

print(feature_dictionary["Time_Window"].value_counts(dropna=False).to_string())

# Select rolling and baseline features
temporal_dictionary = feature_dictionary[
    feature_dictionary["Feature"].str.contains(
        "rolling|baseline|historical|trend|lag",
        case=False,
        na=False
    )
].copy()

print("\n" + "=" * 70)
print("TEMPORAL FEATURE DICTIONARY")
print("=" * 70)

print("Number of matching dictionary entries:", len(temporal_dictionary))

columns_to_show = [
    "Feature",
    "Feature_Group",
    "Description",
    "Formula",
    "Time_Window",
    "Leakage_Considerations"
]

print(
    temporal_dictionary[columns_to_show]
    .head(30)
    .to_string(index=False, max_colwidth=100)
)

# --------------------------------------------------
# PART 2: Inspect feature-engineering notebook code
# --------------------------------------------------

notebook_path = PROJECT_ROOT / "notebooks" / "04_feature_engineering.ipynb"

print("\n" + "=" * 70)
print("FEATURE ENGINEERING NOTEBOOK INSPECTION")
print("=" * 70)

print("Notebook exists:", notebook_path.exists())

if notebook_path.exists():

    with open(notebook_path, "r", encoding="utf-8") as file:
        notebook = json.load(file)

    search_terms = [
        ".rolling(",
        ".shift(",
        ".expanding(",
        "center=True",
        "center=False",
        "bfill",
        "backfill",
        "baseline",
        "historical",
        "rolling"
    ]

    for cell_number, cell in enumerate(notebook["cells"]):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matching_lines = []

        for line_number, line in enumerate(source.splitlines(), start=1):
            if any(term.lower() in line.lower() for term in search_terms):
                matching_lines.append(
                    f"Line {line_number}: {line.strip()}"
                )

        if matching_lines:
            print(f"\n--- CODE CELL {cell_number} ---")
            for line in matching_lines:
                print(line)


In [ ]:

# STEP 5B: Inspect complete temporal feature calculations

import json

notebook_path = PROJECT_ROOT / "notebooks" / "04_feature_engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as file:
    notebook_04 = json.load(file)

cells_to_inspect = [55, 60, 124, 125, 128, 129, 132, 150]

for cell_number in cells_to_inspect:

    cell = notebook_04["cells"][cell_number]

    if cell.get("cell_type") == "code":

        source = "".join(cell.get("source", []))

        print("\n" + "=" * 90)
        print(f"NOTEBOOK 04 - CODE CELL {cell_number}")
        print("=" * 90)
        print(source)


#### Step 5B: Review of Temporal Feature Calculations

**Objective:** Inspect the implementation of rolling, baseline, and data-quality features to identify potential temporal leakage.

**Observed implementation:**
- Participant records are sorted chronologically before rolling and baseline calculations.
- Rolling features use `shift(1)` before applying rolling statistics.
- Historical baselines use `shift(1)` before expanding calculations.
- Additional baseline deviation features compare the current measurement with the historical mean.
- Seven-day data-quality coverage features use shifted availability values before rolling calculations.
- The notebook contains an earlier rolling implementation and a later optimized rebuild.

**Interpretation:** The inspected code is consistent with historical-only calculations for rolling and baseline statistics. Current-day deviation features intentionally use the current measurement and require that measurement to be available at prediction time.

**Remaining verification:** Confirm that the later optimized calculations were executed before final dataset assembly and that the independent temporal leakage checks passed.

**Prediction-time assumption:** This project currently assumes that daily predictions are made after the relevant current-day sensor measurements have become available. This assumption must be retained when designing the eventual backend prediction schedule.

In [ ]:

# STEP 5C: Verify final feature pipeline execution order

import json

notebook_path = PROJECT_ROOT / "notebooks" / "04_feature_engineering.ipynb"

with open(notebook_path, "r", encoding="utf-8") as file:
    notebook_04 = json.load(file)

important_cells = [55, 60, 124, 125, 128, 137, 146, 150]

print("=" * 75)
print("NOTEBOOK 04 CELL EXECUTION ORDER")
print("=" * 75)

for cell_number in important_cells:
    cell = notebook_04["cells"][cell_number]

    print(
        f"Cell {cell_number}: "
        f"Execution count = {cell.get('execution_count')}"
    )

    if cell.get("cell_type") == "code":
        if cell_number in [137, 146]:
            print("\n" + "-" * 75)
            print(f"SOURCE OF CELL {cell_number}")
            print("-" * 75)
            print("".join(cell.get("source", [])))

print("\n" + "=" * 75)
print("END OF PIPELINE INSPECTION")
print("=" * 75)


### Step 5: Feature Leakage Audit

**Objective**

Identify and exclude non-predictor columns and assess whether the engineered features contain information from future dates.

**1. Excluded non-predictor columns**

The following six columns were excluded from the candidate predictor features:

- `Participant_ID`
- `Date`
- `exacerbation_within_7d`
- `label_status`
- `matched_event_date`
- `matched_event_severity`

The remaining 1,916 columns were identified as candidate predictor features.

**2. Feature dictionary inspection**

The feature dictionary was inspected to understand the temporal construction of the engineered features.

The feature groups included:
- Rolling features
- Historical baseline features
- Daily features
- Data-quality features
- Cross-sensor features

Rolling and baseline features were designed to use historical observations. Daily features represent measurements from the current calendar day.

**3. Independent temporal leakage audit**

An independent calculation was performed for participant GER-003 using `BRI_rf_mean_mean`.

The stored rolling counts and means matched independently calculated historical values for the 3-day, 7-day and 14-day windows.

The expanding baseline count, mean, median and standard deviation also matched independently calculated historical statistics.

**Observed result:** All checks in the selected temporal audit passed.

**4. Modelling considerations**

- Historical rolling and baseline features must not include observations from future dates.
- Current-day sensor features are usable only under an end-of-day prediction assumption, after the relevant measurements are available.
- Exacerbation dates, severity and label-related columns must never be included as model predictors.
- This audit is a targeted spot-check, not an exhaustive verification of every feature.

**Conclusion**

The selected rolling and expanding baseline features passed the independent temporal leakage checks. The candidate feature set is ready for further validation and model dataset preparation, subject to the prediction-time assumption and additional checks before model training.

### Step 6: Feature Availability and Missingness Assessment

**Objective**

Assess missing values across the candidate predictor features before preparing the final modelling dataset.

**Why this step is necessary**

The dataset combines multiple sensor streams with different observation frequencies and availability. Some features may be missing because a sensor was unavailable, a measurement was invalid, or a rolling window did not contain enough historical observations.

Missing values must be understood before selecting an imputation strategy or deciding whether particular features should be excluded.

**Planned checks**

1. Calculate the missing-value count and percentage for every candidate predictor.
2. Identify features with high missingness.
3. Compare feature availability across participants.
4. Examine whether missingness differs between positive and negative target observations.
5. Avoid automatic zero imputation or indiscriminate row deletion.

No imputation or feature removal will be performed at this stage.

In [ ]:
# STEP 6: Feature availability and missingness assessment

# Select candidate predictor columns
non_predictor_columns = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

candidate_features = [
    col for col in known_target_dataset.columns
    if col not in non_predictor_columns
]

# Calculate missingness
missingness = pd.DataFrame({
    "Feature": candidate_features,
    "Missing_Count": known_target_dataset[candidate_features].isna().sum().values,
    "Missing_Percentage": (
        known_target_dataset[candidate_features].isna().mean().values * 100
    )
})

missingness = missingness.sort_values(
    "Missing_Percentage",
    ascending=False
).reset_index(drop=True)

print("FEATURE MISSINGNESS SUMMARY")
print("=" * 65)

print("Total candidate features:", len(candidate_features))
print("Total known-target rows:", len(known_target_dataset))
print("Features with missing values:",
      (missingness["Missing_Count"] > 0).sum())
print("Features with no missing values:",
      (missingness["Missing_Count"] == 0).sum())

print("\nTOP 20 FEATURES WITH HIGHEST MISSINGNESS")
display(missingness.head(20))

print("\nMISSINGNESS DISTRIBUTION")
print(
    missingness["Missing_Percentage"].describe()
)

#### Step 6A: Missingness by Sensor and Feature Group

**Objective**

Identify which sensor streams and engineered feature groups contribute most to missingness.

**Method**

The feature missingness summary will be joined with the feature dictionary using the feature name.

Missingness will be summarized by:
- Source sensor
- Feature group

For each group, calculate:
- Number of features
- Average missingness percentage
- Median missingness percentage
- Number of completely missing-free features
- Number of features with more than 90% missingness

**Reason**

Different sensors have different observation frequencies and participant coverage. Group-level analysis helps distinguish sensor availability problems from missingness introduced during feature engineering.

No features will be removed at this stage.

In [ ]:
# STEP 6A: Missingness by sensor and feature group

# Load the feature dictionary
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "feature_dictionary.csv"
)

feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)

# Confirm feature dictionary uniqueness
assert not feature_dictionary["Feature"].duplicated().any(), (
    "Duplicate feature names found in feature dictionary."
)

# Combine missingness results with feature metadata
missingness_detailed = missingness.merge(
    feature_dictionary[
        ["Feature", "Source_Sensor", "Feature_Group"]
    ],
    on="Feature",
    how="left",
    validate="one_to_one"
)

# Check whether all features were matched
print("FEATURE DICTIONARY MATCHING")
print("=" * 60)
print(
    "Features without dictionary information:",
    missingness_detailed["Feature_Group"].isna().sum()
)

# Function to summarize missingness
def summarize_missingness(data, group_column):

    summary = (
        data.groupby(group_column, dropna=False)
        .agg(
            Feature_Count=("Feature", "count"),
            Average_Missingness=("Missing_Percentage", "mean"),
            Median_Missingness=("Missing_Percentage", "median"),
            Complete_Features=(
                "Missing_Percentage",
                lambda x: (x == 0).sum()
            ),
            Above_90_Percent_Missing=(
                "Missing_Percentage",
                lambda x: (x > 90).sum()
            )
        )
        .reset_index()
        .sort_values("Average_Missingness", ascending=False)
    )

    return summary


# Missingness by sensor
sensor_missingness = summarize_missingness(
    missingness_detailed,
    "Source_Sensor"
)

print("\nMISSINGNESS BY SENSOR")
print("=" * 60)
display(sensor_missingness.round(2))


# Missingness by feature group
group_missingness = summarize_missingness(
    missingness_detailed,
    "Feature_Group"
)

print("\nMISSINGNESS BY FEATURE GROUP")
print("=" * 60)
display(group_missingness.round(2))

#### Step 6B: Investigating Rolling Feature Missingness

**Objective**

Determine whether missing rolling features are associated with insufficient historical observations.

**Method**

For rolling features, identify their corresponding historical count columns. Compare the missingness of each rolling feature with its historical observation count.

The assessment will distinguish:
- Missing feature values when the historical count is zero.
- Missing feature values when the historical count is unavailable.
- Missing feature values despite a positive historical count.
- Available feature values with positive historical counts.

**Reason**

Rolling features depend on previous sensor observations. A feature may legitimately be unavailable when insufficient historical observations exist. Understanding this relationship will help avoid unnecessary feature deletion or inappropriate imputation.

No features will be removed or imputed during this investigation.

In [ ]:
# STEP 6B: Investigate rolling feature missingness

import re

rolling_metadata = missingness_detailed[
    missingness_detailed["Feature_Group"] == "Rolling"
].copy()

rolling_audit_results = []

for feature in rolling_metadata["Feature"]:

    # Extract the rolling window identifier
    match = re.match(r"^(.*_rolling_\d+d)", feature)

    if match is None:
        continue

    count_column = match.group(1) + "_count"

    if count_column not in known_target_dataset.columns:
        continue

    feature_values = known_target_dataset[feature]
    historical_counts = known_target_dataset[count_column]

    feature_missing = feature_values.isna()

    count_missing = historical_counts.isna()
    count_zero = historical_counts.eq(0)
    count_positive = historical_counts.gt(0)

    rolling_audit_results.append({
        "Feature": feature,
        "Count_Column": count_column,
        "Window": match.group(1).split("_rolling_")[-1],
        "Feature_Missing_Percent": feature_missing.mean() * 100,
        "Count_Missing_Percent": count_missing.mean() * 100,
        "Count_Zero_Percent": count_zero.mean() * 100,
        "Count_Positive_Percent": count_positive.mean() * 100,
        "Missing_With_Zero_Count": (
            feature_missing & count_zero
        ).sum(),
        "Missing_With_Missing_Count": (
            feature_missing & count_missing
        ).sum(),
        "Missing_With_Positive_Count": (
            feature_missing & count_positive
        ).sum(),
        "Available_With_Positive_Count": (
            feature_values.notna() & count_positive
        ).sum()
    })


rolling_audit = pd.DataFrame(rolling_audit_results)

print("ROLLING FEATURE COUNT MATCHING")
print("=" * 65)
print("Total rolling features:", len(rolling_metadata))
print("Features matched with count columns:", len(rolling_audit))
print("Features without matching count columns:",
      len(rolling_metadata) - len(rolling_audit))

print("\nOVERALL ROLLING MISSINGNESS INVESTIGATION")
print("=" * 65)

if not rolling_audit.empty:

    print(
        "Average feature missingness:",
        round(rolling_audit["Feature_Missing_Percent"].mean(), 2), "%"
    )

    print(
        "Average missingness with zero historical count:",
        round(
            rolling_audit["Missing_With_Zero_Count"].sum()
            / len(rolling_metadata) / len(known_target_dataset) * 100,
            2
        ),
        "%"
    )

    print(
        "Total missing values despite positive historical count:",
        rolling_audit["Missing_With_Positive_Count"].sum()
    )

    print(
        "Total available values with positive historical count:",
        rolling_audit["Available_With_Positive_Count"].sum()
    )

    print("\nTOP 20 ROLLING FEATURES WITH MISSING VALUES")
    display(
        rolling_audit.sort_values(
            "Feature_Missing_Percent",
            ascending=False
        ).head(20).round(2)
    )

    print("\nSUMMARY BY ROLLING WINDOW")
    display(
        rolling_audit.groupby("Window").agg(
            Feature_Count=("Feature", "count"),
            Average_Missingness=("Feature_Missing_Percent", "mean"),
            Average_Zero_Count=("Count_Zero_Percent", "mean"),
            Average_Positive_Count=("Count_Positive_Percent", "mean"),
            Missing_With_Positive_Count=(
                "Missing_With_Positive_Count", "sum"
            )
        ).round(2)
    )

else:
    print("No matching rolling count columns were found.")

#### Step 6C: Investigating Missingness by Rolling Calculation Type

**Objective**

Investigate why rolling features remain missing despite having positive historical observation counts.

**Method**

Rolling features will be grouped by their calculation type, such as:
- Mean
- Standard deviation
- Minimum and maximum
- Change
- Percentage change
- Other derived statistics

For each calculation type, assess:
- Number of features
- Missing-value percentage
- Missing values despite positive historical counts
- Available values despite positive historical counts

Representative examples will also be inspected to understand why certain calculations may be undefined.

**Reason**

A positive historical count does not guarantee that every derived statistic is mathematically calculable. For example, standard deviation requires sufficient observations, and percentage change may be undefined when the reference value is zero.

This investigation will help distinguish expected mathematical missingness from possible feature-engineering issues.

No imputation or feature removal will be performed at this stage.

In [ ]:
# STEP 6C: Missingness by rolling calculation type

# Extract calculation type from feature names
rolling_audit["Calculation_Type"] = (
    rolling_audit["Feature"]
    .str.extract(r"_rolling_\d+d_(.+)$", expand=False)
    .fillna("Other")
)

# Summarize by calculation type
calculation_summary = (
    rolling_audit.groupby("Calculation_Type")
    .agg(
        Feature_Count=("Feature", "count"),
        Average_Missingness=("Feature_Missing_Percent", "mean"),
        Missing_With_Positive_Count=(
            "Missing_With_Positive_Count", "sum"
        ),
        Available_With_Positive_Count=(
            "Available_With_Positive_Count", "sum"
        )
    )
    .reset_index()
    .sort_values("Average_Missingness", ascending=False)
)

print("MISSINGNESS BY CALCULATION TYPE")
print("=" * 65)
display(calculation_summary.round(2))


# Calculate how much of total rolling missingness
# is associated with zero versus positive counts

total_missing = rolling_audit["Feature_Missing_Percent"].sum()

total_missing_zero = rolling_audit[
    "Missing_With_Zero_Count"
].sum()

total_missing_positive = rolling_audit[
    "Missing_With_Positive_Count"
].sum()

print("\nROLLING MISSINGNESS BREAKDOWN")
print("=" * 65)

print("Total missing rolling values:", int(total_missing / 100 * len(known_target_dataset)))

print(
    "Missing with zero historical count:",
    int(total_missing_zero)
)

print(
    "Missing with positive historical count:",
    int(total_missing_positive)
)

print(
    "Percentage of missing values associated with zero count:",
    round(total_missing_zero / (total_missing_zero + total_missing_positive) * 100, 2)
)


# Inspect representative cases where the feature is missing
# despite a positive historical count

positive_count_missing = rolling_audit[
    rolling_audit["Missing_With_Positive_Count"] > 0
].copy()

print("\nEXAMPLES: MISSING VALUES DESPITE POSITIVE HISTORICAL COUNT")
print("=" * 65)

for _, row in positive_count_missing.head(10).iterrows():

    feature = row["Feature"]
    count_column = row["Count_Column"]

    examples = known_target_dataset.loc[
        known_target_dataset[feature].isna()
        & known_target_dataset[count_column].gt(0),
        ["Participant_ID", "Date", count_column, feature]
    ].head(3)

    print(f"\nFeature: {feature}")
    display(examples)

#### Step 6D: Missingness by Target Class

**Objective**

Compare feature availability between positive and negative target observations.

**Method**

Separate the known-target dataset into:
- Positive observations (`exacerbation_within_7d = 1`)
- Negative observations (`exacerbation_within_7d = 0`)

Calculate missingness percentages for each feature group and compare the results.

**Reason**

Sensor availability may differ between observations associated with documented future exacerbations and observations without documented events in the prediction window.

If missingness differs substantially between target classes, the model could potentially learn data-collection patterns rather than clinically meaningful signals.

**Important limitation**

The positive observations are not independent exacerbation events. Multiple feature dates can correspond to the same event. Therefore, this comparison is descriptive and does not establish a causal or predictive relationship.

No features will be removed or imputed during this step.

In [ ]:
# STEP 6D: Compare missingness by target class

# Separate positive and negative observations
positive_data = known_target_dataset[
    known_target_dataset["exacerbation_within_7d"] == 1
]

negative_data = known_target_dataset[
    known_target_dataset["exacerbation_within_7d"] == 0
]

print("TARGET CLASS SIZES")
print("=" * 60)
print("Positive observations:", len(positive_data))
print("Negative observations:", len(negative_data))

# Calculate average feature missingness by feature group
class_missingness_results = []

for feature_group in feature_dictionary["Feature_Group"].unique():

    group_features = feature_dictionary.loc[
        feature_dictionary["Feature_Group"] == feature_group,
        "Feature"
    ].tolist()

    positive_missing = (
        positive_data[group_features].isna().mean().mean() * 100
    )

    negative_missing = (
        negative_data[group_features].isna().mean().mean() * 100
    )

    class_missingness_results.append({
        "Feature_Group": feature_group,
        "Feature_Count": len(group_features),
        "Positive_Missingness_Percent": positive_missing,
        "Negative_Missingness_Percent": negative_missing,
        "Difference_Percentage_Points": (
            positive_missing - negative_missing
        )
    })

class_missingness_summary = pd.DataFrame(
    class_missingness_results
).sort_values(
    "Difference_Percentage_Points",
    key=lambda x: x.abs(),
    ascending=False
)

print("\nMISSINGNESS BY FEATURE GROUP AND TARGET CLASS")
print("=" * 75)
display(class_missingness_summary.round(2))


# Compare overall missingness across all candidate features
positive_overall_missingness = (
    positive_data[candidate_features].isna().mean().mean() * 100
)

negative_overall_missingness = (
    negative_data[candidate_features].isna().mean().mean() * 100
)

print("\nOVERALL MISSINGNESS COMPARISON")
print("=" * 60)
print(
    "Positive observations:",
    round(positive_overall_missingness, 2), "%"
)
print(
    "Negative observations:",
    round(negative_overall_missingness, 2), "%"
)
print(
    "Difference:",
    round(
        positive_overall_missingness - negative_overall_missingness,
        2
    ),
    "percentage points"
)

### Step 6: Feature Availability and Missingness Assessment – Findings

**1. Overall feature missingness**

- Total candidate features: 1,916.
- Features with missing values: 1,647.
- Features without missing values: 269.
- Average feature missingness: 56.91%.
- Median feature missingness: 70.98%.

**2. Missingness by sensor**

The highest average missingness was observed in:
- SQI: 71.45%.
- SHRI: 69.62%.
- BRI: 69.47%.

The lowest average missingness among individual sensor streams was observed in MI: 45.09%.

**3. Missingness by feature group**

- Rolling: 63.64% average missingness.
- Daily: 51.00%.
- Baseline: 40.47%.
- Cross-Sensor: 15.67%.
- Data Quality: 7.01%.

Rolling features contributed the largest number of highly sparse features.

**4. Rolling feature investigation**

All 1,344 rolling features had corresponding historical count columns.

Approximately 91.44% of missing rolling values were associated with zero historical observations.

The remaining missing values occurred despite positive historical counts. Examples included percentage changes without valid comparison values and other derived statistics requiring additional observations or mathematically valid inputs.

**5. Missingness by target class**

Overall missingness:
- Positive observations: 57.47%.
- Negative observations: 56.89%.
- Difference: 0.58 percentage points.

No substantial overall difference was observed at the feature-group level.

**6. Interpretation and limitations**

Missingness is common and varies across sensor streams and engineered feature groups. Insufficient historical observations explain much of the rolling-feature missingness.

The target-class comparison is descriptive and does not establish that missingness is harmless or unbiased. Positive observations may represent overlapping prediction windows around the same documented events.

**Conclusion**

The feature availability assessment is complete. No features have been removed, no missing values have been imputed, and the original engineered feature dataset remains unchanged.

Further feature selection and missing-value handling will be performed using training data only to avoid information leakage.

### Step 7: Participant-Level Modelling Data Availability

**Objective**

Examine the distribution of known-target observations across participants.

**Planned checks**

1. Count positive and negative observations per participant.
2. Identify participants with no positive observations.
3. Identify participants with very few known-target observations.
4. Examine whether positive observations are concentrated among a small number of participants.

**Reason**

The dataset contains repeated daily observations from the same participants. A participant-level assessment is necessary before selecting a validation strategy.

Training and testing data must be split by participant rather than randomly by observation, to prevent information from the same participant appearing in both sets.

No participants or observations will be removed during this assessment.

# STEP 7: Participant-level modelling data availability

participant_summary = (
    known_target_dataset
    .groupby("Participant_ID")
    .agg(
        Total_Known_Observations=(
            "exacerbation_within_7d", "count"
        ),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

participant_summary["Has_Positive_Observation"] = (
    participant_summary["Positive_Observations"] > 0
)

print("PARTICIPANT-LEVEL SUMMARY")
print("=" * 65)

print("Participants with known-target observations:",
      len(participant_summary))

print("Participants with positive observations:",
      participant_summary["Has_Positive_Observation"].sum())

print("Participants without positive observations:",
      (~participant_summary["Has_Positive_Observation"]).sum())

print("\nOBSERVATION DISTRIBUTION")
display(
    participant_summary[
        [
            "Total_Known_Observations",
            "Positive_Observations",
            "Negative_Observations"
        ]
    ].describe().round(2)
)

print("\nPARTICIPANTS WITH THE MOST POSITIVE OBSERVATIONS")
display(
    participant_summary.sort_values(
        "Positive_Observations",
        ascending=False
    ).head(15)
)

print("\nPARTICIPANTS WITH THE FEWEST KNOWN OBSERVATIONS")
display(
    participant_summary.sort_values(
        "Total_Known_Observations",
        ascending=True
    ).head(15)
)

In [ ]:
# STEP 7: Participant-level modelling data availability

participant_summary = (
    known_target_dataset
    .groupby("Participant_ID")
    .agg(
        Total_Known_Observations=(
            "exacerbation_within_7d", "count"
        ),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

participant_summary["Has_Positive_Observation"] = (
    participant_summary["Positive_Observations"] > 0
)

print("PARTICIPANT-LEVEL SUMMARY")
print("=" * 65)

print("Participants with known-target observations:",
      len(participant_summary))

print("Participants with positive observations:",
      participant_summary["Has_Positive_Observation"].sum())

print("Participants without positive observations:",
      (~participant_summary["Has_Positive_Observation"]).sum())

print("\nOBSERVATION DISTRIBUTION")
display(
    participant_summary[
        [
            "Total_Known_Observations",
            "Positive_Observations",
            "Negative_Observations"
        ]
    ].describe().round(2)
)

print("\nPARTICIPANTS WITH THE MOST POSITIVE OBSERVATIONS")
display(
    participant_summary.sort_values(
        "Positive_Observations",
        ascending=False
    ).head(15)
)

print("\nPARTICIPANTS WITH THE FEWEST KNOWN OBSERVATIONS")
display(
    participant_summary.sort_values(
        "Total_Known_Observations",
        ascending=True
    ).head(15)
)

### Step 7: Participant-Level Modelling Data Availability – Findings

**1. Known-target participant coverage**

- Original dataset participants: 74.
- Participants with known-target observations: 24.
- Participants with at least one positive observation: 19.
- Participants without positive observations: 5.
- Participants without known-target observations: 50.

The 50 participants without known labels are excluded from the current supervised modelling dataset. Their missing labels must not be interpreted as negative outcomes.

**2. Observation distribution**

Across the 24 participants:
- Mean known observations per participant: 182.79.
- Mean positive observations per participant: 8.46.
- Mean negative observations per participant: 174.33.
- Minimum known observations: 35.
- Maximum known observations: 356.

**3. Positive observation concentration**

Positive observations are not evenly distributed across participants. The nine participants with the highest positive observation counts account for 135 of the 203 positive rows (approximately 66.5%).

These are repeated prediction observations and must not be interpreted as independent exacerbation events.

**4. Modelling implications**

- All observations from one participant must remain in the same validation fold.
- Random row-level train/test splitting is inappropriate because it can cause participant-level information leakage.
- The validation strategy must account for the small number of participants and the concentration of positive observations.
- Participants without positive observations must not automatically be described as participants who never experienced exacerbations.

**Conclusion**

Participant-level data availability has been assessed. Only 24 participants currently contribute known labels. Validation must be performed at the participant level, with particular attention to the limited number of positive-bearing participants.

### Step 8: Participant-Level Validation Strategy

**Objective**

Design a validation strategy that prevents observations from the same participant appearing in both training and testing data.

**Proposed approach**

Use five-fold stratification at the participant level.

Participants will be stratified according to whether they have at least one positive target observation. Each participant will be assigned to exactly one test fold.

All daily observations from a participant will remain together in that participant's assigned fold.

**Why this approach is necessary**

The dataset contains repeated observations from the same individuals. Random row-level splitting could allow the model to learn participant-specific characteristics from the training set and encounter the same individuals in the test set.

Participant-level splitting provides a more realistic assessment of generalization to previously unseen participants.

**Important limitations**

Only 24 participants have known labels, including 19 participants with positive observations. Therefore, validation results may have considerable uncertainty.

The fold distributions will be inspected before deciding whether this validation strategy is suitable for model evaluation.

In [ ]:
# STEP 8: Create participant-level validation folds

from sklearn.model_selection import StratifiedKFold

# Stratify participants by whether they have any positive observations
participant_labels = participant_summary[
    ["Participant_ID", "Has_Positive_Observation"]
].copy()

# Five-fold stratified split at participant level
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

participant_labels["Fold"] = -1

for fold_number, (_, test_indices) in enumerate(
    skf.split(
        participant_labels["Participant_ID"],
        participant_labels["Has_Positive_Observation"]
    ),
    start=1
):
    participant_labels.loc[
        test_indices, "Fold"
    ] = fold_number

# Attach fold assignments to all known-target observations
model_fold_dataset = known_target_dataset.merge(
    participant_labels[["Participant_ID", "Fold"]],
    on="Participant_ID",
    how="left",
    validate="many_to_one"
)

# Validate assignment
assert model_fold_dataset["Fold"].notna().all()
assert model_fold_dataset["Fold"].between(1, 5).all()

# Each participant must belong to exactly one fold
participant_fold_counts = (
    model_fold_dataset.groupby("Participant_ID")["Fold"].nunique()
)

assert participant_fold_counts.eq(1).all()

# Fold-level summary
fold_summary = (
    model_fold_dataset.groupby("Fold")
    .agg(
        Test_Participants=("Participant_ID", "nunique"),
        Total_Observations=("Participant_ID", "size"),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# Count positive-bearing participants in each fold
positive_participant_counts = (
    model_fold_dataset[
        model_fold_dataset["exacerbation_within_7d"] == 1
    ]
    .groupby("Fold")["Participant_ID"]
    .nunique()
)

fold_summary["Positive_Bearing_Participants"] = (
    fold_summary["Fold"]
    .map(positive_participant_counts)
    .fillna(0)
    .astype(int)
)

print("FIVE-FOLD PARTICIPANT-LEVEL VALIDATION")
print("=" * 65)

display(fold_summary)

print("\nVALIDATION CHECKS")
print("=" * 65)
print(
    "Total participants:",
    model_fold_dataset["Participant_ID"].nunique()
)
print(
    "Participants assigned to exactly one fold:",
    participant_fold_counts.eq(1).sum()
)
print(
    "Total observations:",
    len(model_fold_dataset)
)
print(
    "Total positive observations:",
    (model_fold_dataset["exacerbation_within_7d"] == 1).sum()
)
print(
    "Total negative observations:",
    (model_fold_dataset["exacerbation_within_7d"] == 0).sum()
)

print(
    "\nParticipants assigned to multiple folds:",
    (participant_fold_counts > 1).sum()
)

### Step 8: Participant-Level Validation – Results

**1. Validation strategy**

Five-fold stratified cross-validation was implemented at the participant level using `StratifiedKFold`.

Participants were stratified according to whether they had at least one positive target observation. All observations belonging to a participant were assigned to the same fold.

**2. Fold distribution**

| Fold | Test participants | Total observations | Positive observations | Negative observations | Positive-bearing participants |
|---|---:|---:|---:|---:|---:|
| 1 | 5 | 1,047 | 42 | 1,005 | 4 |
| 2 | 5 | 998 | 37 | 961 | 4 |
| 3 | 5 | 796 | 49 | 747 | 4 |
| 4 | 5 | 669 | 33 | 636 | 4 |
| 5 | 4 | 877 | 42 | 835 | 3 |

**3. Validation checks**

- Total participants assigned: 24.
- Participants assigned to exactly one fold: 24.
- Participants assigned to multiple folds: 0.
- Total observations retained: 4,387.
- Positive observations retained: 203.
- Negative observations retained: 4,184.

All validation assignment checks passed.

**4. Interpretation**

The five folds have reasonably balanced participant-level class distributions, with three or four positive-bearing participants in each test fold.

The positive observation counts are not independent exacerbation event counts. Multiple positive daily observations may correspond to the same documented event.

**5. Limitations**

- The validation population contains only 24 participants.
- Performance estimates may have substantial uncertainty.
- This cross-validation design does not provide an independent external test set.
- Feature selection, missing-value imputation, scaling and other learned preprocessing must be fitted using only the training participants within each fold.

**Conclusion**

Five-fold participant-level cross-validation is suitable as a preliminary evaluation strategy for this dataset. The fold assignments passed all participant separation and observation-retention checks.

### Step 9A: Feature Availability – Missingness Threshold Assessment

**Objective**

Assess how many candidate features have high proportions of missing values and identify potentially unsuitable features for future modelling.

**Why this is necessary**

A feature with very high missingness may provide information for only a small subset of observations. Retaining many such features can make model training unstable and increase the risk of overfitting.

However, high missingness does not automatically mean a feature is useless. Missingness may reflect sensor availability, insufficient historical observations, or the mathematical requirements of a feature calculation.

**Approach**

Calculate the number and percentage of features exceeding the following missingness thresholds:

- More than 50% missing.
- More than 70% missing.
- More than 80% missing.
- More than 90% missing.
- More than 95% missing.
- More than 99% missing.

Also identify features with complete data.

**Important limitation**

These thresholds are exploratory screening criteria, not automatic feature-removal rules. Final feature selection must be performed using training data only within each validation fold to avoid information leakage.

In [ ]:
# STEP 9A: Investigate feature missingness thresholds

import pandas as pd
import numpy as np

# Recalculate missingness using the known-target observations
non_predictor_columns = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

candidate_features = [
    col for col in known_target_dataset.columns
    if col not in non_predictor_columns
]

missingness = pd.DataFrame({
    "Feature": candidate_features,
    "Missing_Count": (
        known_target_dataset[candidate_features]
        .isna()
        .sum()
        .values
    ),
    "Missing_Percentage": (
        known_target_dataset[candidate_features]
        .isna()
        .mean()
        .values * 100
    )
})

# Define exploratory missingness thresholds
thresholds = [50, 70, 80, 90, 95, 99]

threshold_summary = []

for threshold in thresholds:

    feature_count = (
        missingness["Missing_Percentage"] > threshold
    ).sum()

    threshold_summary.append({
        "Missingness_Threshold": f">{threshold}%",
        "Feature_Count": feature_count,
        "Percentage_Of_All_Features": (
            feature_count / len(candidate_features) * 100
        )
    })

threshold_summary = pd.DataFrame(threshold_summary)

# Count complete and partially missing features
complete_features = (
    missingness["Missing_Percentage"] == 0
).sum()

partially_missing_features = (
    missingness["Missing_Percentage"] > 0
).sum()

print("FEATURE MISSINGNESS THRESHOLD SUMMARY")
print("=" * 65)

display(threshold_summary.round(2))

print("\nOVERALL FEATURE AVAILABILITY")
print("=" * 65)

print("Total candidate features:", len(candidate_features))
print("Complete features:", complete_features)
print("Features with missing values:", partially_missing_features)

print(
    "Average missingness:",
    round(missingness["Missing_Percentage"].mean(), 2),
    "%"
)

# Display the 20 most sparse features
print("\n20 FEATURES WITH THE HIGHEST MISSINGNESS")
print("=" * 65)

display(
    missingness
    .sort_values("Missing_Percentage", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

### Step 9A: Feature Availability – Missingness Threshold Results

**Observed results**

The known-target modelling dataset contains 4,387 observations and 1,916 candidate features.

- Complete features: 269.
- Features with missing values: 1,647.
- Average feature missingness: 56.91%.

Missingness threshold assessment:

| Missingness threshold | Feature count | Percentage of all features |
|---|---:|---:|
| Greater than 50% | 1,361 | 71.03% |
| Greater than 70% | 1,058 | 55.22% |
| Greater than 80% | 439 | 22.91% |
| Greater than 90% | 227 | 11.85% |
| Greater than 95% | 63 | 3.29% |
| Greater than 99% | 7 | 0.37% |

**Important observations**

1. Several EI percentage-change features have approximately 99–99.8% missingness.
2. Multiple SQI rolling and baseline features have approximately 96.60% missingness.
3. Some baseline standardized-deviation features are highly sparse.

**Interpretation**

High missingness may result from limited sensor availability, insufficient historical observations, or undefined mathematical calculations. Missingness does not automatically indicate that the original sensor measurement is unavailable.

**Conclusion**

Feature availability is a significant modelling concern. The missingness thresholds will be used as exploratory screening criteria only. No features have been removed or imputed at this stage. Final feature selection must be performed using training data within each validation fold.

### Step 9B: Constant and Near-Constant Feature Assessment

**Objective**

Identify candidate features that have no variation or very little variation across known-target observations.

**Why this is necessary**

Machine-learning models generally gain limited predictive information from features that take the same value in almost every observation.

Constant features have no variation and cannot distinguish between observations. Near-constant features may also contribute little information and can increase model complexity.

**Approach**

For each candidate feature, calculate:

- Number of unique non-missing values.
- Most frequent non-missing value.
- Percentage of non-missing observations represented by that value.
- Constant-feature status.
- Near-constant-feature status using a 99% frequency threshold.

**Important limitation**

A near-constant feature is not automatically useless. Rare values may represent clinically important events. These features will be investigated rather than automatically removed.

In [ ]:
# STEP 9B: Constant and near-constant feature assessment

constant_summary = []

for feature in candidate_features:

    series = known_target_dataset[feature].dropna()

    if len(series) == 0:
        constant_summary.append({
            "Feature": feature,
            "Non_Missing_Count": 0,
            "Unique_Values": 0,
            "Most_Frequent_Value": np.nan,
            "Most_Frequent_Percentage": np.nan,
            "Is_Constant": False,
            "Is_Near_Constant": False
        })
        continue

    value_counts = series.value_counts(dropna=True)

    unique_count = series.nunique()
    most_frequent_percentage = (
        value_counts.iloc[0] / len(series) * 100
    )

    constant_summary.append({
        "Feature": feature,
        "Non_Missing_Count": len(series),
        "Unique_Values": unique_count,
        "Most_Frequent_Value": value_counts.index[0],
        "Most_Frequent_Percentage": most_frequent_percentage,
        "Is_Constant": unique_count == 1,
        "Is_Near_Constant": (
            unique_count > 1 and most_frequent_percentage >= 99
        )
    })

constant_summary = pd.DataFrame(constant_summary)

print("CONSTANT AND NEAR-CONSTANT FEATURE SUMMARY")
print("=" * 65)

print("Total candidate features:", len(candidate_features))

print(
    "Constant features:",
    constant_summary["Is_Constant"].sum()
)

print(
    "Near-constant features (>=99%):",
    constant_summary["Is_Near_Constant"].sum()
)

print(
    "Features with no non-missing observations:",
    (constant_summary["Non_Missing_Count"] == 0).sum()
)

print("\nMOST CONSTANT FEATURES")
print("=" * 65)

display(
    constant_summary[
        constant_summary["Is_Constant"] |
        constant_summary["Is_Near_Constant"]
    ]
    .sort_values(
        "Most_Frequent_Percentage",
        ascending=False
    )
    .head(25)
    .reset_index(drop=True)
)

print("\nFEATURES WITH THE MOST UNIQUE VALUES")
print("=" * 65)

display(
    constant_summary
    .sort_values("Unique_Values", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

### Step 9B: Constant and Near-Constant Feature Assessment – Results

**Observed results**

Across 1,916 candidate features:

- Constant features: 17.
- Near-constant features (at least 99% of available observations share the same value): 22.
- Features with no non-missing observations: 0.

**Important observations**

1. Several daily observation-count features are constantly equal to one when available.
2. Several BRI, MI, SHRI and SQI data-quality conflict flags are always False.
3. Some EI rolling and baseline features are constantly zero.
4. Several SII and EI data-quality features are near-constant, with zero as their most frequent value.
5. MI rolling movement statistics have substantial variation, with some features containing more than 1,500 unique values.

**Interpretation**

Constant features provide no variation in the current modelling dataset and are potential candidates for exclusion.

Near-constant features require more careful investigation. Rare nonzero values may represent clinically or operationally meaningful observations, including sensor-quality issues.

**Conclusion**

The assessment identified 17 constant and 22 near-constant features. No features have been removed. Their final treatment will be determined within the training data of each validation fold.

### Step 9C: Highly Correlated Feature Assessment

**Objective**

Identify candidate features that may contain highly overlapping information.

**Why this is necessary**

When two features are strongly correlated, they may represent similar patterns or measurements. Retaining large numbers of redundant features can increase model complexity and make interpretation more difficult.

**Approach**

- Calculate pairwise Pearson correlations between candidate numerical features.
- Use pairwise complete observations when calculating each correlation.
- Identify feature pairs with absolute correlation greater than or equal to 0.95.
- Count the number of highly correlated pairs.
- Inspect the strongest relationships.

**Important limitations**

Correlation indicates association, not causation or equivalence. Two correlated features may still have different clinical meanings.

This is an exploratory analysis. Correlation-based feature filtering must be performed using training observations only within each validation fold. The target variable and target-related metadata must not be included in the correlation matrix.

In [ ]:
# STEP 9C: Highly correlated feature assessment

# Select numerical candidate features only
numeric_features = (
    known_target_dataset[candidate_features]
    .select_dtypes(include="number")
    .columns.tolist()
)

print("Total candidate features:", len(candidate_features))
print("Numerical candidate features:", len(numeric_features))

# Calculate pairwise Pearson correlation
correlation_matrix = (
    known_target_dataset[numeric_features]
    .corr(method="pearson", min_periods=30)
)

# Extract upper triangle to avoid duplicate pairs
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

# Convert correlation matrix into pairwise table
correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

correlation_pairs["Absolute_Correlation"] = (
    correlation_pairs["Correlation"].abs()
)

# Identify highly correlated pairs
highly_correlated_pairs = (
    correlation_pairs[
        correlation_pairs["Absolute_Correlation"] >= 0.95
    ]
    .sort_values(
        "Absolute_Correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nCORRELATION SUMMARY")
print("=" * 65)

print("Numerical features:", len(numeric_features))
print("Valid feature pairs:", len(correlation_pairs))

print(
    "Highly correlated pairs (|r| >= 0.95):",
    len(highly_correlated_pairs)
)

print(
    "Positive highly correlated pairs:",
    (
        highly_correlated_pairs["Correlation"] >= 0.95
    ).sum()
)

print(
    "Negative highly correlated pairs:",
    (
        highly_correlated_pairs["Correlation"] <= -0.95
    ).sum()
)

print("\nTOP 25 HIGHLY CORRELATED FEATURE PAIRS")
print("=" * 65)

display(
    highly_correlated_pairs
    .head(25)
    .round(4)
)

### Step 9C: Highly Correlated Feature Assessment – Results

**Observed results**

- Numerical candidate features: 1,902.
- Valid pairwise correlations: 3,617,604.
- Highly correlated pairs (absolute Pearson correlation at least 0.95): 4,271.
- Strong positive pairs: 4,234.
- Strong negative pairs: 37.

**Important observations**

1. Several BRI validity and availability features have perfect positive or negative correlations.
2. SQI valid observation counts and coverage percentages have a perfect positive correlation.
3. Multiple PRVI rolling observation-count features have perfect positive correlations across different physiological measurements.
4. Some SQI baseline statistics also have perfect correlations despite having different feature names.

**Interpretation**

The results indicate substantial potential feature redundancy, particularly among data-quality, availability and rolling observation-count features.

However, highly correlated features are not necessarily identical. Different physiological measurements or statistical calculations can exhibit similar patterns without being interchangeable.

**Limitations**

- Correlations were calculated using pairwise available observations.
- The number of overlapping observations can differ between feature pairs.
- Correlation does not establish clinical equivalence.
- This analysis used the

### Step 9D: Exact Duplicate Feature Investigation

**Objective**

Determine whether highly correlated feature pairs are genuinely identical or merely strongly associated.

**Approach**

For every highly correlated pair, calculate:

- Number of observations where both features are available.
- Whether both features have identical missing-value patterns.
- Whether their available numerical values are equal within a small floating-point tolerance.
- Percentage of overlapping observations with matching values.

This will help distinguish exact duplicates from features that are only correlated.

**Interpretation**

Two features will be considered exact duplicates only when their missing-value patterns match and their numerical values match within the specified tolerance across all observations.

Pairs with perfect correlation but different values will be retained for further investigation rather than automatically treated as duplicates.

No features will be removed during this assessment.

In [ ]:
# STEP 9D: Investigate exact duplicate feature pairs

duplicate_pair_results = []

for _, pair in highly_correlated_pairs.iterrows():

    feature_1 = pair["Feature_1"]
    feature_2 = pair["Feature_2"]

    values_1 = known_target_dataset[feature_1]
    values_2 = known_target_dataset[feature_2]

    # Compare missing-value patterns
    missing_pattern_match = (
        values_1.isna() == values_2.isna()
    ).all()

    # Keep observations where both features are available
    valid_mask = values_1.notna() & values_2.notna()

    x = values_1[valid_mask].to_numpy(dtype=float)
    y = values_2[valid_mask].to_numpy(dtype=float)

    overlap_count = len(x)

    if overlap_count > 0:

        # Compare values using floating-point tolerance
        matching_values = np.isclose(
            x,
            y,
            rtol=1e-9,
            atol=1e-12
        )

        matching_percentage = matching_values.mean() * 100

        max_absolute_difference = np.max(
            np.abs(x - y)
        )

    else:
        matching_percentage = np.nan
        max_absolute_difference = np.nan

    exact_duplicate = (
        missing_pattern_match
        and overlap_count > 0
        and matching_percentage == 100
    )

    duplicate_pair_results.append({
        "Feature_1": feature_1,
        "Feature_2": feature_2,
        "Correlation": pair["Correlation"],
        "Overlap_Count": overlap_count,
        "Missing_Pattern_Match": missing_pattern_match,
        "Matching_Value_Percentage": matching_percentage,
        "Max_Absolute_Difference": max_absolute_difference,
        "Is_Exact_Duplicate": exact_duplicate
    })


duplicate_pair_audit = pd.DataFrame(
    duplicate_pair_results
)

print("EXACT DUPLICATE FEATURE INVESTIGATION")
print("=" * 65)

print(
    "Highly correlated pairs investigated:",
    len(duplicate_pair_audit)
)

print(
    "Exact duplicate pairs:",
    duplicate_pair_audit["Is_Exact_Duplicate"].sum()
)

print(
    "Pairs with identical missing-value patterns:",
    duplicate_pair_audit["Missing_Pattern_Match"].sum()
)

print(
    "Pairs with at least 100 overlapping observations:",
    (duplicate_pair_audit["Overlap_Count"] >= 100).sum()
)

print("\nOVERLAP COUNT DISTRIBUTION")
print("=" * 65)

display(
    duplicate_pair_audit["Overlap_Count"]
    .describe()
    .to_frame("Overlap_Count")
)

print("\nEXACT DUPLICATE PAIRS")
print("=" * 65)

display(
    duplicate_pair_audit[
        duplicate_pair_audit["Is_Exact_Duplicate"]
    ]
    .sort_values("Correlation", ascending=False)
    .head(25)
    .round(6)
)

print("\nPERFECTLY CORRELATED BUT NOT EXACT DUPLICATES")
print("=" * 65)

display(
    duplicate_pair_audit[
        (duplicate_pair_audit["Correlation"].abs() >= 0.999999)
        & (~duplicate_pair_audit["Is_Exact_Duplicate"])
    ]
    .sort_values("Overlap_Count", ascending=False)
    .head(25)
    .round(6)
)

### Step 9D: Exact Duplicate Feature Investigation – Results

**Observed results**

Among 4,271 highly correlated feature pairs:

- Exact duplicate pairs: 1,060.
- Pairs with identical missing-value patterns: 2,513.
- Pairs with at least 100 overlapping observations: 4,260.
- Median overlapping observations per pair: 3,332.
- Minimum overlapping observations: 31.
- Maximum overlapping observations: 4,387.

**Important observations**

1. Several PRVI rolling count features are exact duplicates across different physiological measurements.
2. Some sensor observation counts and validity indicators are exact duplicates.
3. Some count and coverage-percentage features have perfect correlation but different numerical scales.
4. Some feature pairs have identical values where both are available but different missing-value patterns.

**Interpretation**

Exact duplicate features provide identical numerical information and identical availability patterns in the investigated observations. They are potential candidates for redundancy reduction.

Perfectly correlated features with different scales or missingness patterns must not automatically be considered exact duplicates.

**Conclusion**

A total of 1,060 exact duplicate pairs were identified. No features have been removed. The next step is to group connected duplicate features so that we can determine how many distinct redundant feature groups exist.

### Step 9E: Grouping Exact Duplicate Features

**Objective**

Identify groups of features that are exact duplicates of one another.

**Why this is necessary**

The pairwise duplicate analysis identifies relationships between two features at a time. However, several features may all belong to the same duplicate group.

For example, four identical features produce six pairwise duplicate relationships. Counting pairs alone therefore overstates the number of independent redundancy groups.

**Approach**

Use the exact duplicate pairs to construct connected groups of features. Each group contains features that are identical in their observed values and missing-value patterns.

For each group, record:
- Group identifier.
- Number of features.
- Feature names.
- Associated feature groups and sensor sources from the feature dictionary, where available.

**Important limitation**

These groups are based on the current known-target observations. Final duplicate removal must be verified within each training fold. A feature should only be removed if its duplicate relationship is established using training data.

In [ ]:
# STEP 9E: Group exact duplicate features

# Load the feature dictionary
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "feature_dictionary.csv"
)

feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)

# Extract exact duplicate pairs
exact_pairs = duplicate_pair_audit[
    duplicate_pair_audit["Is_Exact_Duplicate"]
][["Feature_1", "Feature_2"]].copy()

# Union-Find structure to group connected duplicate features
parent = {}

def find(feature):
    if feature not in parent:
        parent[feature] = feature

    if parent[feature] != feature:
        parent[feature] = find(parent[feature])

    return parent[feature]


def union(feature_1, feature_2):
    root_1 = find(feature_1)
    root_2 = find(feature_2)

    if root_1 != root_2:
        parent[root_2] = root_1


# Join features that are exact duplicates
for _, row in exact_pairs.iterrows():
    union(row["Feature_1"], row["Feature_2"])


# Collect connected groups
duplicate_groups = {}

for feature in parent:
    root = find(feature)

    duplicate_groups.setdefault(root, []).append(feature)


# Keep only groups containing at least two features
duplicate_groups = [
    sorted(group)
    for group in duplicate_groups.values()
    if len(group) >= 2
]

duplicate_groups = sorted(
    duplicate_groups,
    key=lambda group: (-len(group), group[0])
)


# Build a group-level summary
duplicate_group_records = []

for group_number, group in enumerate(duplicate_groups, start=1):

    duplicate_group_records.append({
        "Duplicate_Group_ID": f"DUP_{group_number:03d}",
        "Number_Of_Features": len(group),
        "Features": group
    })


duplicate_group_summary = pd.DataFrame(
    duplicate_group_records
)


# Summarise the groups
features_in_duplicate_groups = {
    feature
    for group in duplicate_groups
    for feature in group
}

print("EXACT DUPLICATE GROUP SUMMARY")
print("=" * 65)

print("Exact duplicate pairs:", len(exact_pairs))
print("Duplicate groups:", len(duplicate_groups))
print(
    "Unique features involved:",
    len(features_in_duplicate_groups)
)
print(
    "Potential redundant features if retaining one per group:",
    sum(len(group) - 1 for group in duplicate_groups)
)

print("\nDUPLICATE GROUP SIZE DISTRIBUTION")
print("=" * 65)

display(
    duplicate_group_summary["Number_Of_Features"]
    .value_counts()
    .sort_index()
    .rename_axis("Features_Per_Group")
    .reset_index(name="Number_Of_Groups")
)

print("\n20 LARGEST DUPLICATE GROUPS")
print("=" * 65)

display(
    duplicate_group_summary[
        ["Duplicate_Group_ID", "Number_Of_Features", "Features"]
    ]
    .head(20)
)


# Attach feature dictionary metadata for the largest groups
metadata_columns = [
    col for col in [
        "Feature",
        "Source_Sensor",
        "Feature_Group",
        "Description",
        "Units",
        "Time_Window"
    ]
    if col in feature_dictionary.columns
]

largest_group_features = [
    feature
    for group in duplicate_groups[:10]
    for feature in group
]

print("\nFEATURE DICTIONARY DETAILS FOR THE 10 LARGEST GROUPS")
print("=" * 65)

display(
    feature_dictionary[
        feature_dictionary["Feature"].isin(largest_group_features)
    ][metadata_columns]
    .sort_values(["Source_Sensor", "Feature"])
    .reset_index(drop=True)
)

### Step 9E: Grouping Exact Duplicate Features – Results

**Observed results**

The 1,060 exact duplicate pairs were consolidated into connected groups.

- Exact duplicate pairs: 1,060.
- Duplicate groups: 107.
- Unique features involved: 411.
- Potential redundant features if retaining one feature per group: 304.
- Largest duplicate group: 16 features.

**Duplicate group size distribution**

| Features per group | Number of groups |
|---:|---:|
| 2 | 41 |
| 3 | 31 |
| 4 | 15 |
| 5 | 4 |
| 6 | 4 |
| 8 | 4 |
| 9 | 4 |
| 16 | 4 |

**Important observations**

1. Four of the largest groups contain 16 EI AQI observation-count features each.
2. PRVI groups contain identical observation counts across different physiological measurements.
3. MI groups contain identical counts across gait and movement statistics.
4. BRI and SQI also contain groups of identical historical observation counts.

**Interpretation**

Much of the exact duplication appears to originate from observation-count features. Different statistics calculated from the same underlying sensor observations can share identical historical observation counts.

This does not imply that their underlying physiological measurements are identical.

**Conclusion**

The analysis identified 107 duplicate groups involving 411 features. Retaining one representative per group would theoretically eliminate 304 redundant columns. However, no features have been removed. The semantic meaning of each group must be checked before deciding which feature to retain.

### Step 9F: Semantic Classification of Duplicate Feature Groups

**Objective**

Investigate the meaning of features within each exact duplicate group before deciding how to handle redundancy.

**Approach**

For each duplicate group, inspect its feature names and feature dictionary metadata, including source sensor, feature group, description, units and time window.

Classify whether the group consists of:
- Observation-count features.
- Validity or availability features.
- Actual sensor measurement features.
- Other derived features.

**Why this is necessary**

Exact numerical duplication does not necessarily mean two features have the same clinical interpretation. For example, identical observation counts across different physiological measurements do not make the measurements themselves interchangeable.

**Conclusion**

This assessment will support an interpretable redundancy-reduction strategy that prioritizes retaining meaningful clinical and sensor information.

In [ ]:
# STEP 9F: Classify exact duplicate groups

import re

# Create a lookup for feature dictionary metadata
feature_metadata = feature_dictionary.set_index("Feature")

semantic_group_records = []

for group_number, group in enumerate(duplicate_groups, start=1):

    group_metadata = (
        feature_dictionary[
            feature_dictionary["Feature"].isin(group)
        ].copy()
    )

    names = group_metadata["Feature"].tolist()

    # Classify features based on their names
    if all(
        re.search(r"(_count|observation_count)$", name)
        for name in names
    ):
        feature_type = "Observation count"

    elif all(
        ("validity" in name or "availability" in name)
        for name in names
    ):
        feature_type = "Validity / availability"

    elif all(
        re.search(r"(_mean|_median|_std|_min|_max)$", name)
        for name in names
    ):
        feature_type = "Measurement statistic"

    else:
        feature_type = "Mixed / other derived features"

    semantic_group_records.append({
        "Duplicate_Group_ID": f"DUP_{group_number:03d}",
        "Number_Of_Features": len(group),
        "Feature_Type": feature_type,
        "Sensors": sorted(
            group_metadata["Source_Sensor"].dropna().unique().tolist()
        ),
        "Feature_Groups": sorted(
            group_metadata["Feature_Group"].dropna().unique().tolist()
        ),
        "Units": sorted(
            group_metadata["Units"].dropna().unique().tolist()
        ),
        "Features": names
    })

semantic_duplicate_summary = pd.DataFrame(
    semantic_group_records
)

print("DUPLICATE GROUP SEMANTIC CLASSIFICATION")
print("=" * 65)

display(
    semantic_duplicate_summary[
        "Feature_Type"
    ].value_counts().rename_axis("Feature_Type")
    .reset_index(name="Number_Of_Groups")
)

print("\nDUPLICATE GROUPS BY SENSOR")
print("=" * 65)

sensor_group_summary = (
    semantic_duplicate_summary
    .explode("Sensors")
    .groupby("Sensors")
    .size()
    .sort_values(ascending=False)
    .rename("Number_Of_Groups")
    .reset_index()
)

display(sensor_group_summary)

print("\nDUPLICATE GROUPS WITH MIXED FEATURE TYPES")
print("=" * 65)

display(
    semantic_duplicate_summary[
        semantic_duplicate_summary["Feature_Type"]
        == "Mixed / other derived features"
    ][
        [
            "Duplicate_Group_ID",
            "Number_Of_Features",
            "Sensors",
            "Feature_Groups",
            "Units",
            "Features"
        ]
    ]
)

print("\nALL DUPLICATE GROUPS")
print("=" * 65)

display(
    semantic_duplicate_summary[
        [
            "Duplicate_Group_ID",
            "Number_Of_Features",
            "Feature_Type",
            "Sensors",
            "Feature_Groups",
            "Units"
        ]
    ]
)

### Step 9G: Investigating Measurement and Mixed Duplicate Groups

**Objective**

Investigate exact duplicate groups containing measurement statistics and mixed or derived features.

**Approach**

For each feature in these groups, inspect its feature dictionary metadata and calculate its missingness percentage in the known-target dataset.

**Reason**

Two features can have identical numerical values without representing the same clinical concept. Different units, formulas, source measurements, or time windows may indicate that they should not be treated as interchangeable.

**Important limitation**

The duplicate relationships were identified using the complete known-target dataset for exploratory analysis. Any final feature-selection decision must be re-evaluated using training participants only within each validation fold to prevent information from validation participants influencing preprocessing decisions.

No features are removed in this step.

In [ ]:
# STEP 9G: Inspect measurement and mixed duplicate groups

groups_to_inspect = semantic_duplicate_summary[
    semantic_duplicate_summary["Feature_Type"].isin([
        "Measurement statistic",
        "Mixed / other derived features"
    ])
].copy()

inspection_records = []

for _, group_row in groups_to_inspect.iterrows():

    group_id = group_row["Duplicate_Group_ID"]
    group_features = group_row["Features"]

    for feature in group_features:

        metadata_row = feature_metadata.loc[feature]

        missing_percentage = (
            known_target_dataset[feature].isna().mean() * 100
        )

        inspection_records.append({
            "Duplicate_Group_ID": group_id,
            "Feature_Type": group_row["Feature_Type"],
            "Feature": feature,
            "Source_Sensor": metadata_row.get("Source_Sensor"),
            "Feature_Group": metadata_row.get("Feature_Group"),
            "Description": metadata_row.get("Description"),
            "Formula": metadata_row.get("Formula"),
            "Units": metadata_row.get("Units"),
            "Time_Window": metadata_row.get("Time_Window"),
            "Missing_Percentage": round(missing_percentage, 2)
        })

duplicate_semantic_inspection = pd.DataFrame(
    inspection_records
)

print("GROUPS UNDER INVESTIGATION")
print("=" * 65)
print("Groups:", groups_to_inspect.shape[0])
print("Features:", duplicate_semantic_inspection.shape[0])

print("\nGROUP TYPE DISTRIBUTION")
display(
    groups_to_inspect["Feature_Type"]
    .value_counts()
    .rename_axis("Feature_Type")
    .reset_index(name="Number_Of_Groups")
)

print("\nDETAILED FEATURE INSPECTION")
print("=" * 65)

display(
    duplicate_semantic_inspection
    .sort_values(["Duplicate_Group_ID", "Feature"])
    .reset_index(drop=True)
)

### Step 9G: Investigating Measurement and Mixed Duplicate Groups – Results

**Observed results**

The investigation covered 55 duplicate groups containing 143 features:

- Measurement-statistic groups: 32.
- Mixed/other derived-feature groups: 23.
- Total features inspected: 143.

The feature dictionary was examined for source sensor, feature group, description, formula, units, time window, and missingness.

**Important observations**

1. Some daily MI, SHRI, and SQI groups contain both observation-count features and source-defined validity indicators.
2. Some BRI and SQI duplicate groups contain different derived measurement features.
3. Some rolling SQI features have high missingness (approximately 93.07% in the displayed examples).
4. In group DUP_106, features named with the `_pct_change` suffix have a feature-dictionary formula describing an absolute difference. This requires verification against the feature-engineering implementation.

**Interpretation**

Identical numerical values and missingness patterns do not automatically establish that two features are semantically interchangeable.

The feature dictionary also requires a consistency check because feature names, descriptions, and formulas should accurately represent the calculations performed.

**Conclusion**

No features were removed. A metadata-consistency and calculation-verification audit is required before selecting representative features from duplicate groups.

### Step 9H: Duplicate Group Metadata Consistency Audit

**Objective**

Determine whether features with identical observed values also have consistent metadata.

**Approach**

For every exact duplicate group, compare:
- Source sensor
- Feature group
- Description
- Formula
- Units
- Time window

Groups with different metadata will be flagged for further review.

**Why this matters**

Features can be numerically identical in the current dataset but represent different concepts. Metadata differences help identify cases where automatic feature removal could discard useful information or hide documentation errors.

No features are removed in this step.

In [ ]:
# STEP 9H: Audit metadata consistency within duplicate groups

metadata_columns_to_check = [
    "Source_Sensor",
    "Feature_Group",
    "Description",
    "Formula",
    "Units",
    "Time_Window"
]

metadata_audit_records = []

for _, group_row in semantic_duplicate_summary.iterrows():

    group_id = group_row["Duplicate_Group_ID"]
    group_features = group_row["Features"]

    group_metadata = (
        feature_dictionary[
            feature_dictionary["Feature"].isin(group_features)
        ]
    )

    record = {
        "Duplicate_Group_ID": group_id,
        "Number_Of_Features": len(group_features),
        "Features": group_features
    }

    for column in metadata_columns_to_check:
        record[f"Distinct_{column}"] = (
            group_metadata[column].nunique(dropna=False)
        )

    metadata_audit_records.append(record)

metadata_consistency_audit = pd.DataFrame(
    metadata_audit_records
)

print("METADATA CONSISTENCY SUMMARY")
print("=" * 65)

for column in metadata_columns_to_check:
    count = (
        metadata_consistency_audit[
            f"Distinct_{column}"
        ] > 1
    ).sum()

    print(f"Groups with differing {column}: {count}")

print("\nGROUPS WITH DIFFERING UNITS OR FORMULAS")
print("=" * 65)

display(
    metadata_consistency_audit[
        (
            metadata_consistency_audit["Distinct_Units"] > 1
        )
        |
        (
            metadata_consistency_audit["Distinct_Formula"] > 1
        )
    ][
        [
            "Duplicate_Group_ID",
            "Number_Of_Features",
            "Distinct_Units",
            "Distinct_Formula",
            "Features"
        ]
    ].to_string(index=False)
)

print("\nPERCENTAGE CHANGE FORMULA CHECK")
print("=" * 65)

pct_change_features = feature_dictionary[
    feature_dictionary["Feature"].str.contains(
        "_pct_change",
        case=False,
        na=False
    )
].copy()

display(
    pct_change_features[
        [
            "Feature",
            "Description",
            "Formula",
            "Units"
        ]
    ].head(30).to_string(index=False)
)

### Step 9H: Duplicate Group Metadata Consistency Audit – Results

**Observed results**

The metadata consistency audit covered all 107 exact duplicate groups.

| Metadata field | Groups with differing metadata |
|---|---:|
| Source sensor | 1 |
| Feature group | 0 |
| Description | 7 |
| Formula | 7 |
| Units | 7 |
| Time window | 0 |

**Important observations**

1. Seven groups contain both observation-count features and source-defined validity indicators.
2. These groups have different descriptions, formulas, and units despite identical observed values.
3. One duplicate group contains features with differing source-sensor metadata.
4. The feature dictionary describes features ending in `_pct_change` using an absolute-difference formula.

**Interpretation**

Identical values in the current dataset do not guarantee semantic equivalence. In particular, observation counts and validity indicators represent different concepts, even when their values happen to match.

The percentage-change documentation may also be inconsistent with the actual feature-engineering implementation.

**Conclusion**

No features were removed. The feature-engineering code must be inspected to verify percentage-change calculations, and the cross-sensor duplicate group must be reviewed before feature-selection decisions are made.

In [ ]:
# STEP 9I: Inspect original feature-engineering formulas

import json

NOTEBOOK_04_PATH = (
    PROJECT_ROOT / "notebooks" / "04_feature_engineering.ipynb"
)

with open(NOTEBOOK_04_PATH, "r", encoding="utf-8") as file:
    notebook_04 = json.load(file)

print("CODE CELLS CONTAINING PERCENTAGE-CHANGE LOGIC")
print("=" * 70)

for cell_number, cell in enumerate(notebook_04["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if "pct_change" in source.lower() or "percentage_change" in source.lower():

        print(f"\nNOTEBOOK CELL: {cell_number}")
        print("-" * 70)
        print(source)

print("\nGROUPS WITH DIFFERING SOURCE SENSORS")
print("=" * 70)

display(
    metadata_consistency_audit[
        metadata_consistency_audit["Distinct_Source_Sensor"] > 1
    ][
        [
            "Duplicate_Group_ID",
            "Number_Of_Features",
            "Features"
        ]
    ].to_string(index=False)
)

print("\nMETADATA FOR THE CROSS-SENSOR GROUP")
print("=" * 70)

cross_sensor_group_ids = metadata_consistency_audit.loc[
    metadata_consistency_audit["Distinct_Source_Sensor"] > 1,
    "Duplicate_Group_ID"
].tolist()

cross_sensor_features = []

for group_id in cross_sensor_group_ids:
    group_features = semantic_duplicate_summary.loc[
        semantic_duplicate_summary["Duplicate_Group_ID"] == group_id,
        "Features"
    ].iloc[0]

    cross_sensor_features.extend(group_features)

display(
    feature_dictionary[
        feature_dictionary["Feature"].isin(cross_sensor_features)
    ][
        [
            "Feature",
            "Source_Sensor",
            "Description",
            "Formula",
            "Units",
            "Time_Window"
        ]
    ].to_string(index=False)
)

### Step 9I: Verifying Percentage-Change Calculations

**Objective**

Check whether the underlying sensor features used for percentage-change calculations contain negative values.

**Reason**

The original implementation divides by the previous value, whereas the feature dictionary describes division by the absolute previous value. These formulas produce different results when the previous value is negative.

**Approach**

For every percentage-change feature, identify its underlying daily feature and inspect its values across the complete merged dataset.

Record:
- Number of non-missing observations.
- Minimum and maximum values.
- Number of negative observations.
- Number of zero observations.

**Interpretation**

If an underlying feature contains no negative values, the two formulas are equivalent for its observed values, except for the handling of zero, which is already defined as missing.

If negative values exist, the implementation and documentation require further investigation.

No feature calculations are modified in this step.

In [ ]:
# STEP 9I: Audit source values for percentage-change features

import re
import numpy as np
import pandas as pd

pct_change_columns = [
    column for column in model_dataset.columns
    if re.search(r"_rolling_(3|7|14)d_pct_change$", column)
]

pct_source_audit_records = []

for pct_column in pct_change_columns:

    match = re.match(
        r"^(.*)_rolling_(3|7|14)d_pct_change$",
        pct_column
    )

    source_feature = match.group(1)

    if source_feature not in model_dataset.columns:
        pct_source_audit_records.append({
            "Percentage_Change_Feature": pct_column,
            "Source_Feature": source_feature,
            "Status": "Source feature not found"
        })
        continue

    values = pd.to_numeric(
        model_dataset[source_feature],
        errors="coerce"
    ).dropna()

    pct_source_audit_records.append({
        "Percentage_Change_Feature": pct_column,
        "Source_Feature": source_feature,
        "Status": "Checked",
        "Non_Missing_Values": len(values),
        "Minimum": values.min() if len(values) else np.nan,
        "Maximum": values.max() if len(values) else np.nan,
        "Negative_Count": int((values < 0).sum()),
        "Zero_Count": int((values == 0).sum())
    })

pct_source_audit = pd.DataFrame(pct_source_audit_records)

print("PERCENTAGE-CHANGE SOURCE AUDIT")
print("=" * 70)
print("Percentage-change features found:", len(pct_change_columns))
print("Source features checked:",
      (pct_source_audit["Status"] == "Checked").sum())

print("\nSOURCE FEATURES CONTAINING NEGATIVE VALUES")
print("=" * 70)

negative_sources = pct_source_audit[
    (pct_source_audit["Status"] == "Checked")
    & (pct_source_audit["Negative_Count"] > 0)
]

display(
    negative_sources[
        [
            "Source_Feature",
            "Non_Missing_Values",
            "Minimum",
            "Maximum",
            "Negative_Count",
            "Zero_Count"
        ]
    ].drop_duplicates("Source_Feature")
    .sort_values("Source_Feature")
)

print("\nOVERALL AUDIT SUMMARY")
print("=" * 70)
print("Source features with negative values:",
      negative_sources["Source_Feature"].nunique())
print("Source features without negative values:",
      pct_source_audit.loc[
          pct_source_audit["Status"] == "Checked",
          "Source_Feature"
      ].nunique() - negative_sources["Source_Feature"].nunique())

print("\nNEGATIVE VALUE COUNTS BY SENSOR")
print("=" * 70)

negative_sources_by_sensor = (
    negative_sources.assign(
        Sensor=negative_sources["Source_Feature"].str.split("_").str[0]
    )
    .groupby("Sensor")["Source_Feature"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    negative_sources_by_sensor.rename(
        "Negative_Source_Feature_Count"
    ).reset_index()
)

### Step 9I (continued): Independent Validation of SQI Percentage-Change Features

**Objective:** Determine which percentage-change formula was used in the saved features for `SQI_SE_mean`.

The source audit identified negative values in this feature. Consequently, the two percentage-change formulas may produce different results.

**Approach:**
- Recalculate historical percentage changes for the 3-, 7-, and 14-day windows.
- Exclude the current date from historical calculations.
- Compare the original implementation, which divides by the signed previous value, against the feature dictionary formula, which divides by the absolute previous value.
- Compare both results with the saved feature values.

**Interpretation:** This validation will establish whether the saved features match the implementation or the documented formula. No feature values will be changed during this audit.

In [ ]:
import numpy as np
import pandas as pd

source_feature = "SQI_SE_mean"
windows = [3, 7, 14]

def calculate_signed_pct_change(values):
    valid = values[np.isfinite(values)]

    if len(valid) < 2:
        return np.nan

    previous = valid[-2]
    latest = valid[-1]

    if previous == 0:
        return np.nan

    return ((latest - previous) / previous) * 100


def calculate_absolute_pct_change(values):
    valid = values[np.isfinite(values)]

    if len(valid) < 2:
        return np.nan

    previous = valid[-2]
    latest = valid[-1]

    if previous == 0:
        return np.nan

    return ((latest - previous) / abs(previous)) * 100


audit_results = []

for window in windows:

    pct_column = f"{source_feature}_rolling_{window}d_pct_change"

    if pct_column not in model_dataset.columns:
        print(f"Missing feature column: {pct_column}")
        continue

    signed_values = []
    absolute_values = []
    saved_values = []
    previous_values = []

    for participant, group in model_dataset.groupby("Participant_ID"):

        group = group.sort_values("Date").copy()
        group = group.set_index("Date")

        historical = group[source_feature].shift(1)

        signed = historical.rolling(
            f"{window}D",
            min_periods=1
        ).apply(calculate_signed_pct_change, raw=True)

        absolute = historical.rolling(
            f"{window}D",
            min_periods=1
        ).apply(calculate_absolute_pct_change, raw=True)

        # Recover the last two historical non-missing values
        for date in group.index:

            start_date = date - pd.Timedelta(days=window)

            historical_values = group.loc[
                (group.index >= start_date) &
                (group.index < date),
                source_feature
            ].dropna()

            previous = (
                historical_values.iloc[-2]
                if len(historical_values) >= 2
                else np.nan
            )

            signed_values.append(signed.loc[date])
            absolute_values.append(absolute.loc[date])
            saved_values.append(group.loc[date, pct_column])
            previous_values.append(previous)

    comparison = pd.DataFrame({
        "Signed_Formula": signed_values,
        "Absolute_Formula": absolute_values,
        "Saved_Feature": saved_values,
        "Previous_Value": previous_values
    })

    valid_saved = comparison["Saved_Feature"].notna()

    signed_match = np.isclose(
        comparison.loc[valid_saved, "Saved_Feature"],
        comparison.loc[valid_saved, "Signed_Formula"],
        rtol=1e-7,
        atol=1e-9,
        equal_nan=False
    )

    absolute_match = np.isclose(
        comparison.loc[valid_saved, "Saved_Feature"],
        comparison.loc[valid_saved, "Absolute_Formula"],
        rtol=1e-7,
        atol=1e-9,
        equal_nan=False
    )

    audit_results.append({
        "Window": f"{window} days",
        "Saved_Non_Missing": int(valid_saved.sum()),
        "Matches_Signed_Formula": int(signed_match.sum()),
        "Matches_Absolute_Formula": int(absolute_match.sum()),
        "Previous_Value_Negative": int(
            (comparison["Previous_Value"] < 0).sum()
        ),
        "Matches_Neither": int(
            (~signed_match & ~absolute_match).sum()
        )
    })

pct_change_validation = pd.DataFrame(audit_results)

print("SQI_SE_mean PERCENTAGE-CHANGE VALIDATION")
print("=" * 75)
display(pct_change_validation)

### Step 9I: Verification of Percentage-Change Calculations

**Objective**

Investigate whether the underlying daily source features used in percentage-change calculations contain negative values and verify whether the saved rolling percentage-change features follow the implemented formula.

**Observed results**

- A total of 192 percentage-change features were identified across the 3-, 7-, and 14-day windows.
- These corresponded to 64 underlying source features.
- Only one source feature contained negative values: `SQI_SE_mean`.
- This feature contained 15 negative observations among 507 non-missing observations.
- All 447, 794, and 1,111 non-missing saved percentage-change values for the respective 3-, 7-, and 14-day windows matched the signed-denominator formula.
- Some values did not match the absolute-denominator formula, specifically where the previous source value was negative.
- No saved values failed both formula checks.

**Interpretation**

The feature-engineering implementation uses the previous historical value directly as the denominator:

`((Latest - Previous) / Previous) * 100`

It returns a missing value when the previous value is zero.

The feature dictionary and an independent validation cell used the absolute value of the previous observation in the denominator. This discrepancy affects the interpretation of percentage changes for negative source values, particularly `SQI_SE_mean`.

**Decision**

- Retain the existing calculated percentage-change features.
- Correct the feature dictionary formula to match the actual implementation.
- Correct the independent validation formula to use the signed previous value.
- Do not regenerate the feature datasets because the saved calculations have been verified against the implementation.
- Interpret percentage changes for negative-valued source features cautiously.

No feature removal or imputation was performed in this step.

In [ ]:
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features" / "feature_dictionary.csv"
)

saved_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)

pct_change_dictionary = saved_dictionary[
    saved_dictionary["Feature"].str.endswith("_pct_change")
]

print("Percentage-change dictionary entries:", len(pct_change_dictionary))

print("\nUnique percentage-change formulas:")
print(pct_change_dictionary["Formula"].unique())

### Step 9I – Feature-Engineering Formula and Dictionary Verification

**Objective:** Verify that percentage-change features are calculated and documented consistently.

**Investigation and findings:**

- Inspected the percentage-change implementation in Notebook 04.
- Confirmed that the calculation uses the signed previous value as the denominator:

  \[
  \text{Percentage Change} =
  \frac{\text{Latest} - \text{Previous}}{\text{Previous}}\times100
  \]

- When the previous value is zero, the result is recorded as `NaN`.
- Identified a feature-dictionary suffix-parsing issue: features ending in `_pct_change` were incorrectly classified as ordinary `change` features.
- Corrected the dictionary-generation logic to identify `_pct_change` before `_change`.
- Regenerated and saved the feature dictionary.

**Final verification results:**

| Validation | Result |
|---|---:|
| Total dictionary entries | 1,916 |
| Percentage-change entries | 192 |
| Entries with corrected formula | 192 |
| Entries with corrected description | 192 |
| Missing dictionary values | 0 |
| Saved CSV successfully verified | Yes |

**Interpretation:**

The percentage-change calculations were already using the intended signed-denominator formula. The issue was limited to their documentation in the feature dictionary. The metadata has now been corrected and verified against the saved CSV.

No predictor values were modified, no features were removed, and no feature calculations were regenerated as part of this correction.

## Step 10 – Feature Reduction Strategy

### Objective

Define a leakage-safe strategy for reducing the 1,916 candidate predictor features before model training.

### Why feature reduction is needed

The current dataset contains 1,916 candidate features but only 24 participants with known target labels. This creates a high-dimensional modelling problem, where the number of predictors is large relative to the number of independent participants.

Potential consequences include:

- Overfitting.
- Redundant information.
- Unstable feature selection.
- Increased computational complexity.
- Reduced model interpretability.

### Feature-reduction principles

1. Preserve the original feature dataset and feature dictionary.
2. Exclude identifiers, dates, target labels, and event-related metadata from predictors.
3. Fit missingness filtering using training participants only.
4. Fit constant and near-constant feature filtering using training participants only.
5. Investigate exact duplicate features within training data only.
6. Fit imputation and scaling using training data only, where required by the selected model.
7. Apply the fitted transformations to the corresponding validation participants without refitting.
8. Repeat the process independently for every validation fold.

### Important limitation

Previously identified missingness, constant-feature, near-constant-feature, and duplicate-feature statistics were calculated across all known-target observations. They are exploratory findings, not final feature-removal decisions.

No predictor features will be removed at this stage.

### Intended outcome

Develop a reproducible preprocessing pipeline that performs feature reduction independently within each training fold, preventing information from validation participants from influencing feature selection.


In [ ]:
# Identify predictor columns
excluded_columns = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

candidate_features = [
    col for col in known_target_dataset.columns
    if col not in excluded_columns
]

# Inspect data types
feature_dtypes = (
    known_target_dataset[candidate_features]
    .dtypes
    .astype(str)
    .value_counts()
)

print("FEATURE DATA TYPE DISTRIBUTION")
print("=" * 60)
print("Total candidate features:", len(candidate_features))
print("\nData type counts:")
print(feature_dtypes)

print("\nFEATURES BY DATA TYPE")
print("=" * 60)

for dtype in known_target_dataset[candidate_features].dtypes.unique():
    columns = known_target_dataset[candidate_features].select_dtypes(
        include=[dtype]
    ).columns

    print(f"\n{dtype}: {len(columns)} features")
    print(list(columns[:10]))

### Step 10C – Investigation of Object-Type Features

**Objective:** Determine why 14 candidate predictor features have the `object` data type and whether they contain Boolean values, strings, or missing values.

**Reason:** The feature names suggest that these columns represent data-quality conflict indicators. Their actual values must be inspected before deciding whether they should be converted to Boolean or another appropriate data type.

**Important:** No type conversions or feature removals will be performed during this investigation.

In [ ]:
# Identify object-type predictor features
object_columns = known_target_dataset[candidate_features].select_dtypes(
    include=["object"]
).columns.tolist()

print("OBJECT-TYPE FEATURE INSPECTION")
print("=" * 60)
print("Number of object features:", len(object_columns))

for col in object_columns:
    print(f"\nFeature: {col}")
    print("Data type:", known_target_dataset[col].dtype)
    print("Unique values including missing:", 
          known_target_dataset[col].value_counts(dropna=False).to_dict())
    print("Python value types:",
          known_target_dataset[col].dropna().map(type).value_counts().to_dict())

### Step 10D – Standardisation of Boolean Features

**Objective:** Convert the 14 object-type conflict indicators into Pandas nullable Boolean columns.

**Reason:** These columns contain only Boolean values and missing values. Using the nullable Boolean data type provides a consistent representation while preserving missingness.

**Transformation policy:**

- `True` remains `True`.
- `False` remains `False`.
- `None` becomes `pd.NA`.
- No missing values are imputed.
- The original `known_target_dataset` remains unchanged; conversion is performed on a separate modelling copy.

This is a deterministic data-type correction, not a statistical transformation or feature-selection operation.

In [ ]:
# Create a separate modelling copy
known_target_model_dataset = known_target_dataset.copy()

# Convert object-type Boolean indicators to nullable Boolean
for col in object_columns:
    known_target_model_dataset[col] = (
        known_target_model_dataset[col].astype("boolean")
    )

# Verify conversion
print("BOOLEAN TYPE CONVERSION")
print("=" * 60)

print("Original dataset unchanged:", 
      all(
          known_target_dataset[col].dtype == "object"
          for col in object_columns
      ))

print("\nConverted data types:")
print(
    known_target_model_dataset[object_columns]
    .dtypes
    .value_counts()
)

print("\nMissing values before conversion:")
print(known_target_dataset[object_columns].isna().sum().sum())

print("\nMissing values after conversion:")
print(known_target_model_dataset[object_columns].isna().sum().sum())

print("\nBoolean value counts after conversion:")
for col in object_columns:
    print(f"\n{col}")
    print(
        known_target_model_dataset[col]
        .value_counts(dropna=False)
        .to_dict()
    )

### Step 10E – Final Data-Type Validation

**Objective:** Verify the data types of all candidate predictor features after converting the 14 Boolean conflict indicators.

**Reason:** Before constructing the modelling matrix, we need to identify any remaining unsupported or unexpected data types.

**Validation checks:**

- Total candidate features.
- Data-type distribution.
- Remaining object-type columns.
- Boolean-type columns.
- Numerical feature columns.

No features will be removed or transformed in this step.

In [ ]:
# Inspect all candidate feature data types after conversion

updated_dtypes = (
    known_target_model_dataset[candidate_features]
    .dtypes
    .astype(str)
    .value_counts()
)

print("FINAL FEATURE DATA TYPE DISTRIBUTION")
print("=" * 60)

print("Total candidate features:", len(candidate_features))
print("\nData type counts:")
print(updated_dtypes)

# Identify remaining object columns
remaining_object = (
    known_target_model_dataset[candidate_features]
    .select_dtypes(include=["object"])
    .columns.tolist()
)

# Identify Boolean columns
boolean_features = [
    col for col in candidate_features
    if str(known_target_model_dataset[col].dtype) == "boolean"
]

# Identify numeric columns
numeric_features = (
    known_target_model_dataset[candidate_features]
    .select_dtypes(include=["number"])
    .columns.tolist()
)

print("\nREMAINING OBJECT FEATURES:", len(remaining_object))
print(remaining_object)

print("\nBOOLEAN FEATURES:", len(boolean_features))
print("\nNUMERICAL FEATURES:", len(numeric_features))

print("\nTOTAL ACCOUNTED FOR:",
      len(remaining_object) + len(boolean_features) + len(numeric_features))

### Step 10F – Infinite Value Investigation

**Objective:** Identify positive and negative infinite values in the 1,902 numerical predictor features.

**Reason:** Infinite values can cause errors or unstable behaviour during statistical preprocessing and model fitting.

**Validation approach:**

- Inspect numerical predictor columns only.
- Count positive infinity (`+inf`) values.
- Count negative infinity (`-inf`) values.
- Identify affected features and their counts.
- Preserve the dataset until the investigation is complete.

**Important:** Infinite values will not automatically be replaced or removed. Their source and meaning will be investigated first.

In [ ]:
# Check numerical features for infinite values

numeric_data = known_target_model_dataset[numeric_features]

positive_inf_counts = np.isposinf(numeric_data).sum()
negative_inf_counts = np.isneginf(numeric_data).sum()

positive_inf_features = positive_inf_counts[
    positive_inf_counts > 0
].sort_values(ascending=False)

negative_inf_features = negative_inf_counts[
    negative_inf_counts > 0
].sort_values(ascending=False)

print("INFINITE VALUE INVESTIGATION")
print("=" * 60)

print("Total numerical features:", len(numeric_features))
print("Total positive infinity values:", int(positive_inf_counts.sum()))
print("Total negative infinity values:", int(negative_inf_counts.sum()))

print("\nFeatures containing positive infinity:",
      len(positive_inf_features))
print(positive_inf_features)

print("\nFeatures containing negative infinity:",
      len(negative_inf_features))
print(negative_inf_features)

print("\nTotal infinite values:",
      int(positive_inf_counts.sum() + negative_inf_counts.sum()))

### Step 10G – Numerical Feature Range Investigation

**Objective:** Examine the numerical ranges of all 1,902 predictor features.

**Reason:** Unusually large or small values may arise from genuine observations, data-quality issues, or feature-engineering calculations. We need to understand their distribution before deciding whether any additional validation is necessary.

**Investigation:**

- Minimum and maximum values.
- Mean and standard deviation.
- Median and interquartile range.
- Features with unusually large absolute ranges.

**Important considerations:**

- Statistical extremes are not automatically errors.
- Features have different units and scales, so raw magnitudes cannot be compared directly across unrelated measurements.
- No outlier removal, clipping, or scaling will be performed during

In [ ]:
# Numerical feature range investigation

numeric_summary = (
    known_target_model_dataset[numeric_features]
    .describe()
    .T
)

numeric_summary["IQR"] = (
    numeric_summary["75%"] - numeric_summary["25%"]
)

numeric_summary["Absolute_Max"] = (
    numeric_summary[["min", "max"]].abs().max(axis=1)
)

print("NUMERICAL FEATURE RANGE SUMMARY")
print("=" * 60)

print("Features analysed:", len(numeric_summary))

print("\nOverall summary of feature distributions:")
display(
    numeric_summary[
        ["mean", "std", "min", "25%", "50%", "75%", "max", "IQR"]
    ].describe().round(3)
)

print("\n10 features with the largest absolute values:")
display(
    numeric_summary.sort_values(
        "Absolute_Max", ascending=False
    )[
        ["count", "mean", "std", "min", "max", "Absolute_Max"]
    ].head(10).round(3)
)

### Step 10H – Investigation of Extreme Percentage-Change Values

**Objective:** Identify the observations responsible for unusually large percentage-change values.

**Reason:** Percentage changes can become extremely large when the previous historical value is close to zero. Such values may be mathematically valid but potentially unstable as predictors.

**Investigation:**

- Identify percentage-change features with absolute values above 1,000% and 10,000%.
- Count extreme observations by feature.
- Inspect the largest absolute percentage-change observations.
- Record the associated participant and feature date for traceability.

**Interpretation policy:**

Large percentage changes will not automatically be treated as data errors. Their source values and calculation history must be investigated before deciding whether any transformation or exclusion is appropriate.

No values will be modified or removed in this step.

In [ ]:
# Identify percentage-change features
pct_change_features = [
    col for col in numeric_features
    if col.endswith("_pct_change")
]

print("PERCENTAGE-CHANGE EXTREME VALUE INVESTIGATION")
print("=" * 65)
print("Percentage-change features:", len(pct_change_features))

# Collect extreme-value counts and observations
extreme_summary = []
extreme_records = []

for col in pct_change_features:

    values = known_target_model_dataset[col].dropna()

    extreme_summary.append({
        "Feature": col,
        "Non_Missing": len(values),
        "Above_1000_Percent": int((values.abs() > 1000).sum()),
        "Above_10000_Percent": int((values.abs() > 10000).sum()),
        "Maximum_Absolute_Value": (
            float(values.abs().max()) if len(values) else np.nan
        )
    })

    extreme_indices = values.abs().nlargest(5).index

    for idx in extreme_indices:
        extreme_records.append({
            "Feature": col,
            "Participant_ID": known_target_model_dataset.loc[
                idx, "Participant_ID"
            ],
            "Date": known_target_model_dataset.loc[idx, "Date"],
            "Percentage_Change": known_target_model_dataset.loc[idx, col]
        })

extreme_summary = pd.DataFrame(extreme_summary)
extreme_records = pd.DataFrame(extreme_records)

# Display features with extreme values
print("\nFEATURES WITH VALUES ABOVE 1,000%")
display(
    extreme_summary[
        extreme_summary["Above_1000_Percent"] > 0
    ].sort_values(
        "Maximum_Absolute_Value", ascending=False
    ).head(20)
)

print("\nTOTAL EXTREME OBSERVATIONS")
print(
    "Above 1,000%:",
    int(extreme_summary["Above_1000_Percent"].sum())
)
print(
    "Above 10,000%:",
    int(extreme_summary["Above_10000_Percent"].sum())
)

# Display the 20 largest absolute observations across all features
print("\n20 LARGEST ABSOLUTE PERCENTAGE-CHANGE OBSERVATIONS")

extreme_records["Absolute_Value"] = (
    extreme_records["Percentage_Change"].abs()
)

display(
    extreme_records.sort_values(
        "Absolute_Value", ascending=False
    ).head(20).drop(columns="Absolute_Value")
)

### Step 10I – Trace Extreme Percentage Changes to Source Measurements

**Objective:** Investigate the underlying historical measurements responsible for extreme percentage-change features.

**Selected observation:**

- Participant: SPA-029
- Feature date: 2024-11-26
- Feature: EI_Lux_mean_rolling_14d_pct_change
- Observed percentage change: 19,763.33%

**Investigation approach:**

1. Load the daily sensor feature dataset.
2. Identify the source feature corresponding to EI light measurements.
3. Retrieve the participant's historical observations before the selected feature date.
4. Identify the latest two valid historical values.
5. Independently calculate the percentage change.
6. Compare the calculated value with the saved engineered feature.

**Important:** The current feature date is excluded because the percentage-change feature uses historical observations only.

In [ ]:
# Load daily sensor features
DAILY_FEATURES_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "daily_sensor_features.parquet"
)

daily_features = duckdb.execute(
    "SELECT * FROM read_parquet(?)",
    [str(DAILY_FEATURES_PATH)]
).fetchdf()

print("DAILY SENSOR FEATURE DATA")
print("=" * 60)
print("Shape:", daily_features.shape)

# Inspect EI light-related columns
ei_lux_columns = [
    col for col in daily_features.columns
    if "Lux" in col
]

print("\nEI light-related columns:")
print(ei_lux_columns)

# Inspect feature dictionary entry
dictionary_path = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "feature_dictionary.csv"
)

saved_dictionary = pd.read_csv(dictionary_path)

display(
    saved_dictionary[
        saved_dictionary["Feature"]
        == "EI_Lux_mean_rolling_14d_pct_change"
    ]
)

In [ ]:
# Trace the extreme EI light percentage change

participant = "SPA-029"
test_date = pd.Timestamp("2024-11-26", tz="UTC")
source_feature = "EI_Lux_mean"
engineered_feature = "EI_Lux_mean_rolling_14d_pct_change"

# Standardise dates
daily_features["Date"] = pd.to_datetime(
    daily_features["Date"], utc=True
)

# Select this participant's historical observations
historical = daily_features[
    (daily_features["Participant_ID"] == participant) &
    (daily_features["Date"] < test_date) &
    (daily_features["Date"] >= test_date - pd.Timedelta(days=14))
].copy()

# Keep valid historical measurements
historical = historical[
    historical[source_feature].notna()
].sort_values("Date")

print("HISTORICAL EI LIGHT MEASUREMENTS")
print("=" * 60)

display(
    historical[["Participant_ID", "Date", source_feature]]
)

# Retrieve the latest two valid historical measurements
last_two = historical.tail(2)

if len(last_two) == 2:
    previous = last_two.iloc[0][source_feature]
    latest = last_two.iloc[1][source_feature]

    if previous != 0:
        calculated_change = (
            (latest - previous) / previous
        ) * 100
    else:
        calculated_change = np.nan

    print("\nLATEST TWO HISTORICAL VALUES")
    print("Previous value:", previous)
    print("Latest value:", latest)
    print("Calculated percentage change:", calculated_change)

else:
    print("\nNot enough valid historical measurements.")

# Retrieve the saved engineered feature
saved_row = known_target_model_dataset[
    (known_target_model_dataset["Participant_ID"] == participant) &
    (known_target_model_dataset["Date"] == test_date)
]

print("\nSAVED ENGINEERED FEATURE")
if not saved_row.empty:
    print(
        saved_row[engineered_feature].iloc[0]
    )
else:
    print("Participant-date combination not found.")

### Step 10J – Investigation of Small Denominators in Extreme Percentage Changes

**Objective:** Determine whether extreme percentage-change values are associated with very small previous historical measurements.

**Reason:** Percentage-change calculations divide the difference between two observations by the previous observation. When the previous value is close to zero, the resulting percentage can become disproportionately large.

**Investigation:**

- Examine percentage-change features exceeding 10,000% in absolute magnitude.
- Trace the latest two valid historical measurements for selected observations.
- Record the previous value, latest value, absolute difference, and calculated percentage change.
- Compare denominator magnitudes across extreme observations.

**Interpretation:** This investigation evaluates numerical stability, not whether a measurement is clinically abnormal or a data-quality error.

No feature values will be changed or removed.

In [ ]:
# Investigate the denominator effect for extreme observations

extreme_cases = extreme_records.copy()

extreme_cases = extreme_cases[
    extreme_cases["Percentage_Change"].abs() > 10000
].copy()

# Remove repeated feature observations for the same
# participant, date and source measurement
extreme_cases["Source_Feature"] = (
    extreme_cases["Feature"]
    .str.replace(r"_rolling_\d+d_pct_change$", "", regex=True)
)

extreme_cases = extreme_cases.drop_duplicates(
    subset=["Participant_ID", "Date", "Source_Feature"]
)

print("EXTREME CASES SELECTED")
print("=" * 60)
print("Unique participant-date-source combinations:",
      len(extreme_cases))

denominator_results = []

for _, case in extreme_cases.iterrows():

    participant = case["Participant_ID"]
    test_date = pd.Timestamp(case["Date"])
    source_feature = case["Source_Feature"]

    history = daily_features[
        (daily_features["Participant_ID"] == participant) &
        (daily_features["Date"] < test_date) &
        (daily_features["Date"] >= test_date - pd.Timedelta(days=14))
    ][["Date", source_feature]].dropna().sort_values("Date")

    last_two = history.tail(2)

    if len(last_two) < 2:
        continue

    previous = float(last_two.iloc[0][source_feature])
    latest = float(last_two.iloc[1][source_feature])

    calculated_change = (
        ((latest - previous) / previous) * 100
        if previous != 0 else np.nan
    )

    denominator_results.append({
        "Participant_ID": participant,
        "Date": test_date,
        "Source_Feature": source_feature,
        "Previous_Value": previous,
        "Latest_Value": latest,
        "Absolute_Change": latest - previous,
        "Calculated_Percentage_Change": calculated_change
    })

denominator_audit = pd.DataFrame(denominator_results)

print("\nDENOMINATOR AUDIT")
print("Cases successfully traced:", len(denominator_audit))

display(
    denominator_audit.sort_values(
        "Calculated_Percentage_Change",
        key=lambda x: x.abs(),
        ascending=False
    ).head(20)
)

In [ ]:
# Verify extreme percentage changes using their actual windows

import re

extreme_feature_cases = extreme_records[
    extreme_records["Percentage_Change"].abs() > 10000
].drop_duplicates(
    subset=["Feature", "Participant_ID", "Date"]
).copy()

verification_results = []

for _, case in extreme_feature_cases.iterrows():

    feature = case["Feature"]
    participant = case["Participant_ID"]
    test_date = pd.Timestamp(case["Date"])

    # Extract the rolling window from the feature name
    window_match = re.search(r"_rolling_(\d+)d_pct_change$", feature)

    if not window_match:
        continue

    window_days = int(window_match.group(1))

    # Identify the original daily source feature
    source_feature = re.sub(
        r"_rolling_\d+d_pct_change$",
        "",
        feature
    )

    # Retrieve historical observations within the correct window
    history = daily_features[
        (daily_features["Participant_ID"] == participant) &
        (daily_features["Date"] < test_date) &
        (daily_features["Date"] >=
         test_date - pd.Timedelta(days=window_days))
    ][["Date", source_feature]].dropna().sort_values("Date")

    last_two = history.tail(2)

    if len(last_two) < 2:
        continue

    previous = float(last_two.iloc[0][source_feature])
    latest = float(last_two.iloc[1][source_feature])

    calculated = (
        ((latest - previous) / previous) * 100
        if previous != 0 else np.nan
    )

    saved = known_target_model_dataset.loc[
        (known_target_model_dataset["Participant_ID"] == participant) &
        (known_target_model_dataset["Date"] == test_date),
        feature
    ]

    saved_value = saved.iloc[0] if not saved.empty else np.nan

    verification_results.append({
        "Feature": feature,
        "Participant_ID": participant,
        "Date": test_date,
        "Window_Days": window_days,
        "Previous_Value": previous,
        "Latest_Value": latest,
        "Calculated_Change": calculated,
        "Saved_Change": saved_value,
        "Match": (
            np.isclose(calculated, saved_value, rtol=1e-9, atol=1e-9)
            if pd.notna(calculated) and pd.notna(saved_value)
            else False
        )
    })

window_verification = pd.DataFrame(verification_results)

print("WINDOW-SPECIFIC VERIFICATION")
print("=" * 60)
print("Cases checked:", len(window_verification))
print("Matching cases:", window_verification["Match"].sum())
print("Non-matching cases:", (~window_verification["Match"]).sum())

display(window_verification)

### Step 10J – Results: Small-Denominator Investigation

**Observed results:**

- Fourteen unique participant-date-source combinations were identified among the extreme cases selected for the 14-day investigation.
- The source measurements and previous/latest values were successfully retrieved for all 14 cases.
- The denominator was particularly small in several cases, including 0.001, 0.002, 0.045 and 0.150.
- A separate window-specific verification checked 39 extreme feature calculations across the 3-, 7- and 14-day windows.
- All 39 calculated values matched the saved feature values. There were zero mismatches.

**Interpretation:**

The investigated extreme percentage changes are consistent with the implemented signed percentage-change formula. Their large magnitudes are substantially associated with small previous measurement values.

Some values repeat across different rolling windows and consecutive feature dates because the same underlying historical measurements can be reused. Therefore, the number of extreme feature cells should not be interpreted as the number of independent measurement events.

**Decision:**

- No percentage-change features were removed.
- No values were capped, clipped, or recalculated.
- Original source measurements were preserved.
- Potential numerical instability will be considered during model development.
- Any future transformation, threshold selection, or feature filtering must be learned using training data only to prevent validation leakage.

This investigation validates the arithmetic of the selected extreme cases; it does not establish clinical abnormality or prove that all percentage-change observations are error-free.

### Step 10K – Compare Percentage Change and Absolute Change

**Objective:** Examine how percentage-change magnitudes relate to absolute measurement differences in the selected extreme cases.

**Reason:** Percentage change depends on the previous measurement as its denominator. A small denominator can produce a very large percentage even when the absolute difference is comparatively modest.

**Approach:**

- Reuse the 14 successfully traced extreme cases from Step 10J.
- Compare previous value, latest value, absolute difference, and absolute percentage change.
- Calculate the latest-to-previous ratio where the previous value is positive.
- Identify cases where a large percentage change accompanies a relatively small absolute difference.

**Important limitation:** This is a focused investigation of selected extreme cases, not a comparison across all observations or a clinical assessment. No original features will be modified or removed.

In [ ]:
# Compare percentage change with absolute change
comparison = denominator_audit.copy()

comparison["Absolute_Percentage_Change"] = (
    comparison["Calculated_Percentage_Change"].abs()
)

comparison["Latest_to_Previous_Ratio"] = np.where(
    comparison["Previous_Value"] > 0,
    comparison["Latest_Value"] / comparison["Previous_Value"],
    np.nan
)

comparison["Previous_Value_Small"] = (
    comparison["Previous_Value"] < 1
)

comparison = comparison.sort_values(
    "Absolute_Percentage_Change",
    ascending=False
)

print("EXTREME CASE COMPARISON")
print("=" * 70)

display(
    comparison[
        [
            "Participant_ID",
            "Date",
            "Source_Feature",
            "Previous_Value",
            "Latest_Value",
            "Absolute_Change",
            "Absolute_Percentage_Change",
            "Latest_to_Previous_Ratio",
            "Previous_Value_Small"
        ]
    ].round(4)
)

print("\nSUMMARY")
print("=" * 70)
print("Cases investigated:", len(comparison))
print(
    "Cases with previous value below 1:",
    comparison["Previous_Value_Small"].sum()
)
print(
    "Median absolute difference:",
    comparison["Absolute_Change"].abs().median()
)
print(
    "Maximum absolute difference:",
    comparison["Absolute_Change"].abs().max()
)
print(
    "Median absolute percentage change:",
    comparison["Absolute_Percentage_Change"].median()
)

### Step 10K – Results: Percentage Change versus Absolute Change

**Observed results:**

- Fourteen unique extreme participant-date-source combinations were compared.
- Seven cases had previous measurement values below 1.
- Median absolute measurement difference was 137.2225.
- Maximum absolute measurement difference was 284.915.
- Median absolute percentage change was approximately 14,815.81%.

**Interpretation:**

Small previous measurements can substantially amplify percentage-change values. However, not all extreme cases have previous values below 1. Some cases also involve large absolute differences between historical measurements.

Therefore, extreme percentages cannot be attributed exclusively to small denominators. Both the relative change and the underlying measurement scale must be considered.

The observations come from different sensor measurements and units, so their absolute differences should not be compared directly as if they represented the same physical quantity.

**Decision:**

- Preserve original percentage-change features.
- Do not apply a universal percentage cap based on this small investigation.
- Consider absolute change and alternative transformations as experimental predictors during model development.
- Fit any thresholds or transformations using training participants only.
- No original measurements or engineered feature values were modified.

### Step 10L – Participant Distribution of Extreme Percentage Changes

**Objective:** Examine how extreme percentage-change observations are distributed across participants.

**Approach:**

- Use the previously identified 665 feature-observation cells exceeding 1,000% in absolute magnitude.
- Count extreme feature cells by participant.
- Count distinct source dates and source measurements where possible.
- Identify whether a small number of participants contribute a disproportionate share of extreme values.

**Interpretation:** A concentration of extreme observations in a few participants may indicate participant-specific measurement patterns or data characteristics. It does not, by itself, establish data errors or clinical significance.

No feature values will be modified or removed.

In [ ]:
# Participant distribution of extreme percentage-change values

extreme_cells = []

for feature in pct_change_features:

    values = known_target_model_dataset[
        feature
    ].abs()

    mask = values > 1000

    if mask.any():
        selected = known_target_model_dataset.loc[
            mask, ["Participant_ID", "Date"]
        ].copy()

        selected["Feature"] = feature

        selected["Source_Feature"] = (
            feature.replace("_rolling_3d_pct_change", "")
                   .replace("_rolling_7d_pct_change", "")
                   .replace("_rolling_14d_pct_change", "")
        )

        selected["Absolute_Percentage_Change"] = values[mask].values

        extreme_cells.append(selected)

if extreme_cells:

    extreme_distribution = pd.concat(
        extreme_cells,
        ignore_index=True
    )

    participant_extremes = (
        extreme_distribution
        .groupby("Participant_ID")
        .agg(
            Extreme_Feature_Cells=("Feature", "size"),
            Unique_Dates=("Date", "nunique"),
            Unique_Source_Features=("Source_Feature", "nunique"),
            Maximum_Absolute_Percentage=(
                "Absolute_Percentage_Change", "max"
            )
        )
        .sort_values(
            "Extreme_Feature_Cells",
            ascending=False
        )
    )

    print("EXTREME VALUES BY PARTICIPANT")
    print("=" * 65)
    display(participant_extremes)

    print("\nSUMMARY")
    print("=" * 65)
    print("Extreme feature cells:", len(extreme_distribution))
    print(
        "Participants with extreme values:",
        len(participant_extremes)
    )
    print(
        "Distinct participant-date combinations:",
        extreme_distribution[
            ["Participant_ID", "Date"]
        ].drop_duplicates().shape[0]
    )

else:
    print("No observations exceeded the selected threshold.")

### Step 10L – Results: Participant Distribution of Extreme Percentage Changes

**Observed results:**

- 665 feature-observation cells exceeded 1,000% in absolute percentage change.
- These cells were distributed across 18 participants.
- There were 172 distinct participant-date combinations.
- GER-003 contributed 114 extreme feature cells, GER-002 contributed 90, and SPA-019 contributed 67.
- These three participants accounted for approximately 40.8% of all extreme feature cells.

**Interpretation:**

Extreme percentage-change values are distributed across multiple participants but are not evenly distributed. A relatively small number of participants contribute a substantial proportion of the extreme feature cells.

The difference between 665 feature cells and 172 participant-date combinations demonstrates that repeated calculations across multiple source measurements and rolling windows contribute to the total.

This concentration may reflect participant-specific measurement patterns, sensor characteristics, or data availability. It does not establish data errors or clinical significance.

**Limitations:**

This analysis covers the 24 participants with known target observations. Participants without known labels were not included. The extreme feature cells are not independent observations.

**Decision:**

- Retain the original percentage-change features.
- Do not remove participants because of extreme feature values.
- Investigate participant-level behaviour during model validation.
- Ensure all validation splits keep each participant entirely within one fold.
- No original feature values were modified or removed.

### Step 10M – Sensor-Level Distribution of Extreme Percentage Changes

**Objective:** Identify which sensor streams contribute the largest number of extreme percentage-change feature cells.

**Approach:**

- Extract the sensor prefix from each percentage-change feature.
- Count extreme feature cells exceeding 1,000% in absolute magnitude by sensor.
- Calculate each sensor's percentage contribution to the total.
- Count distinct participant-date combinations per sensor.

**Interpretation:** This provides a descriptive view of where extreme percentage changes occur. It does not establish that a sensor is faulty or that its measurements are clinically abnormal.

No features will be removed or modified.

In [ ]:
# Sensor-level distribution of extreme percentage changes

sensor_extremes = extreme_distribution.copy()

# Extract sensor name from feature name
sensor_extremes["Sensor"] = (
    sensor_extremes["Feature"].str.split("_").str[0]
)

# Aggregate extreme feature cells and participants
sensor_summary = (
    sensor_extremes
    .groupby("Sensor")
    .agg(
        Extreme_Feature_Cells=("Feature", "size"),
        Participants=("Participant_ID", "nunique")
    )
)

# Count distinct participant-date combinations separately
sensor_dates = (
    sensor_extremes[
        ["Sensor", "Participant_ID", "Date"]
    ]
    .drop_duplicates()
    .groupby("Sensor")
    .size()
    .rename("Unique_Participant_Dates")
)

# Combine results
sensor_summary = sensor_summary.join(sensor_dates)

# Calculate percentage contribution
total_extreme_cells = sensor_summary[
    "Extreme_Feature_Cells"
].sum()

sensor_summary["Percentage_of_Extreme_Cells"] = (
    sensor_summary["Extreme_Feature_Cells"]
    / total_extreme_cells
    * 100
)

# Sort by number of extreme feature cells
sensor_summary = sensor_summary.sort_values(
    "Extreme_Feature_Cells",
    ascending=False
)

print("EXTREME PERCENTAGE CHANGES BY SENSOR")
print("=" * 65)

display(sensor_summary.round(2))

print("\nSUMMARY")
print("=" * 65)
print("Total extreme feature cells:", total_extreme_cells)
print(
    "Sensors contributing extreme values:",
    len(sensor_summary)
)

### Step 10M – Results: Sensor-Level Distribution of Extreme Percentage Changes

**Observed results:**

A total of 665 feature-observation cells exceeded 1,000% in absolute percentage change.

| Sensor | Extreme feature cells | Participants | Percentage of extreme cells |
|---|---:|---:|---:|
| EI | 339 | 15 | 50.98% |
| MI | 278 | 12 | 41.80% |
| SHRI | 21 | 1 | 3.16% |
| PEI | 12 | 2 | 1.80% |
| SQI | 12 | 1 | 1.80% |
| SII | 3 | 1 | 0.45% |

EI and MI together contributed 617 of 665 extreme feature cells, approximately 92.78%.

**Interpretation:**

Extreme percentage-change values are concentrated primarily in EI and MI features. The remaining sensor streams contribute comparatively few extreme cells.

This concentration may reflect differences in measurement scales, variability, availability, or previous-value magnitudes. It does not independently establish sensor malfunction or clinical abnormality.

**Decision:**

- Preserve all original features and source measurements.
- Prioritize EI and MI for further investigation.
- Do not apply a universal cap or remove either sensor stream based on this analysis alone.
- Evaluate alternative transformations during model development using training data only.

### Step 10N – Measurement-Level Distribution of Extreme Percentage Changes

**Objective:** Identify which individual EI and MI source measurements contribute most to extreme percentage-change feature cells.

**Approach:**

- Filter the extreme percentage-change records to EI and MI.
- Extract their source measurement names.
- Count extreme feature cells by source measurement.
- Calculate the number of participants and distinct participant-date combinations associated with each measurement.

**Interpretation:** This analysis identifies which measurement features deserve closer numerical-stability investigation. It does not establish that a measurement is erroneous or clinically abnormal.

No features or observations will be modified or removed.

In [ ]:
# Measurement-level distribution of EI and MI extreme values

ei_mi_extremes = extreme_distribution[
    extreme_distribution["Feature"].str.startswith(
        ("EI_", "MI_")
    )
].copy()

measurement_summary = (
    ei_mi_extremes
    .groupby("Source_Feature")
    .agg(
        Extreme_Feature_Cells=("Feature", "size"),
        Participants=("Participant_ID", "nunique")
    )
)

# Distinct participant-date combinations
measurement_dates = (
    ei_mi_extremes[
        ["Source_Feature", "Participant_ID", "Date"]
    ]
    .drop_duplicates()
    .groupby("Source_Feature")
    .size()
    .rename("Unique_Participant_Dates")
)

measurement_summary = measurement_summary.join(
    measurement_dates
)

measurement_summary["Percentage_of_EI_MI_Extremes"] = (
    measurement_summary["Extreme_Feature_Cells"]
    / measurement_summary["Extreme_Feature_Cells"].sum()
    * 100
)

measurement_summary = measurement_summary.sort_values(
    "Extreme_Feature_Cells",
    ascending=False
)

print("EI AND MI MEASUREMENT-LEVEL EXTREME VALUES")
print("=" * 70)

display(measurement_summary.round(2))

print("\nSUMMARY")
print("=" * 70)
print("Total EI/MI extreme cells:", len(ei_mi_extremes))
print(
    "Distinct source measurements:",
    measurement_summary.shape[0]
)

### Step 10N – Results: Measurement-Level Distribution of Extreme Percentage Changes

**Observed results:**

A total of 617 extreme feature cells were associated with EI and MI measurements, distributed across 11 source measurements.

The largest contributors were:

- EI_Lux_std_mean: 105 cells (17.02%).
- EI_Lux_mean: 102 cells (16.53%).
- EI_Lux_max: 96 cells (15.56%).
- MI_walked_dist_mean: 94 cells (15.24%).
- MI_gs_std_mean: 59 cells (9.56%).

The three Lux-related measurements accounted for 303 cells, approximately 49.1% of the EI/MI extreme cells.

**Interpretation:**

Extreme percentage changes are concentrated in EI light-related measurements and selected MI movement-related measurements.

This may reflect the numerical behaviour of percentage change when previous measurements are small, as well as differences in measurement scale and variability.

The results do not establish sensor malfunction or clinical abnormality.

**Decision:**

- Preserve all original measurements and engineered features.
- Prioritize EI Lux-related and MI movement-related features for numerical-stability investigation.
- Evaluate alternative feature representations during model development.
- Avoid universal percentage-change thresholds across measurements with different units and scales.
- No features were removed or modified.

### Step 10O – Previous-Value Distribution of Extreme EI and MI Measurements

**Objective:** Investigate whether extreme percentage changes are associated with consistently small previous measurements or occasional near-zero values.

**Approach:**

- Examine the previous values recorded in the 617 EI and MI extreme feature cells.
- Summarize their minimum, median, maximum, and quartiles by source measurement.
- Count previous values equal to zero and below 1.
- Compare the distributions across different source measurements.

**Interpretation:** This investigation examines the denominator behaviour underlying extreme percentage changes. Because measurements use different units and scales, absolute previous-value thresholds are descriptive only and should not be treated as universal clinical or data-quality thresholds.

No source measurements or engineered feature values will be modified.

In [ ]:
# Reconstruct previous values for EI and MI extreme cases

ei_mi_previous_values = denominator_audit[
    denominator_audit["Source_Feature"].str.startswith(
        ("EI_", "MI_")
    )
].copy()

print("EI AND MI DENOMINATOR AUDIT")
print("=" * 70)
print("Cases available in denominator audit:",
      len(ei_mi_previous_values))

# Summarise previous values by source measurement
previous_value_summary = (
    ei_mi_previous_values
    .groupby("Source_Feature")
    .agg(
        Cases=("Previous_Value", "size"),
        Minimum_Previous=("Previous_Value", "min"),
        Q1_Previous=("Previous_Value", lambda x: x.quantile(0.25)),
        Median_Previous=("Previous_Value", "median"),
        Q3_Previous=("Previous_Value", lambda x: x.quantile(0.75)),
        Maximum_Previous=("Previous_Value", "max"),
        Previous_Values_Zero=("Previous_Value", lambda x: (x == 0).sum()),
        Previous_Values_Below_1=(
            "Previous_Value",
            lambda x: (x.abs() < 1).sum()
        )
    )
    .sort_values("Cases", ascending=False)
)

print("\nPREVIOUS-VALUE DISTRIBUTION")
print("=" * 70)

display(previous_value_summary.round(4))

print("\nOVERALL DENOMINATOR SUMMARY")
print("=" * 70)

print(
    "Previous values equal to zero:",
    (ei_mi_previous_values["Previous_Value"] == 0).sum()
)

print(
    "Previous values with absolute magnitude below 1:",
    (ei_mi_previous_values["Previous_Value"].abs() < 1).sum()
)

print(
    "Total cases in this denominator audit:",
    len(ei_mi_previous_values)
)


### Step 10O – Previous-Value Distribution of Extreme EI and MI Measurements

**Objective**

Investigate whether extreme percentage changes in EI and MI measurements are associated with small previous measurement values, which may amplify the percentage-change calculation.

**Observed results**

The investigation covered 14 previously traced extreme percentage-change cases:

| Source measurement | Cases | Previous values below 1 |
|---|---:|---:|
| EI_Lux_mean | 5 | 2 |
| EI_Lux_std_mean | 5 | 1 |
| MI_gs_std_mean | 2 | 2 |
| MI_smwd_std_mean | 2 | 2 |
| **Total** | **14** | **7** |

- No previous values were exactly zero.
- Seven of the 14 previous values had an absolute magnitude below 1.
- The MI measurements had particularly small previous values in the investigated cases.
- The threshold of 1 was used descriptively and is not a universal measurement-quality threshold.

**Interpretation**

Small previous values can substantially amplify percentage changes because the previous measurement is the denominator in the calculation.

For example, a change from 0.001 to 0.152 produces a percentage change of 15,100%, despite an absolute change of only 0.151.

The investigated values are consistent with the implemented signed percentage-change formula. However, this analysis does not establish that the underlying sensor measurements are erroneous.

**Limitations**

This investigation covers only 14 previously traced cases. It does not represent all 617 extreme EI and MI feature cells identified in Step 10M.

**Decision**

No measurements or features were removed, capped, or modified. A broader denominator investigation is required before making any feature-transformation decisions.

### Step 10P – Comprehensive Denominator Investigation of Extreme EI and MI Percentage Changes

**Objective**

Extend the denominator investigation from the 14 previously traced cases to all extreme EI and MI percentage-change feature cells identified in Step 10M.

An extreme percentage-change feature cell is defined as an observation where the absolute percentage change exceeds 1,000%.

**Investigation**

For each extreme feature cell, the analysis will attempt to retrieve:

- Participant identifier and feature date.
- Source measurement and rolling window.
- Previous and latest historical measurement values.
- Absolute measurement change.
- Recalculated percentage change.
- Whether the previous value is zero or has an absolute magnitude below 1.

**Why this matters**

A very small previous measurement can produce an extremely large percentage change even when the absolute measurement change is relatively modest. A comprehensive investigation will help determine how frequently this occurs across EI and MI measurements.

**Important considerations**

- The threshold of 1 is descriptive, not a universal clinical or data-quality threshold.
- Repeated participant-date combinations across different rolling windows are not independent observations.
- Missing source measurements will not be imputed.
- No features will be removed, capped, or modified during this investigation.

In [ ]:
print("EI/MI extreme records:")
print(ei_mi_extremes.columns.tolist())

print("\nDenominator audit columns:")
print(denominator_audit.columns.tolist())

print("\nEI/MI extreme records:", len(ei_mi_extremes))
print("Previously traced denominator cases:", len(denominator_audit))

print("\nSample EI/MI extreme records:")
display(ei_mi_extremes.head(5))

print("\nSample denominator audit records:")
display(denominator_audit.head(5))

In [ ]:
# Load daily source measurements

DAILY_SOURCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "daily_sensor_features.parquet"
)

daily_source = duckdb.execute(
    "SELECT * FROM read_parquet(?)",
    [str(DAILY_SOURCE_PATH)]
).fetchdf()

print("Daily source dataset loaded successfully")
print("Shape:", daily_source.shape)
print("Participants:", daily_source["Participant_ID"].nunique())
print("Date range:",
      daily_source["Date"].min(),
      "to",
      daily_source["Date"].max())

print("\nAvailable columns:")
print(daily_source.columns.tolist())

In [ ]:
# Step 10P – Reconstruct denominator values for all EI/MI extremes

import re
import numpy as np
import pandas as pd

# Standardise source data
daily_source["Participant_ID"] = (
    daily_source["Participant_ID"].astype("string")
)

daily_source["Date"] = pd.to_datetime(
    daily_source["Date"], utc=True
)

daily_source = daily_source.sort_values(
    ["Participant_ID", "Date"]
)

# Standardise extreme records
extreme_records = ei_mi_extremes.copy()

extreme_records["Participant_ID"] = (
    extreme_records["Participant_ID"].astype("string")
)

extreme_records["Date"] = pd.to_datetime(
    extreme_records["Date"], utc=True
)

# Prepare participant-specific source data
participant_sources = {
    participant: group.set_index("Date").sort_index()
    for participant, group in daily_source.groupby("Participant_ID")
}

reconstructed_records = []

for _, record in extreme_records.iterrows():

    participant = record["Participant_ID"]
    target_date = record["Date"]
    source_feature = record["Source_Feature"]
    feature_name = record["Feature"]

    # Extract rolling window
    match = re.search(
        r"_rolling_(\d+)d_pct_change$",
        feature_name
    )

    if match is None:
        continue

    window_days = int(match.group(1))

    if participant not in participant_sources:
        continue

    participant_data = participant_sources[participant]

    if source_feature not in participant_data.columns:
        continue

    # Reconstruct historical window.
    # Includes source dates from target_date - window through
    # the day before target_date, matching the shifted rolling feature.

    window_start = target_date - pd.Timedelta(days=window_days)

    historical = participant_data.loc[
        (participant_data.index >= window_start) &
        (participant_data.index < target_date),
        source_feature
    ]

    historical = pd.to_numeric(
        historical, errors="coerce"
    )

    historical = historical[
        np.isfinite(historical)
    ]

    if len(historical) < 2:
        continue

    previous_value = historical.iloc[-2]
    latest_value = historical.iloc[-1]

    if previous_value == 0:
        calculated_pct_change = np.nan
    else:
        calculated_pct_change = (
            (latest_value - previous_value) /
            previous_value
        ) * 100

    reconstructed_records.append({
        "Participant_ID": participant,
        "Date": target_date,
        "Feature": feature_name,
        "Source_Feature": source_feature,
        "Window_Days": window_days,
        "Previous_Value": previous_value,
        "Latest_Value": latest_value,
        "Absolute_Change": latest_value - previous_value,
        "Calculated_Percentage_Change": calculated_pct_change,
        "Historical_Observations": len(historical)
    })


# Create comprehensive audit
comprehensive_denominator_audit = pd.DataFrame(
    reconstructed_records
)

print("COMPREHENSIVE DENOMINATOR RECONSTRUCTION")
print("=" * 70)

print("Original extreme feature cells:", len(extreme_records))
print("Successfully reconstructed:",
      len(comprehensive_denominator_audit))
print("Not reconstructed:",
      len(extreme_records) - len(comprehensive_denominator_audit))

print("\nRECONSTRUCTION BY SENSOR")

display(
    comprehensive_denominator_audit
    .assign(
        Sensor=lambda x: x["Source_Feature"].str.split("_").str[0]
    )
    .groupby("Sensor")
    .size()
    .rename("Reconstructed_Cases")
    .to_frame()
)


# Validate against the previously investigated 14 cases
comparison = denominator_audit.merge(
    comprehensive_denominator_audit,
    on=["Participant_ID", "Date", "Source_Feature"],
    how="left",
    suffixes=("_old", "_new")
)

comparison["Previous_Match"] = np.isclose(
    comparison["Previous_Value_old"],
    comparison["Previous_Value_new"],
    rtol=1e-9,
    atol=1e-12,
    equal_nan=True
)

comparison["Latest_Match"] = np.isclose(
    comparison["Latest_Value_old"],
    comparison["Latest_Value_new"],
    rtol=1e-9,
    atol=1e-12,
    equal_nan=True
)

print("\nVALIDATION AGAINST PREVIOUS 14 CASES")
print("=" * 70)

print("Previously investigated cases:", len(comparison))
print("Previous values matching:",
      comparison["Previous_Match"].sum())
print("Latest values matching:",
      comparison["Latest_Match"].sum())

print("All previous values match:",
      comparison["Previous_Match"].all())

print("All latest values match:",
      comparison["Latest_Match"].all())

### Step 10P – Comprehensive Denominator Reconstruction of Extreme EI and MI Percentage Changes

**Objective**

Reconstruct the previous and latest source measurement values for every EI and MI percentage-change feature cell exceeding an absolute percentage change of 1,000%.

**Observed results**

A total of 617 extreme feature cells were investigated.

| Sensor | Reconstructed feature cells |
|---|---:|
| EI | 339 |
| MI | 278 |
| **Total** | **617** |

- Successfully reconstructed: 617
- Not reconstructed: 0
- All 617 cases were traced to their corresponding source measurements.
- The reconstructed historical windows excluded the current feature date.
- Previous and latest values were compared against the earlier denominator investigation. All 39 window-specific comparison records matched.

**Interpretation**

The reconstruction confirms that all identified extreme EI and MI percentage-change feature cells can be traced to their historical source measurements.

The agreement with the earlier investigation supports the correctness of the reconstruction procedure.

This step establishes traceability, not the clinical validity or data quality of the underlying measurements.

**Decision**

All 617 reconstructed cases will be retained for further investigation. No features or measurements were removed, capped, or modified.

In [ ]:
# Step 10Q – Overall denominator distribution

audit = comprehensive_denominator_audit.copy()

audit["Previous_Absolute_Value"] = audit["Previous_Value"].abs()

print("OVERALL DENOMINATOR DISTRIBUTION")
print("=" * 70)

print("Total reconstructed cases:", len(audit))

print(
    "Previous values equal to zero:",
    (audit["Previous_Value"] == 0).sum()
)

print(
    "Previous values with absolute magnitude below 0.01:",
    (audit["Previous_Absolute_Value"] < 0.01).sum()
)

print(
    "Previous values with absolute magnitude below 0.1:",
    (audit["Previous_Absolute_Value"] < 0.1).sum()
)

print(
    "Previous values with absolute magnitude below 1:",
    (audit["Previous_Absolute_Value"] < 1).sum()
)

print(
    "Previous values with absolute magnitude >= 1:",
    (audit["Previous_Absolute_Value"] >= 1).sum()
)

print("\nPREVIOUS VALUE SUMMARY")
display(
    audit["Previous_Value"].describe().to_frame(
        name="Previous_Value"
    ).round(6)
)

print("\nABSOLUTE CHANGE SUMMARY")
display(
    audit["Absolute_Change"].abs().describe().to_frame(
        name="Absolute_Change"
    ).round(6)
)

### Step 10Q – Overall Denominator Distribution of Extreme EI and MI Percentage Changes

**Objective**

Assess the distribution of previous measurement values and absolute changes across all 617 reconstructed extreme EI and MI percentage-change feature cells.

**Observed results**

| Previous measurement magnitude | Cases | Percentage |
|---|---:|---:|
| Below 0.01 | 74 | 12.0% |
| Below 0.1 | 142 | 23.0% |
| Below 1 | 260 | 42.1% |
| At least 1 | 357 | 57.9% |
| Exactly zero | 0 | 0% |

Previous measurement summary:
- Median: 1.645
- Mean: 5.745
- Minimum: 0.001
- Maximum: 37.8

Absolute change summary:
- Median: 78.0
- Mean: 131.046
- Minimum: 0.039
- Maximum: 595.6

**Interpretation**

Small previous measurement values are a substantial contributor to extreme percentage changes. However, 57.9% of the investigated feature cells had previous values with an absolute magnitude of at least 1.

No previous values were exactly zero, so division by zero was not responsible for the investigated extreme values.

The absolute changes also varied considerably. This suggests that extreme percentage changes may reflect both denominator amplification and substantial changes in the underlying measurements.

The absolute changes combine different measurement types and units and therefore should not be interpreted as directly comparable magnitudes.

**Limitations**

The 617 records represent feature cells, not independent observations. The same underlying participant-date-source measurement may appear in multiple rolling windows.

The denominator thresholds are descriptive and are not clinical or data-quality thresholds.

**Decision**

No measurements were removed, capped, or modified. Further investigation will compare denominator distributions and absolute changes separately by sensor and source measurement.

In [ ]:
# Step 10R – Sensor-specific denominator analysis

audit = comprehensive_denominator_audit.copy()

audit["Sensor"] = (
    audit["Source_Feature"].str.split("_").str[0]
)

audit["Previous_Absolute_Value"] = (
    audit["Previous_Value"].abs()
)

sensor_denominator_summary = (
    audit.groupby("Sensor")
    .agg(
        Cases=("Previous_Value", "size"),
        Median_Previous=("Previous_Absolute_Value", "median"),
        Mean_Previous=("Previous_Absolute_Value", "mean"),
        Below_001=(
            "Previous_Absolute_Value",
            lambda x: (x < 0.01).sum()
        ),
        Below_01=(
            "Previous_Absolute_Value",
            lambda x: (x < 0.1).sum()
        ),
        Below_1=(
            "Previous_Absolute_Value",
            lambda x: (x < 1).sum()
        ),
        At_Least_1=(
            "Previous_Absolute_Value",
            lambda x: (x >= 1).sum()
        ),
        Median_Absolute_Change=(
            "Absolute_Change",
            lambda x: x.abs().median()
        ),
        Maximum_Absolute_Change=(
            "Absolute_Change",
            lambda x: x.abs().max()
        )
    )
)

print("SENSOR-SPECIFIC DENOMINATOR SUMMARY")
print("=" * 90)

display(sensor_denominator_summary.round(4))

print("\nSOURCE MEASUREMENT-SPECIFIC SUMMARY")
print("=" * 90)

source_denominator_summary = (
    audit.groupby("Source_Feature")
    .agg(
        Cases=("Previous_Value", "size"),
        Median_Previous=(
            "Previous_Absolute_Value", "median"
        ),
        Below_1=(
            "Previous_Absolute_Value",
            lambda x: (x < 1).sum()
        ),
        Median_Absolute_Change=(
            "Absolute_Change",
            lambda x: x.abs().median()
        ),
        Maximum_Absolute_Change=(
            "Absolute_Change",
            lambda x: x.abs().max()
        )
    )
    .sort_values("Cases", ascending=False)
)

display(source_denominator_summary.round(4))

### Step 10R – Sensor-Specific and Measurement-Specific Denominator Analysis

**Objective**

Compare previous measurement values and absolute changes across EI and MI sensors and their individual source measurements.

**Observed results**

Sensor-level findings:

| Metric | EI | MI |
|---|---:|---:|
| Extreme feature cells | 339 | 278 |
| Median previous value | 2.83 | 0.50 |
| Previous values below 0.01 | 3 | 71 |
| Previous values below 0.1 | 48 | 94 |
| Previous values below 1 | 93 | 167 |
| Previous values at least 1 | 246 | 111 |
| Median absolute change | 136.95 | 7.30 |

Measurement-level findings:

- All 59 investigated MI_gs_std_mean cases had previous values below 1.
- All 35 MI_sl_std_mean cases had previous values below 1.
- All 43 MI_walked_time_mean cases had previous values below 1.
- None of the 94 MI_walked_dist_mean cases had previous values below 1.
- None of the 96 EI_Lux_max cases had previous values below 1.
- EI_Lux_max and MI_walked_dist_mean had relatively large median absolute changes of 280.0 and 196.7, respectively.

**Interpretation**

The extreme percentage changes exhibit different patterns across source measurements.

Some MI measurements show substantial denominator amplification because their previous values are consistently small. Other measurements, particularly EI_Lux_max and MI_walked_dist_mean, show large absolute changes despite previous values of at least 1.

Therefore, extreme percentage changes cannot be explained by one universal denominator effect.

The numerical thresholds are descriptive only. Different source measurements have different units and scales.

**Limitations**

The investigated records are feature cells and may repeat the same underlying source observation across multiple rolling windows. The results should not be interpreted as counts of independent physiological events.

**Decision**

No measurements were removed, capped, or modified. Any future transformation of percentage-change features should be evaluated separately by source measurement and fitted using training participants only.

### Step 10S – Investigating Repetition Across Rolling Windows

**Objective**

Determine how frequently extreme percentage-change observations are repeated across the 3-day, 7-day and 14-day rolling windows.

**Why this matters**

The same participant, date and source measurement may produce extreme percentage changes in multiple rolling windows. Consequently, the number of extreme feature cells may overstate the number of unique underlying measurement situations.

**Investigation**

The analysis will calculate:

- Total extreme feature cells.
- Unique participant-date-source combinations.
- Combinations appearing in multiple rolling windows.
- Combinations appearing in all three windows.
- Distribution of extreme cells across rolling windows.

This analysis will help distinguish feature-level counts from unique underlying observations.

In [ ]:
# Step 10S – Rolling-window repetition analysis

audit = comprehensive_denominator_audit.copy()

unique_keys = ["Participant_ID", "Date", "Source_Feature"]

print("ROLLING-WINDOW REPETITION ANALYSIS")
print("=" * 70)

total_cells = len(audit)
unique_combinations = audit[unique_keys].drop_duplicates()

print("Total extreme feature cells:", total_cells)
print("Unique participant-date-source combinations:", len(unique_combinations))
print("Repeated feature cells:", total_cells - len(unique_combinations))

print(
    "Percentage of cells beyond one per unique combination:",
    round((total_cells - len(unique_combinations)) / total_cells * 100, 2)
)

# Count how many rolling windows contain each unique combination
window_repetition = (
    audit.groupby(unique_keys)
    .agg(
        Number_of_Windows=("Window_Days", "nunique"),
        Windows=(
            "Window_Days",
            lambda x: ", ".join(map(str, sorted(x.unique())))
        ),
    )
    .reset_index()
)

print("\nDISTRIBUTION OF WINDOW REPETITION")
display(
    window_repetition["Number_of_Windows"]
    .value_counts()
    .sort_index()
    .rename_axis("Number_of_Windows")
    .to_frame("Unique_Combinations")
)

all_three_windows = window_repetition[
    window_repetition["Number_of_Windows"] == 3
]

print(
    "\nUnique combinations appearing in all three windows:",
    len(all_three_windows)
)

print("\nEXTREME CELLS BY ROLLING WINDOW")

window_summary = (
    audit.groupby("Window_Days")
    .agg(
        Extreme_Feature_Cells=("Feature", "size"),
        Unique_Participant_Dates=("Date", "nunique"),
        Unique_Source_Measurements=("Source_Feature", "nunique"),
    )
    .sort_index()
)

display(window_summary)

print("\nEXAMPLES OF REPEATED COMBINATIONS")

repeated_examples = (
    window_repetition[
        window_repetition["Number_of_Windows"] > 1
    ]
    .sort_values("Number_of_Windows", ascending=False)
    .head(10)
)

display(repeated_examples)

### Step 10S – Investigating Repetition Across Rolling Windows

**Objective**

Determine how frequently extreme percentage-change feature cells are repeated across the 3-day, 7-day and 14-day rolling windows.

**Observed results**

| Measure | Result |
|---|---:|
| Total extreme feature cells | 617 |
| Unique participant-date-source combinations | 279 |
| Additional repeated feature cells | 338 |
| Repetition proportion | 54.78% |

Distribution of unique combinations:

| Number of rolling windows | Unique combinations |
|---|---:|
| One window | 78 |
| Two windows | 64 |
| Three windows | 137 |

Distribution by rolling window:

| Window | Extreme feature cells | Unique participant dates |
|---|---:|---:|
| 3 days | 137 | 69 |
| 7 days | 201 | 98 |
| 14 days | 279 | 119 |

All 279 unique participant-date-source combinations appeared in the 14-day window.

**Interpretation**

The 617 extreme feature cells represent 279 unique participant-date-source combinations. Therefore, 338 additional feature cells arise from repeated appearances across rolling windows.

This demonstrates that feature-cell counts should not be interpreted as counts of independent underlying measurement situations.

The 14-day window contains the highest number of extreme feature cells, followed by the 7-day and 3-day windows.

**Limitations**

A unique participant-date-source combination is not necessarily an independent clinical event or independent measurement episode. Repeated observations from the same participant may also be correlated over time.

**Decision**

All features and measurements remain unchanged. Subsequent investigations should distinguish rolling-window feature cells from unique participant-date-source combinations.

### Step 10T – Consistency of Source Measurements Across Rolling Windows

**Objective**

Determine whether repeated extreme percentage-change feature cells use identical previous and latest source measurements across the 3-day, 7-day and 14-day rolling windows.

**Investigation**

For each unique participant-date-source combination, compare:

- Previous source measurement values.
- Latest source measurement values.
- Absolute changes.
- Reconstructed percentage changes.

**Why this matters**

If the reconstructed values are identical across windows, the repeated feature cells represent the same underlying source measurement change.

If they differ, the rolling windows are using different historical measurement pairs, and the repetition should not be treated as entirely equivalent.

No features or measurements will be modified during this investigation.

In [ ]:
# Step 10T – Compare source measurement pairs across windows

audit = comprehensive_denominator_audit.copy()

comparison_columns = [
    "Previous_Value",
    "Latest_Value",
    "Absolute_Change",
    "Calculated_Percentage_Change"
]

# Group by unique underlying measurement situation
consistency_summary = (
    audit.groupby(
        ["Participant_ID", "Date", "Source_Feature"]
    )[comparison_columns]
    .agg(["min", "max", "nunique"])
)

# Determine whether values are identical across windows
for column in comparison_columns:
    consistency_summary[(column, "consistent")] = (
        consistency_summary[(column, "nunique")] == 1
    )

consistency_columns = [
    (column, "consistent")
    for column in comparison_columns
]

print("SOURCE MEASUREMENT CONSISTENCY ACROSS WINDOWS")
print("=" * 75)

print("Unique combinations:", len(consistency_summary))

for column in comparison_columns:
    print(
        f"{column}:",
        consistency_summary[(column, "consistent")].sum(),
        "consistent out of",
        len(consistency_summary)
    )

# Combinations where at least one value differs
inconsistent_mask = ~consistency_summary[
    consistency_columns
].all(axis=1)

inconsistent_combinations = consistency_summary[
    inconsistent_mask
]

print("\nCOMBINATIONS WITH DIFFERENCES")
print("=" * 75)

print("Inconsistent combinations:",
      len(inconsistent_combinations))

display(inconsistent_combinations.head(15))

### Step 10T – Consistency of Source Measurements Across Rolling Windows

**Objective**

Determine whether repeated extreme percentage-change feature cells use identical source measurement pairs across the 3-day, 7-day and 14-day rolling windows.

**Observed results**

A total of 279 unique participant-date-source combinations were investigated.

| Measurement | Consistent combinations |
|---|---:|
| Previous source value | 279 / 279 |
| Latest source value | 279 / 279 |
| Absolute change | 279 / 279 |
| Calculated percentage change | 279 / 279 |

- Inconsistent combinations: 0
- All 279 combinations had identical reconstructed measurement values across their respective rolling windows.

**Interpretation**

The repeated extreme feature cells across rolling windows represent the same underlying previous and latest source measurements for each participant-date-source combination.

Consequently, the 617 extreme feature cells should not be interpreted as 617 independent underlying measurement changes. They correspond to 279 unique participant-date-source combinations, with repeated representations across rolling windows.

**Limitations**

This investigation concerns the reconstructed source measurement pairs associated with the identified extreme percentage-change features. It does not establish that all rolling-window features in the dataset are interchangeable or redundant.

**Decision**

No feature values were changed, and no rolling-window features were removed. The repetition is documented as a feature-representation issue rather than evidence of additional independent measurement changes.

### Step 10U – Final Summary of Extreme Percentage-Change Investigation

**Objective**

Consolidate the investigation of extreme percentage-change features, identify their underlying causes, and establish an appropriate strategy for subsequent model preparation.

**Summary of observed findings**

| Investigation | Finding |
|---|---|
| Extreme percentage-change feature cells | 665 across all sensors |
| EI and MI extreme cells | 617 |
| Unique EI/MI participant-date-source combinations | 279 |
| Additional repeated cells across windows | 338 |
| Combinations appearing in all three windows | 137 |
| Source measurement reconstruction | All 617 cells reconstructed |
| Window-specific validation records | 39 / 39 matched |
| Previous-value denominator equal to zero | 0 |
| EI/MI cases with previous value below 1 | 260 / 617 |
| Inconsistent source measurement pairs across windows | 0 / 279 |

**Observed patterns**

- EI and MI account for most identified extreme percentage-change cells.
- Some features have very small previous measurement values, which can amplify percentage changes.
- Other features exhibit large absolute changes, indicating that small denominators are not the only explanation.
- Repeated rolling-window feature cells represent the same reconstructed source measurement pair for each unique participant-date-source combination.
- The source measurements were retained during preprocessing.

**Interpretation**

Extreme percentage changes are not automatically evidence of erroneous sensor data. They may result from a combination of small previous values, genuine large measurement changes, measurement variability, and the mathematical properties of percentage change.

The repeated rolling-window representations also mean that feature-cell counts should not be interpreted as counts of independent measurement changes.

**Modelling decision**

1. Do not globally delete or clip extreme percentage-change values.
2. Do not modify the original source measurements.
3. Preserve the existing engineered features for now.
4. Evaluate feature usefulness and redundancy using training-fold

In [ ]:
# Step 10U – Final summary of extreme percentage changes

final_extreme_summary = pd.DataFrame({
    "Investigation": [
        "All-sensor extreme feature cells",
        "EI and MI extreme feature cells",
        "Unique EI/MI participant-date-source combinations",
        "Additional repeated EI/MI feature cells",
        "Combinations appearing in all three windows",
        "EI/MI cells successfully reconstructed",
        "Window-specific validation records matched",
        "Previous values equal to zero",
        "Previous values below 1",
        "Inconsistent source measurement combinations"
    ],
    "Result": [
        len(extreme_cells),
        len(ei_mi_extremes),
        len(unique_combinations),
        total_cells - len(unique_combinations),
        len(all_three_windows),
        len(comprehensive_denominator_audit),
        39,
        int((comprehensive_denominator_audit["Previous_Value"] == 0).sum()),
        int((comprehensive_denominator_audit["Previous_Value"].abs() < 1).sum()),
        len(inconsistent_combinations)
    ]
})

display(final_extreme_summary)

In [ ]:
# Extract sensor name from Source_Feature
extreme_cells_combined["Sensor"] = (
    extreme_cells_combined["Source_Feature"]
    .str.split("_")
    .str[0]
)

print("EXTREME FEATURE COUNT VALIDATION")
print("=" * 65)

print("Total extreme feature records:",
      len(extreme_cells_combined))

print("\nALL-SENSOR DISTRIBUTION")

sensor_distribution = (
    extreme_cells_combined
    .groupby("Sensor")
    .size()
    .rename("Extreme_Feature_Cells")
    .to_frame()
)

sensor_distribution["Percentage"] = (
    sensor_distribution["Extreme_Feature_Cells"]
    / len(extreme_cells_combined) * 100
).round(2)

display(sensor_distribution)

### 10S. Rolling-Window Repetition and Sensor Distribution of Extreme Percentage Changes

**Objective:** Examine how extreme percentage-change feature records are distributed across sensor streams and understand the extent to which rolling-window features repeat the same underlying observations.

**Method:**
- Combined the 47 individual DataFrames containing extreme percentage-change records.
- Extracted the sensor identifier from each record's `Source_Feature`.
- Counted extreme records by sensor and calculated each sensor's percentage contribution.

**Observed results:**

| Sensor | Extreme records | Percentage |
|---|---:|---:|
| EI | 339 | 50.98% |
| MI | 278 | 41.80% |
| PEI | 12 | 1.80% |
| SHRI | 21 | 3.16% |
| SII | 3 | 0.45% |
| SQI | 12 | 1.80% |
| **Total** | **665** | **100%** |

EI and MI together account for 617 records (92.78%) of all extreme feature records.

**Interpretation:**

Extreme percentage changes are concentrated in EI and MI features. However, these records should not be interpreted as independent events. The same underlying participant-date-source measurement may appear in multiple rolling-window features (3-day, 7-day and 14-day windows).

This analysis describes the distribution of extreme feature records; it does not establish that the corresponding sensor streams are unreliable or that the observations are erroneous.

**Decision:** Retain the observations and features. No automatic removal or clipping is performed at this stage. Further feature selection and transformation decisions should be evaluated using training data only.

In [ ]:
# Count how many rolling-window features represent each
# underlying participant-date-source combination.

window_repetition = (
    extreme_cells_combined
    .groupby(
        ["Participant_ID", "Date", "Source_Feature"]
    )
    .agg(
        Number_of_Window_Features=("Feature", "nunique"),
        Total_Feature_Records=("Feature", "count")
    )
    .reset_index()
)

print("ROLLING-WINDOW REPETITION ANALYSIS")
print("=" * 65)

print("Total extreme feature records:",
      len(extreme_cells_combined))

print("Unique participant-date-source cases:",
      len(window_repetition))

print("\nNUMBER OF CASES BY WINDOW REPETITION")

display(
    window_repetition
    ["Number_of_Window_Features"]
    .value_counts()
    .sort_index()
    .rename_axis("Number_of_Windows")
    .rename("Number_of_Cases")
    .to_frame()
)

### 10S.1 Rolling-Window Repetition Analysis

**Objective:** Determine how many extreme percentage-change feature records represent distinct underlying participant-date-source cases and how frequently the same case appears across multiple rolling windows.

**Method:**
- Grouped extreme feature records by `Participant_ID`, `Date`, and `Source_Feature`.
- Counted the distinct rolling-window features associated with each group.
- Classified cases according to whether they appeared in one, two, or three rolling windows.

**Observed results:**

- Total extreme feature records: 665
- Unique participant-date-source cases: 305

| Number of rolling windows | Number of distinct cases |
|---:|---:|
| 1 | 87 |
| 2 | 76 |
| 3 | 142 |
| **Total** | **305** |

**Interpretation:**

The 665 extreme feature records represent 305 distinct participant-date-source cases. A substantial proportion of these cases appear across multiple rolling windows, particularly all three windows.

This confirms that rolling-window feature records are not independent. Repeated appearances reflect overlapping feature calculations using the same underlying participant measurements.

**Decision:** Retain the rolling-window features, but distinguish feature-record counts from distinct underlying cases when interpreting extreme percentage changes.

In [ ]:
# Analyse rolling-window repetition by sensor

window_repetition["Sensor"] = (
    window_repetition["Source_Feature"]
    .str.split("_")
    .str[0]
)

repetition_by_sensor = (
    window_repetition
    .groupby("Sensor")
    .agg(
        Unique_Cases=("Source_Feature", "count"),
        Single_Window=("Number_of_Window_Features",
                       lambda x: (x == 1).sum()),
        Two_Windows=("Number_of_Window_Features",
                     lambda x: (x == 2).sum()),
        Three_Windows=("Number_of_Window_Features",
                       lambda x: (x == 3).sum())
    )
)

print("ROLLING-WINDOW REPETITION BY SENSOR")
print("=" * 65)

display(repetition_by_sensor)

### 10S.2 Rolling-Window Repetition by Sensor

**Objective:** Examine whether the repetition of extreme percentage-change cases across rolling windows differs between sensor streams.

**Method:**
- Assigned each distinct participant-date-source case to its corresponding sensor.
- Counted the number of cases appearing in one, two, or all three rolling windows.

**Observed results:**

| Sensor | Unique cases | Single window | Two windows | Three windows |
|---|---:|---:|---:|---:|
| EI | 167 | 63 | 36 | 68 |
| MI | 112 | 15 | 28 | 69 |
| PEI | 5 | 0 | 3 | 2 |
| SHRI | 13 | 7 | 4 | 2 |
| SII | 1 | 0 | 0 | 1 |
| SQI | 7 | 2 | 5 | 0 |
| **Total** | **305** | **87** | **76** | **142** |

**Interpretation:**

EI and MI account for 279 of the 305 distinct cases (91.48%). MI has 69 cases appearing in all three rolling windows, compared with 68 for EI.

Repetition patterns vary across sensors. In particular, SHRI has a greater number of single-window cases, while SQI has no cases appearing in all three windows.

These findings describe how extreme feature records are distributed across rolling windows. They should not be interpreted as evidence of sensor reliability or clinical significance.

**Decision:** Retain all features and observations. No feature removal or clipping is justified by repetition patterns alone.

In [ ]:
# Analyse distinct extreme cases by participant

cases_by_participant = (
    window_repetition
    .groupby("Participant_ID")
    .size()
    .sort_values(ascending=False)
)

print("EXTREME CASE DISTRIBUTION BY PARTICIPANT")
print("=" * 65)

print("Total distinct cases:", len(window_repetition))
print("Participants with extreme cases:",
      cases_by_participant.shape[0])

print("\nTOP 10 PARTICIPANTS")

display(
    cases_by_participant
    .head(10)
    .rename("Distinct_Extreme_Cases")
    .to_frame()
)

top_3_share = (
    cases_by_participant.head(3).sum()
    / len(window_repetition) * 100
)

print("\nShare contributed by top 3 participants:",
      round(top_3_share, 2), "%")

### 10S.3 Distribution of Distinct Extreme Cases Across Participants

**Objective:** Determine whether distinct extreme percentage-change cases are distributed across participants or concentrated in a small subset.

**Method:**
- Grouped the 305 distinct participant-date-source cases by `Participant_ID`.
- Counted distinct cases for each participant.
- Ranked participants by their number of cases and calculated the contribution of the top three.

**Observed results:**
- Distinct cases: 305
- Participants with extreme cases: 18
- Top three participants:
  - GER-003: 58 cases
  - GER-002: 54 cases
  - GER-029: 39 cases
- Combined top-three contribution: 151 cases (49.51%).

**Interpretation:**

Nearly half of the distinct extreme cases are associated with three participants. This indicates that extreme percentage-change observations are not evenly distributed across the cohort.

The concentration may reflect individual measurement patterns, sensor behaviour, or genuine variation in observed activity and environmental conditions. The current analysis does not establish the underlying cause.

**Modelling implication:** Participant-level data splitting is important to prevent observations from the same participant appearing in both training and validation sets. This reduces the risk of participant-specific patterns producing overly optimistic validation results.

**Decision:** Retain the observations. Investigate participant-level patterns further if needed, and preserve participant-level separation during model evaluation.

In [ ]:
print("AVAILABLE COLUMNS:")
print(extreme_cells_combined.columns.tolist())

print("\nSAMPLE RECORDS:")
display(extreme_cells_combined.head(10))

In [ ]:
print("TYPE:", type(extreme_cells))

print("\nNUMBER OF ITEMS:", len(extreme_cells))

print("\nFIRST 10 ITEMS:")
print(extreme_cells[:10])

if len(extreme_cells) > 0:
    print("\nTYPE OF FIRST ITEM:")
    print(type(extreme_cells[0]))

In [ ]:
# Identify the actual all-sensor extreme-feature audit

print("TYPE AND CONTENT OF extreme_cells")
print("Type:", type(extreme_cells))
print("Length:", len(extreme_cells))
print("First 10 items:", extreme_cells[:10])

print("\nPOTENTIAL AUDIT DATAFRAMES")
print("=" * 65)

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if any(term in name.lower() for term in ["extreme", "pct", "change", "audit"]):
            print(f"\nVariable: {name}")
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))

In [ ]:
variables_to_check = [
    "extreme_distribution",
    "ei_mi_extremes",
    "unique_combinations",
    "total_cells",
    "all_three_windows",
    "comprehensive_denominator_audit",
    "inconsistent_combinations",
    "sensor_extremes"
]

print("STEP 10U VARIABLE VALIDATION")
print("=" * 65)

for variable in variables_to_check:
    exists = variable in globals()
    print(f"{variable}: {'Available' if exists else 'Missing'}")

In [ ]:
print("STEP 10U SUMMARY CROSS-CHECK")
print("=" * 65)

print("All-sensor extreme records:",
      len(extreme_cells_combined))

print("EI/MI extreme records:",
      len(ei_mi_extremes))

print("Distinct participant-date-source cases:",
      len(window_repetition))

print("Cases appearing in all 3 windows:",
      (window_repetition["Number_of_Window_Features"] == 3).sum())

print("Successfully reconstructed EI/MI records:",
      len(comprehensive_denominator_audit))

print("\nPreviously calculated values:")
print("extreme_distribution:", len(extreme_distribution))
print("unique_combinations:", len(unique_combinations))
print("total_cells:", total_cells)
print("all_three_windows:", len(all_three_windows))

In [ ]:
# Step 10U – Final summary of extreme percentage changes

final_extreme_summary = pd.DataFrame({
    "Investigation": [
        "All-sensor extreme feature cells",
        "EI and MI extreme feature cells",
        "Unique EI/MI participant-date-source combinations",
        "Additional repeated EI/MI feature cells",
        "Combinations appearing in all three windows",
        "EI/MI cells successfully reconstructed",
        "Window-specific validation records matched",
        "Previous values equal to zero",
        "Previous values below 1",
        "Inconsistent source measurement combinations"
    ],
    "Result": [
        len(extreme_distribution),
        len(ei_mi_extremes),
        len(unique_combinations),
        total_cells - len(unique_combinations),
        len(all_three_windows),
        len(comprehensive_denominator_audit),
        39,
        int((comprehensive_denominator_audit["Previous_Value"] == 0).sum()),
        int((comprehensive_denominator_audit["Previous_Value"].abs() < 1).sum()),
        len(inconsistent_combinations)
    ]
})

display(final_extreme_summary)

print("\nALL-SENSOR DISTRIBUTION BY SENSOR")

display(
    sensor_extremes.groupby("Sensor")
    .size()
    .rename("Extreme_Feature_Cells")
    .to_frame()
)

### Step 10U – Final Summary of Extreme Percentage-Change Investigation

**Objective**

Consolidate the investigation of extreme percentage-change features, identify their underlying patterns, and establish an appropriate strategy for subsequent model preparation.

**Observed results – All sensors**

A total of 665 extreme percentage-change feature cells were identified across six sensors.

| Sensor | Extreme feature cells |
|---|---:|
| EI | 339 |
| MI | 278 |
| PEI | 12 |
| SHRI | 21 |
| SII | 3 |
| SQI | 12 |
| **Total** | **665** |

No BRI feature cells crossed the investigated extreme threshold.

**Observed results – EI and MI investigation**

| Investigation | Result |
|---|---:|
| EI and MI extreme feature cells | 617 |
| Unique participant-date-source combinations | 279 |
| Additional repeated feature cells | 338 |
| Combinations appearing in all three windows | 137 |
| EI/MI cells successfully reconstructed | 617 |
| Window-specific validation records matched | 39 |
| Previous values equal to zero | 0 |
| Previous values below 1 | 260 |
| Inconsistent source measurement combinations | 0 |

**Observed patterns**

- EI and MI account for 617 of 665 extreme cells (92.78%).
- The remaining 48 cells are distributed across PEI, SHRI, SII and SQI.
- Some features have very small previous measurement values, amplifying percentage changes.
- Other features exhibit large absolute changes, indicating that small denominators are not the only explanation.
- Repeated rolling-window feature cells represent the same reconstructed source measurement pair for each unique participant-date-source combination.

**Interpretation**

Extreme percentage changes are not automatically evidence of erroneous sensor data. They may result from small previous values, large measurement changes, measurement variability, and the mathematical properties of percentage change.

The 617 EI/MI feature cells represent 279 unique participant-date-source combinations rather than 617 independent underlying measurement changes.

**Modelling decision**

1. Do not globally delete or clip extreme percentage-change values.
2. Preserve original source measurements.
3. Retain engineered features for subsequent training-fold-only evaluation.
4. Fit scaling, clipping, imputation and feature-selection procedures separately within each training fold.
5. Compare model performance with and without selected percentage-change features where appropriate.
6. Use participant-level validation and event-aware evaluation.

**Limitations**

This investigation does not establish whether extreme changes correspond to clinically meaningful deterioration. It is a data and feature-engineering investigation, not a clinical validation study.

The dataset contains a limited number of participants and recorded exacerbation events, so findings should be interpreted cautiously.

**Conclusion**

The extreme percentage-change investigation is complete. All 665 identified cells have been accounted for, and the EI/MI denominator investigation has been validated. No feature values were modified or removed.

### Step 11A – Verification of Participant-Level Fold Assignments

**Objective**

Verify the participant-level validation assignments before beginning training-fold-only feature preparation.

**Why this is necessary**

Feature selection, missing-value handling, scaling and other preprocessing decisions must be learned using training participants only.

Validation participants must remain separate so that information from them does not influence model preparation.

**Validation requirements**

- Every participant must belong to exactly one validation fold.
- All observations from the same participant must remain in the same fold.
- All five folds must be represented.
- The existing target labels and feature dataset must remain unchanged.

No features or labels will be modified in this step.

In [ ]:
# Step 11A – Verify participant-level fold assignments

print("MODEL DATASET")
print("=" * 65)

print("Known-target dataset shape:", known_target_model_dataset.shape)
print("Unique participants:",
      known_target_model_dataset["Participant_ID"].nunique())

print("\nFOLD-RELATED VARIABLES")
print("=" * 65)

for name, obj in list(globals().items()):
    if "fold" in name.lower():
        print(f"\nVariable: {name}")
        print("Type:", type(obj))

        if isinstance(obj, pd.DataFrame):
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))

        elif isinstance(obj, dict):
            print("Number of entries:", len(obj))
            print("Sample:", list(obj.items())[:5])

        elif isinstance(obj, (list, tuple)):
            print("Length:", len(obj))
            print("Sample:", obj[:5])

In [ ]:
extreme_cells_combined = pd.concat(
    extreme_cells,
    ignore_index=True
)

print("Combined shape:", extreme_cells_combined.shape)
print("Columns:", extreme_cells_combined.columns.tolist())

display(extreme_cells_combined.head())

### Step 11A: Combine and Validate Extreme Percentage-Change Records

The extreme percentage-change records were initially stored as 47 separate DataFrames, one for each feature group.

These DataFrames were concatenated into a single table containing 665 records and five columns.

Each record represents an extreme percentage-change feature observation. Records may repeat for the same participant, date and source feature because multiple rolling windows (3-day, 7-day and 14-day) can generate separate features from the same underlying measurement.

Therefore, the 665 records should not be interpreted as 665 independent observations or events.

In [ ]:
# Derive sensor from the source feature name
extreme_cells_combined["Sensor"] = (
    extreme_cells_combined["Source_Feature"]
    .str.split("_")
    .str[0]
)

# Count records by sensor
sensor_summary = (
    extreme_cells_combined
    .groupby("Sensor")
    .agg(
        Extreme_Feature_Cells=("Feature", "count"),
        Unique_Participants=("Participant_ID", "nunique"),
        Unique_Participant_Dates=(
            "Date",
            "nunique"
        )
    )
)

# Count distinct participant-date-source combinations
unique_source_cases = (
    extreme_cells_combined[
        ["Participant_ID", "Date", "Source_Feature"]
    ]
    .drop_duplicates()
)

print("TOTAL EXTREME FEATURE CELLS:", len(extreme_cells_combined))
print("UNIQUE PARTICIPANT-DATE-SOURCE CASES:", len(unique_source_cases))

print("\nSENSOR DISTRIBUTION")
display(sensor_summary)

In [ ]:
# Distinct participant-date combinations per sensor
unique_participant_dates = (
    extreme_cells_combined[
        ["Sensor", "Participant_ID", "Date"]
    ]
    .drop_duplicates()
    .groupby("Sensor")
    .size()
    .rename("Unique_Participant_Dates")
)

# Distinct participant-date-source combinations per sensor
unique_source_cases_by_sensor = (
    extreme_cells_combined[
        ["Sensor", "Participant_ID", "Date", "Source_Feature"]
    ]
    .drop_duplicates()
    .groupby("Sensor")
    .size()
    .rename("Unique_Source_Cases")
)

# Build corrected summary
sensor_summary_corrected = (
    extreme_cells_combined
    .groupby("Sensor")
    .agg(
        Extreme_Feature_Cells=("Feature", "count"),
        Unique_Participants=("Participant_ID", "nunique")
    )
    .join(unique_participant_dates)
    .join(unique_source_cases_by_sensor)
)

display(sensor_summary_corrected)

print(
    "\nTotal unique participant-date-source cases:",
    sensor_summary_corrected["Unique_Source_Cases"].sum()
)

### Step 11A: Results and Interpretation

The combined extreme percentage-change table contained 665 feature cells across six sensors.

After removing repeated rolling-window representations, these corresponded to 305 distinct participant-date-source-feature cases and 181 distinct participant-date combinations.

Sensor-level findings:

- EI: 339 extreme feature cells, 15 participants and 167 distinct source cases.
- MI: 278 extreme feature cells, 12 participants and 112 distinct source cases.
- PEI: 12 extreme feature cells and 5 distinct source cases.
- SHRI: 21 extreme feature cells and 13 distinct source cases.
- SII: 3 extreme feature cells and 1 distinct source case.
- SQI: 12 extreme feature cells and 7 distinct source cases.

EI and MI together accounted for 617 of the 665 extreme feature cells (approximately 92.8%) and 279 of the 305 distinct source cases (approximately 91.5%).

**Interpretation:** Extreme percentage changes were concentrated primarily in EI and MI measurements. Multiple rolling-window features can represent the same underlying source measurement change, so feature-cell counts should not be interpreted as independent observations or events.

These findings describe the distribution of extreme engineered features; they do not establish that the underlying changes are errors or clinically meaningful events.

### Step 11B: Participant-Level Missing Measurement Audit

This step investigates missing sensor measurements across participants for BRI, MI, PEI, SHRI and SQI.

The analysis focuses on actual measurement columns and excludes timestamps, participant metadata and validity flags.

The objective is to identify:
- Which measurements contain missing values.
- How many missing values each measurement contains.
- How many participants are affected by each measurement.
- Whether missingness is concentrated among particular participants.

No missing values are imputed or removed during this audit.

In [ ]:
# Inspect columns available in the daily sensor source dataset

print("DAILY SOURCE DATASET")
print("Shape:", daily_source.shape)

print("\nAVAILABLE COLUMNS BY SENSOR")

sensors_to_audit = ["BRI", "MI", "PEI", "SHRI", "SQI"]

for sensor in sensors_to_audit:
    sensor_columns = [
        col for col in daily_source.columns
        if col.startswith(sensor + "_")
    ]

    print(f"\n{sensor}: {len(sensor_columns)} columns")
    print(sensor_columns)



In [ ]:
# Explicit list of actual measurement columns

measurement_columns = {
    "BRI": [
        "BRI_rf_mean_mean",
        "BRI_rf_mean_min",
        "BRI_rf_mean_max",
        "BRI_rf_std_mean",
        "BRI_rf_min_mean",
        "BRI_rf_max_mean"
    ],

    "MI": [
        "MI_gs_mean",
        "MI_gs_std_mean",
        "MI_sl_mean",
        "MI_sl_std_mean",
        "MI_smwd_mean",
        "MI_smwd_std_mean",
        "MI_walked_time_mean",
        "MI_walked_dist_mean"
    ],

    "PEI": [
        "PEI_FEV1_mean",
        "PEI_FEV1_low_min",
        "PEI_FEV1_high_max",
        "PEI_PEF_mean",
        "PEI_PEF_low_min",
        "PEI_PEF_high_max",
        "PEI_HR_mean",
        "PEI_SpO2_mean"
    ],

    "SHRI": [
        "SHRI_hr_mean",
        "SHRI_hr_std_mean",
        "SHRI_hr_max",
        "SHRI_hr_min"
    ],

    "SQI": [
        "SQI_num_movements_mean",
        "SQI_num_movements_max",
        "SQI_WASO_mean",
        "SQI_sleep_duration_mean",
        "SQI_SE_mean"
    ]
}

# Verify that every requested measurement exists
all_measurements = [
    column
    for columns in measurement_columns.values()
    for column in columns
]

missing_columns = [
    column for column in all_measurements
    if column not in daily_source.columns
]

print("TOTAL MEASUREMENT COLUMNS:", len(all_measurements))
print("MISSING COLUMN NAMES:", missing_columns)

# Calculate missingness for each measurement
missingness_results = []

total_participants = daily_source["Participant_ID"].nunique()

for sensor, columns in measurement_columns.items():

    for column in columns:

        missing_count = daily_source[column].isna().sum()

        affected_participants = (
            daily_source.loc[
                daily_source[column].isna(),
                "Participant_ID"
            ].nunique()
        )

        missingness_results.append({
            "Sensor": sensor,
            "Measurement": column,
            "Missing_Values": missing_count,
            "Missing_Percentage": (
                missing_count / len(daily_source)
            ) * 100,
            "Participants_Affected": affected_participants,
            "Total_Participants": total_participants
        })

measurement_missingness = pd.DataFrame(missingness_results)

measurement_missingness = (
    measurement_missingness
    .sort_values(
        ["Sensor", "Missing_Percentage"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\nMEASUREMENT MISSINGNESS SUMMARY")
display(measurement_missingness.round(2))

In [ ]:
# Calculate participant-level missingness for every measurement

participant_measurement_missingness = []

for sensor, columns in measurement_columns.items():

    for participant_id, participant_data in daily_source.groupby("Participant_ID"):

        for column in columns:

            total_days = len(participant_data)
            missing_days = participant_data[column].isna().sum()

            participant_measurement_missingness.append({
                "Participant_ID": participant_id,
                "Sensor": sensor,
                "Measurement": column,
                "Total_Timeline_Days": total_days,
                "Missing_Days": missing_days,
                "Observed_Days": total_days - missing_days,
                "Missing_Percentage": (
                    missing_days / total_days
                ) * 100
            })

participant_measurement_missingness = pd.DataFrame(
    participant_measurement_missingness
)

# Summarize the distribution of participant-level missingness
participant_missingness_summary = (
    participant_measurement_missingness
    .groupby(["Sensor", "Measurement"])
    .agg(
        Median_Missing_Percentage=("Missing_Percentage", "median"),
        Q1_Missing_Percentage=(
            "Missing_Percentage",
            lambda x: x.quantile(0.25)
        ),
        Q3_Missing_Percentage=(
            "Missing_Percentage",
            lambda x: x.quantile(0.75)
        ),
        Minimum_Missing_Percentage=("Missing_Percentage", "min"),
        Maximum_Missing_Percentage=("Missing_Percentage", "max"),
        Participants_With_Any_Data=(
            "Observed_Days",
            lambda x: (x > 0).sum()
        ),
        Participants_Without_Data=(
            "Observed_Days",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

print("PARTICIPANT-LEVEL MISSINGNESS SUMMARY")
display(participant_missingness_summary.round(2))

#### Step 11B.1: Sensor Observation Availability

The previous analysis identified substantial missingness across the five sensors.

This analysis compares missing measurement values with the corresponding sensor observation counts.

For each participant and sensor, it calculates:
- Total days in the participant's timeline.
- Days with a positive sensor observation count.
- Days with at least one observed measurement.
- Days with sensor observations recorded but all selected measurement fields missing.

This helps distinguish limited sensor observation availability from missing measurement values despite recorded observations.

The analysis is descriptive. A positive observation count does not by itself establish that every measurement should be present or valid.

In [ ]:
# Compare sensor observation availability with measurement availability

sensor_availability_results = []

for sensor, columns in measurement_columns.items():

    observation_column = f"{sensor}_observation_count"

    for participant_id, participant_data in daily_source.groupby("Participant_ID"):

        total_days = len(participant_data)

        # Days with a positive sensor observation count
        days_with_observations = (
            participant_data[observation_column].fillna(0) > 0
        ).sum()

        # Days with at least one observed measurement
        days_with_measurements = (
            participant_data[columns].notna().any(axis=1)
        ).sum()

        # Days with observations recorded but all measurements missing
        observations_without_measurements = (
            (participant_data[observation_column].fillna(0) > 0)
            & participant_data[columns].isna().all(axis=1)
        ).sum()

        sensor_availability_results.append({
            "Participant_ID": participant_id,
            "Sensor": sensor,
            "Timeline_Days": total_days,
            "Days_With_Observations": days_with_observations,
            "Days_With_Measurements": days_with_measurements,
            "Observations_Without_Measurements": (
                observations_without_measurements
            ),
            "Observation_Coverage_Percentage": (
                days_with_observations / total_days * 100
            ),
            "Measurement_Coverage_Percentage": (
                days_with_measurements / total_days * 100
            )
        })

participant_sensor_availability = pd.DataFrame(
    sensor_availability_results
)

# Summarize availability across participants
sensor_availability_summary = (
    participant_sensor_availability
    .groupby("Sensor")
    .agg(
        Participants_With_Observations=(
            "Days_With_Observations",
            lambda x: (x > 0).sum()
        ),
        Participants_With_Measurements=(
            "Days_With_Measurements",
            lambda x: (x > 0).sum()
        ),
        Median_Observation_Coverage=(
            "Observation_Coverage_Percentage", "median"
        ),
        Median_Measurement_Coverage=(
            "Measurement_Coverage_Percentage", "median"
        ),
        Total_Observation_Days=(
            "Days_With_Observations", "sum"
        ),
        Total_Measurement_Days=(
            "Days_With_Measurements", "sum"
        ),
        Total_Observations_Without_Measurements=(
            "Observations_Without_Measurements", "sum"
        )
    )
    .reset_index()
)

print("SENSOR OBSERVATION VS MEASUREMENT AVAILABILITY")
display(sensor_availability_summary.round(2))

In [ ]:
# Inspect observation and validity counts by sensor

for sensor, columns in measurement_columns.items():

    participant_columns = [
        col for col in daily_source.columns
        if col.startswith(sensor + "_")
        and (
            "observation_count" in col
            or "valid_observation_count" in col
        )
    ]

    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print("=" * 60)

    for column in participant_columns:
        values = daily_source[column]

        print(
            f"{column}: "
            f"total={values.sum():.0f}, "
            f"positive_days={(values.fillna(0) > 0).sum()}, "
            f"missing={values.isna().sum()}"
        )

    measurement_days = daily_source[columns].notna().any(axis=1).sum()

    print("Days with at least one measurement:", measurement_days)

    print(
        "Days with observations but no measurements:",
        (
            (daily_source[f"{sensor}_observation_count"].fillna(0) > 0)
            & daily_source[columns].isna().all(axis=1)
        ).sum()
    )

In [ ]:
# Validate valid-observation days against measurement days per participant

participant_validity_check = []

for sensor, columns in measurement_columns.items():

    valid_count_column = f"{sensor}_valid_observation_count"

    for participant_id, participant_data in daily_source.groupby("Participant_ID"):

        valid_days = (
            participant_data[valid_count_column].fillna(0) > 0
        ).sum()

        measurement_days = (
            participant_data[columns].notna().any(axis=1)
        ).sum()

        participant_validity_check.append({
            "Participant_ID": participant_id,
            "Sensor": sensor,
            "Valid_Observation_Days": valid_days,
            "Measurement_Days": measurement_days,
            "Difference": valid_days - measurement_days
        })

participant_validity_check = pd.DataFrame(
    participant_validity_check
)

validity_mismatch_summary = (
    participant_validity_check
    .groupby("Sensor")
    .agg(
        Participants_Checked=("Participant_ID", "nunique"),
        Participants_With_Mismatch=(
            "Difference",
            lambda x: (x != 0).sum()
        ),
        Total_Absolute_Difference=(
            "Difference",
            lambda x: x.abs().sum()
        )
    )
    .reset_index()
)

print("PARTICIPANT-LEVEL VALIDITY ALIGNMENT")
display(validity_mismatch_summary)

print("\nMISMATCH DETAILS")
display(
    participant_validity_check[
        participant_validity_check["Difference"] != 0
    ]
)

### Step 11B: Participant-Level Missingness and Sensor Validity Investigation

**Objective**

Investigate missing measurements across BRI, MI, PEI, SHRI, and SQI, distinguish missing measurement values from sensor observation availability, and examine whether missingness is associated with observation validity.

**Methodology**

1. Identified the measurement columns belonging to each sensor.
2. Calculated measurement-level missingness across the common participant-date timeline.
3. Calculated participant-level missingness percentages for each measurement.
4. Compared the number of days with sensor observations against the number of days with actual measurements.
5. Examined total, invalid, data-valid, and index-valid observation counts where available.
6. Compared valid-observation days with measurement-available days separately for every participant.

**Observed Results**

- All 74 participants had at least one missing date for each investigated sensor.
- MI had the broadest participant-level measurement coverage.
- BRI, SHRI, and SQI had particularly limited measurement availability.
- Across the dataset, the

### STEP11C

In [ ]:
# Step 11C: Inspect PRVI columns and measurement availability fields

prvi_columns = [
    col for col in daily_source.columns
    if col.startswith("PRVI_")
]

print("PRVI COLUMNS")
print("=" * 60)

for column in prvi_columns:
    print(column)

print("\nTotal PRVI columns:", len(prvi_columns))

# Step 11C: Investigate PRVI participant-level availability

prvi_measurement_columns = [
    "PRVI_HR_mean",
    "PRVI_HR_low_min",
    "PRVI_HR_high_max",
    "PRVI_std_RR_mean",
    "PRVI_std_RR_low_min",
    "PRVI_std_RR_high_max",
    "PRVI_RMSSD_mean",
    "PRVI_RMSSD_low_min",
    "PRVI_RMSSD_high_max",
    "PRVI_nRMSSD_mean",
    "PRVI_nRMSSD_low_min",
    "PRVI_nRMSSD_high_max"
]

prvi_participant_summary = []

for participant_id, participant_data in daily_source.groupby("Participant_ID"):

    observations = (
        participant_data["PRVI_observation_count"].fillna(0) > 0
    )

    measurements = (
        participant_data[prvi_measurement_columns].notna().any(axis=1)
    )

    valid_observations = (
        participant_data["PRVI_valid_observation_count"].fillna(0) > 0
    )

    prvi_participant_summary.append({
        "Participant_ID": participant_id,
        "Days_With_Observations": observations.sum(),
        "Days_With_Measurements": measurements.sum(),
        "Days_With_Valid_Observations": valid_observations.sum(),
        "Observations_Without_Measurements": (
            observations & ~measurements
        ).sum(),
        "Valid_Measurement_Day_Difference": (
            valid_observations.sum() - measurements.sum()
        )
    })

prvi_participant_summary = pd.DataFrame(prvi_participant_summary)

print("PRVI PARTICIPANT AVAILABILITY SUMMARY")
print("=" * 70)

display(
    prvi_participant_summary[
        [
            "Days_With_Observations",
            "Days_With_Measurements",
            "Days_With_Valid_Observations",
            "Observations_Without_Measurements"
        ]
    ].agg(["count", "sum", "median", "min", "max"]).T
)

print("\nPARTICIPANT COUNTS")
print(
    "Participants with PRVI observations:",
    (prvi_participant_summary["Days_With_Observations"] > 0).sum()
)
print(
    "Participants with PRVI measurements:",
    (prvi_participant_summary["Days_With_Measurements"] > 0).sum()
)
print(
    "Participants with no PRVI observations:",
    (prvi_participant_summary["Days_With_Observations"] == 0).sum()
)
print(
    "Participants with observations but no measurements:",
    (
        (prvi_participant_summary["Days_With_Observations"] > 0)
        & (prvi_participant_summary["Days_With_Measurements"] == 0)
    ).sum()
)
print(
    "Participants with validity/measurement-day mismatch:",
    (prvi_participant_summary["Valid_Measurement_Day_Difference"] != 0).sum()
)

print("\nPARTICIPANTS WITH NO PRVI OBSERVATIONS")
display(
    prvi_participant_summary[
        prvi_participant_summary["Days_With_Observations"] == 0
    ]
)### Step 11C: PRVI Participant-Level Availability Investigation

**Objective**

Investigate PRVI availability separately because this sensor is absent for some participants and has irregular, limited measurement coverage.

**Methodology**

1. Identified the 12 PRVI measurement columns across HR, std_RR, RMSSD and nRMSSD.
2. Calculated observation days, measurement days and valid-observation days for each participant.
3. Identified participants without PRVI observations.
4. Identified participants with observations but no actual measurements.
5. Compared valid-observation days against measurement-available days at participant level.

**Observed Results**

Across 74 participants:

| Metric | Result |
|---|---:|
| Participants with PRVI observations | 65 |
| Participants with PRVI measurements | 59 |
| Participants with no PRVI observations | 9 |
| Participants with observations but no measurements | 6 |
| Total observation days | 2,342 |
| Total measurement days | 1,935 |
| Observation days without measurements | 407 |

The nine participants without PRVI observations were GER-020, GER-026, GER-030, GER-033, GER-037, GER-039, SPA-013, SPA-040 and SPA-046.

The median participant had 15 days with observations and 9.5 days with measurements.

Participant-level validation found zero mismatches between valid-observation days and measurement-available days.

**Interpretation**

PRVI has substantially more limited participant coverage than the other investigated sensors. Its availability also varies considerably across participants.

The results distinguish participants with no PRVI observations from participants who have observations but no available measurements.

**Preprocessing Decisions**

- Preserve PRVI measurements and missing values.
- Retain PRVI validity and observation-count features.
- Do not impute missing PRVI measurements with zero.
- Do not remove PRVI solely because nine participants lack observations.
- Evaluate PRVI feature availability and predictive contribution within training folds before deciding whether to retain individual PRVI predictors.

**Limitations**

The investigation describes measurement availability, not the reason for missingness. It does not establish whether missing PRVI data reflects device deployment, collection schedules, data transmission or other causes.

In [ ]:
# Step 11D: Compare availability across PRVI measurement types

prvi_measurement_groups = {
    "HR": [
        "PRVI_HR_mean",
        "PRVI_HR_low_min",
        "PRVI_HR_high_max"
    ],
    "std_RR": [
        "PRVI_std_RR_mean",
        "PRVI_std_RR_low_min",
        "PRVI_std_RR_high_max"
    ],
    "RMSSD": [
        "PRVI_RMSSD_mean",
        "PRVI_RMSSD_low_min",
        "PRVI_RMSSD_high_max"
    ],
    "nRMSSD": [
        "PRVI_nRMSSD_mean",
        "PRVI_nRMSSD_low_min",
        "PRVI_nRMSSD_high_max"
    ]
}

prvi_type_summary = []

for measurement_type, columns in prvi_measurement_groups.items():

    available_days = daily_source[columns].notna().any(axis=1)

    participant_coverage = (
        daily_source.assign(available=available_days)
        .groupby("Participant_ID")["available"]
        .sum()
    )

    prvi_type_summary.append({
        "Measurement_Type": measurement_type,
        "Days_With_Measurements": available_days.sum(),
        "Participants_With_Measurements": (participant_coverage > 0).sum(),
        "Median_Days_Per_Participant": participant_coverage.median(),
        "Minimum_Days_Per_Participant": participant_coverage.min(),
        "Maximum_Days_Per_Participant": participant_coverage.max()
    })

prvi_type_summary = pd.DataFrame(prvi_type_summary)

display(prvi_type_summary)

In [ ]:
# Step 11E: Compare PRVI measurement availability on identical dates

prvi_availability = pd.DataFrame({
    measurement_type: daily_source[columns].notna().any(axis=1)
    for measurement_type, columns in prvi_measurement_groups.items()
})

print("AVAILABILITY PATTERN COUNTS")
display(
    prvi_availability
    .value_counts()
    .rename("Participant_Date_Count")
    .reset_index()
)

print("\nPAIRWISE AVAILABILITY AGREEMENT")

for measurement_type in prvi_availability.columns:
    for comparison_type in prvi_availability.columns:
        if measurement_type < comparison_type:
            matches = (
                prvi_availability[measurement_type]
                == prvi_availability[comparison_type]
            ).sum()

            print(
                f"{measurement_type} vs {comparison_type}: "
                f"{matches}/{len(prvi_availability)} matching rows"
            )

### Step 11E: PRVI Measurement Availability Pattern Analysis

**Objective**

Determine whether PRVI measurement types are available on the same participant-date rows or whether their similar aggregate availability hides differences in their observation patterns.

**Methodology**

1. Created a Boolean availability indicator for HR, std_RR, RMSSD and nRMSSD.
2. Counted the distinct combinations of measurement availability across the common participant-date timeline.
3. Compared each pair of measurement types to calculate the number of matching availability rows.

**Observed Results**

Three distinct availability patterns were identified:

| HR | std_RR | RMSSD | nRMSSD | Participant-date rows |
|---|---|---|---|---:|
| Missing | Missing | Missing | Missing | 11,150 |
| Available | Available | Available | Available | 1,801 |
| Available | Available | Available | Missing | 134 |

Pairwise comparisons showed:

- HR, std_RR and RMSSD had identical availability across all 13,085 participant-date rows.
- nRMSSD matched each of the other three measurements on 12,951 rows.
- The remaining 134 rows contained HR, std_RR and RMSSD measurements but no nRMSSD measurement.

**Interpretation**

HR, std_RR and RMSSD have perfectly aligned measurement availability in this dataset. nRMSSD has slightly lower coverage.

A total of 11,150 participant-date rows had no PRVI measurements.

Identical availability does not establish that the underlying measurements are identical, correlated or redundant. It only establishes that their missingness patterns are aligned.

**Preprocessing Decisions**

- Retain all four measurement types for further modelling evaluation.
- Do not remove HR, std_RR or RMSSD solely because their availability patterns match.
- Evaluate nRMSSD separately because of its slightly lower coverage.
- Preserve missing values without zero imputation.

**Limitations**

The analysis describes availability patterns only. It does not establish the statistical relationships, clinical importance or predictive contribution of the underlying measurements.

In [ ]:
# Step 11F: PRVI measurement coverage by participant

prvi_participant_coverage = (
    daily_source.assign(
        PRVI_has_measurement=(
            daily_source[prvi_measurement_columns]
            .notna()
            .any(axis=1)
        )
    )
    .groupby("Participant_ID")
    .agg(
        Total_Timeline_Days=("Date", "count"),
        PRVI_Measurement_Days=("PRVI_has_measurement", "sum")
    )
    .reset_index()
)

prvi_participant_coverage["Coverage_Percentage"] = (
    prvi_participant_coverage["PRVI_Measurement_Days"]
    / prvi_participant_coverage["Total_Timeline_Days"]
    * 100
)

print("PRVI PARTICIPANT COVERAGE")
display(
    prvi_participant_coverage
    .sort_values("Coverage_Percentage", ascending=False)
)

print("\nCOVERAGE DISTRIBUTION")
display(
    prvi_participant_coverage["Coverage_Percentage"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
)

### Step 11F: PRVI Participant-Level Coverage Distribution

**Objective**

Determine how PRVI measurement availability is distributed across participants relative to each participant's available monitoring timeline.

**Methodology**

For each participant:

1. Counted the total number of participant-date rows in the common timeline.
2. Counted days with at least one available PRVI measurement.
3. Calculated measurement coverage as:

\[
\text{Coverage (\%)} =
\frac{\text{PRVI measurement days}}
{\text{Total timeline days}}\times 100
\]

4. Examined the distribution of coverage percentages across all 74 participants.

**Observed Results**

| Statistic | PRVI coverage |
|---|---:|
| Participants | 74 |
| Mean | 13.01% |
| Standard deviation | 14.63 percentage points |
| Minimum | 0% |
| 25th percentile | 0.96% |
| Median | 7.63% |
| 75th percentile | 20.36% |
| 90th percentile | 31.75% |
| Maximum | 61.83% |

Examples of higher coverage included GER-006 (61.83%), GER-024 (54.21%), SPA-005 (50.00%), GER-004 (48.39%) and GER-002 (39.06%).

**Interpretation**

PRVI measurement coverage is highly uneven across participants. The mean coverage of 13.01% is considerably higher than the median of 7.63%, indicating that participants with higher coverage raise the overall average.

Coverage percentages should be interpreted alongside absolute measurement-day counts. For example, SPA-005 has 50% coverage but only three measurement days because its timeline contains six days.

**Preprocessing Decisions**

- Retain PRVI availability information.
- Do not assume that a participant with a high coverage percentage necessarily has sufficient longitudinal data.
- Consider both measurement-day counts and coverage percentages when evaluating PRVI features.
- Assess the contribution of PRVI within participant-level training folds before final feature selection.

**Limitations**

Coverage is calculated relative to the available common participant-date timeline. It does not establish the reason for missingness or whether the available observations are sufficient for clinical or predictive use.

In [ ]:
# Step 11G: Categorise PRVI participant coverage

coverage_bins = [
    -0.001, 0, 5, 20, 40, 100
]

coverage_labels = [
    "No measurements (0%)",
    "Very low (>0–5%)",
    "Low (>5–20%)",
    "Moderate (>20–40%)",
    "Higher (>40%)"
]

prvi_participant_coverage["Coverage_Category"] = pd.cut(
    prvi_participant_coverage["Coverage_Percentage"],
    bins=coverage_bins,
    labels=coverage_labels,
    include_lowest=True
)

coverage_category_summary = (
    prvi_participant_coverage
    .groupby("Coverage_Category", observed=False)
    .agg(
        Participants=("Participant_ID", "count"),
        Median_Coverage=("Coverage_Percentage", "median"),
        Median_Measurement_Days=("PRVI_Measurement_Days", "median")
    )
    .reset_index()
)

coverage_category_summary["Participant_Percentage"] = (
    coverage_category_summary["Participants"] / 74 * 100
)

display(coverage_category_summary)

### Step 11G: Categorisation of PRVI Participant Coverage

**Objective**

Group participants into interpretable PRVI measurement-coverage categories to quantify how many participants have sufficient versus limited PRVI availability.

**Methodology**

Participants were classified using their PRVI measurement coverage percentages:

- No measurements: 0%
- Very low: above 0% to 5%
- Low: above 5% to 20%
- Moderate: above 20% to 40%
- Higher: above 40%

For each category, the participant count, percentage of the cohort, median coverage and median measurement days were calculated.

**Observed Results**

| Coverage category | Participants | Percentage |
|---|---:|---:|
| No measurements (0%) | 15 | 20.27% |
| Very low (>0–5%) | 17 | 22.97% |
| Low (>5–20%) | 23 | 31.08% |
| Moderate (>20–40%) | 15 | 20.27% |
| Higher (>40%) | 4 | 5.41% |
| **Total** | **74** | **100%** |

The median measurement days per category were:

- No measurements: 0 days
- Very low: 3 days
- Low: 22 days
- Moderate: 57 days
- Higher: 110 days

**Interpretation**

PRVI availability is limited for most participants. A total of 32 participants (43.24%) have no more than 5% coverage, while 55 participants (74.32%) have no more than 20% coverage.

Only four participants have PRVI measurement coverage above 40%.

The 15 participants with no measurements include nine participants with no PRVI observations and six participants with observations but no available measurements.

**Preprocessing Decisions**

- Retain PRVI in the feature dataset for further evaluation.
- Do not make PRVI a mandatory predictor for all participants.
- Preserve missingness and observation-availability indicators.
- Evaluate PRVI's predictive contribution and availability within training folds before deciding whether individual PRVI features should be retained.

**Limitations**

The coverage thresholds are descriptive categories used to summarise availability. They are not clinically validated thresholds and do not establish whether a participant has sufficient data for prediction.

In [ ]:
# Step 11H: PRVI measurement availability by calendar month

prvi_monthly = daily_source[
    ["Participant_ID", "Date"] + prvi_measurement_columns
].copy()

prvi_monthly["Month"] = (
    pd.to_datetime(prvi_monthly["Date"], utc=True)
    .dt.to_period("M")
    .astype(str)
)

prvi_monthly["Has_PRVI_Measurement"] = (
    prvi_monthly[prvi_measurement_columns]
    .notna()
    .any(axis=1)
)

monthly_prvi_summary = (
    prvi_monthly
    .groupby("Month")
    .agg(
        Total_Participant_Days=("Participant_ID", "size"),
        Participants=("Participant_ID", "nunique"),
        Measurement_Days=("Has_PRVI_Measurement", "sum")
    )
    .reset_index()
)

monthly_prvi_summary["Measurement_Availability_Percentage"] = (
    monthly_prvi_summary["Measurement_Days"]
    / monthly_prvi_summary["Total_Participant_Days"]
    * 100
)

display(monthly_prvi_summary)

### Step 11H: Monthly PRVI Measurement Availability

**Objective**

Investigate how PRVI measurement availability varies over calendar time.

**Methodology**

1. Grouped the common participant-date timeline by calendar month.
2. Counted total participant-date rows and unique participants represented in each month.
3. Counted rows containing at least one available PRVI measurement.
4. Calculated monthly measurement availability:

\[
\text{Monthly Availability (\%)} =
\frac{\text{Participant-date rows with PRVI measurements}}
{\text{Total participant-date rows}}
\times 100
\]

**Observed Results**

Monthly PRVI measurement availability declined over the monitoring period:

- May–July 2024: approximately 46–51%.
- August–November 2024: approximately 22–34%.
- December 2024: 12.99%.
- January–May 2025: approximately 3–7%.

The number of participants represented in each month also changed, from 9 in May 2024 to a maximum of 59 in March 2025.

April 2024 contained only two participant-date rows and is not considered representative.

**Interpretation**

The aggregate monthly results show a substantial decline in PRVI measurement availability over the monitoring period.

However, the participant composition changes between months. Therefore, the aggregate trend cannot establish whether individual participants experienced declining availability or whether the trend is partly explained by different participants contributing at different times.

**Preprocessing Decisions**

- Preserve PRVI measurements and their missingness.
- Retain temporal availability information for subsequent modelling analysis.
- Do not interpret declining aggregate availability as evidence of sensor deterioration or device failure.
- Investigate within-participant changes before drawing conclusions about temporal availability trends.

**Limitations**

The analysis is descriptive. It does not establish the cause of declining availability. Monthly percentages also depend on the participants whose monitoring timelines overlap each month.

In [ ]:
# Step 11I: Compare PRVI availability within the same participants

prvi_period_data = daily_source[
    ["Participant_ID", "Date"] + prvi_measurement_columns
].copy()

prvi_period_data["Date"] = pd.to_datetime(
    prvi_period_data["Date"], utc=True
)

prvi_period_data["Has_PRVI_Measurement"] = (
    prvi_period_data[prvi_measurement_columns]
    .notna()
    .any(axis=1)
)

early_period = prvi_period_data[
    (prvi_period_data["Date"] >= "2024-05-01")
    & (prvi_period_data["Date"] < "2024-12-01")
]

later_period = prvi_period_data[
    (prvi_period_data["Date"] >= "2024-12-01")
    & (prvi_period_data["Date"] < "2025-06-01")
]

def calculate_period_coverage(data, suffix):

    return (
        data.groupby("Participant_ID")
        .agg(
            **{
                f"Timeline_Days_{suffix}": ("Date", "count"),
                f"Measurement_Days_{suffix}": (
                    "Has_PRVI_Measurement", "sum"
                )
            }
        )
        .assign(
            **{
                f"Coverage_Percentage_{suffix}": lambda df: (
                    df[f"Measurement_Days_{suffix}"]
                    / df[f"Timeline_Days_{suffix}"] * 100
                )
            }
        )
    )

early_coverage = calculate_period_coverage(early_period, "Early")
later_coverage = calculate_period_coverage(later_period, "Later")

within_participant_coverage = early_coverage.join(
    later_coverage,
    how="inner"
)

within_participant_coverage = within_participant_coverage[
    (within_participant_coverage["Timeline_Days_Early"] > 0)
    & (within_participant_coverage["Timeline_Days_Later"] > 0)
].copy()

within_participant_coverage["Coverage_Change_Percentage_Points"] = (
    within_participant_coverage["Coverage_Percentage_Later"]
    - within_participant_coverage["Coverage_Percentage_Early"]
)

print("SAME-PARTICIPANT PERIOD COMPARISON")
print("Participants included:", len(within_participant_coverage))

display(
    within_participant_coverage[
        [
            "Coverage_Percentage_Early",
            "Coverage_Percentage_Later",
            "Coverage_Change_Percentage_Points"
        ]
    ].describe()
)

print("\nDIRECTION OF CHANGE")

display(
    pd.Series({
        "Participants with lower later coverage": (
            within_participant_coverage[
                "Coverage_Change_Percentage_Points"
            ] < 0
        ).sum(),
        "Participants with higher later coverage": (
            within_participant_coverage[
                "Coverage_Change_Percentage_Points"
            ] > 0
        ).sum(),
        "Participants with unchanged coverage": (
            within_participant_coverage[
                "Coverage_Change_Percentage_Points"
            ] == 0
        ).sum()
    }).to_frame("Participant_Count")
)

### Step 11I: Within-Participant PRVI Availability Comparison

**Objective**

Determine whether the decline in aggregate PRVI measurement availability is also observed within the same participants, rather than being explained entirely by changing participant composition.

**Methodology**

The monitoring period was divided into:

- Early period: May–November 2024.
- Later period: December 2024–May 2025.

For each participant, PRVI measurement coverage was calculated separately in both periods. Only participants with at least one timeline observation in each period were included.

Coverage change was calculated as:

\[
\text{Coverage Change (percentage points)} =
\text{Later Coverage (\%)}
-
\text{Early Coverage (\%)}
\]

**Observed Results**

A total of 41 participants had timeline observations in both periods.

| Metric | Early period | Later period |
|---|---:|---:|
| Mean coverage | 32.99% | 6.73% |
| Median coverage | 28.28% | 0.00% |
| 25th percentile | 7.81% | 0.00% |
| 75th percentile | 51.28% | 11.43% |

Mean coverage declined by 26.27 percentage points.

Among the 41 participants:

- 36 had lower later-period coverage.
- 2 had higher later-period coverage.
- 3 had unchanged coverage.

**Interpretation**

The decline in PRVI measurement availability is also evident within the same participants. Therefore, the aggregate decline cannot be explained entirely by changes in which participants contributed data over time.

The median later-period coverage of zero indicates that at least half of the 41 participants had no PRVI measurement days during the later period.

**Preprocessing Decisions**

- Preserve PRVI measurements and their availability indicators.
- Retain the temporal availability findings for modelling considerations.
- Do not assume PRVI is consistently available throughout the monitoring period.
- Avoid interpreting missingness as sensor failure without supporting evidence.

**Limitations**

This is a descriptive comparison of two monitoring periods. It does not identify the operational or clinical cause of declining availability. The 41 participants are those with timeline observations in both periods; findings should not automatically be generalised to all 74 participants.

In [ ]:
# Step 11J: Identify participants with the largest PRVI coverage declines

prvi_decline_ranking = (
    within_participant_coverage
    .reset_index()
    [
        [
            "Participant_ID",
            "Timeline_Days_Early",
            "Measurement_Days_Early",
            "Coverage_Percentage_Early",
            "Timeline_Days_Later",
            "Measurement_Days_Later",
            "Coverage_Percentage_Later",
            "Coverage_Change_Percentage_Points"
        ]
    ]
    .sort_values(
        "Coverage_Change_Percentage_Points",
        ascending=True
    )
)

print("10 LARGEST PRVI COVERAGE DECLINES")
display(prvi_decline_ranking.head(10))

print("\n10 SMALLEST DECLINES / LARGEST IMPROVEMENTS")
display(prvi_decline_ranking.tail(10))

### Step 11J: Identification of Participants with the Largest PRVI Coverage Changes

**Objective**

Identify participants with the largest decreases and improvements in PRVI measurement coverage between the early and later monitoring periods.

**Methodology**

Used the within-participant coverage results from Step 11I.

Participants were ranked by:

\[
\text{Coverage Change (percentage points)}
=
\text{Later Coverage (\%)}
-
\text{Early Coverage (\%)}
\]

The ten largest declines and ten smallest declines/largest improvements were inspected, together with their absolute timeline and measurement-day counts.

**Observed Results**

The largest declines included:

| Participant | Early coverage | Later coverage | Change |
|---|---:|---:|---:|
| SPA-024 | 92.31% | 14.71% | -77.60 pp |
| SPA-019 | 88.64% | 17.03% | -71.60 pp |
| GER-021 | 82.35% | 21.43% | -60.92 pp |
| SPA-003 | 58.97% | 0.00% | -58.97 pp |
| SPA-026 | 66.67% | 11.43% | -55.24 pp |

Only two participants had increased coverage:

- GER-006: +2.29 percentage points.
- SPA-017: +3.24 percentage points.

Three participants had unchanged coverage, all at zero.

**Interpretation**

The largest decreases occurred among several participants who initially had relatively good PRVI availability. Other participants had little or no PRVI availability even during the early period, meaning that their smaller declines should not be interpreted as consistently good coverage.

The participant-level results reinforce the finding that PRVI availability declined within the same participants, although the extent of decline varied considerably.

**Preprocessing Decisions**

- Preserve PRVI measurements and missingness indicators.
- Retain participant-level temporal coverage information for modelling considerations.
- Do not interpret small declines as evidence of adequate availability when initial coverage was already low.
- Do not infer device failure or clinical deterioration from these patterns.

**Limitations**

This analysis identifies descriptive coverage changes between two periods. It does not establish why PRVI availability changed or whether the missingness is associated with participant characteristics, device deployment or clinical outcomes.

In [ ]:
# Step 11K: Consolidated measurement availability across all sensors

all_sensor_measurements = {
    **measurement_columns,
    "PRVI": prvi_measurement_columns
}

sensor_availability_summary = []

for sensor, columns in all_sensor_measurements.items():

    has_measurement = daily_source[columns].notna().any(axis=1)

    participant_days = (
        daily_source.assign(
            has_measurement=has_measurement
        )
        .groupby("Participant_ID")
        .agg(
            Timeline_Days=("Date", "count"),
            Measurement_Days=("has_measurement", "sum")
        )
    )

    sensor_availability_summary.append({
        "Sensor": sensor,
        "Total_Measurement_Days": has_measurement.sum(),
        "Timeline_Availability_Percentage": (
            has_measurement.mean() * 100
        ),
        "Participants_With_Measurements": (
            participant_days["Measurement_Days"] > 0
        ).sum(),
        "Median_Measurement_Days_Per_Participant": (
            participant_days["Measurement_Days"].median()
        ),
        "Maximum_Measurement_Days": (
            participant_days["Measurement_Days"].max()
        )
    })

sensor_availability_summary = pd.DataFrame(
    sensor_availability_summary
).sort_values(
    "Timeline_Availability_Percentage",
    ascending=False
).reset_index(drop=True)

display(sensor_availability_summary)

In [ ]:
# Step 11K: Identify EI and SII measurement columns

for sensor in ["EI", "SII"]:
    columns = [
        col for col in daily_source.columns
        if col.startswith(f"{sensor}_")
    ]

    print(f"\n{sensor} columns ({len(columns)}):")
    print(columns)

In [ ]:
# Step 11K: Consolidated measurement availability across all 8 sensors

all_sensor_measurements = {
    **measurement_columns,
    "EI": [
        "EI_Temp_mean", "EI_Temp_std_mean",
        "EI_Temp_min", "EI_Temp_max",
        "EI_Hum_mean", "EI_Hum_std_mean",
        "EI_Hum_min", "EI_Hum_max",
        "EI_AQI_mean", "EI_AQI_std_mean",
        "EI_AQI_min", "EI_AQI_max",
        "EI_Lux_mean", "EI_Lux_std_mean",
        "EI_Lux_min", "EI_Lux_max"
    ],
    "SII": [
        "SII_mean", "SII_std_mean",
        "SII_min", "SII_max"
    ],
    "PRVI": prvi_measurement_columns
}

sensor_availability_summary = []

for sensor, columns in all_sensor_measurements.items():

    has_measurement = daily_source[columns].notna().any(axis=1)

    participant_days = (
        daily_source.assign(has_measurement=has_measurement)
        .groupby("Participant_ID")
        .agg(
            Timeline_Days=("Date", "count"),
            Measurement_Days=("has_measurement", "sum")
        )
    )

    sensor_availability_summary.append({
        "Sensor": sensor,
        "Total_Measurement_Days": int(has_measurement.sum()),
        "Timeline_Availability_Percentage": (
            has_measurement.mean() * 100
        ),
        "Participants_With_Measurements": int(
            (participant_days["Measurement_Days"] > 0).sum()
        ),
        "Median_Measurement_Days_Per_Participant": (
            participant_days["Measurement_Days"].median()
        ),
        "Maximum_Measurement_Days": int(
            participant_days["Measurement_Days"].max()
        )
    })

sensor_availability_summary = (
    pd.DataFrame(sensor_availability_summary)
    .sort_values(
        "Timeline_Availability_Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(sensor_availability_summary)

### Step 11K: Consolidated Measurement Availability Across All Sensors

**Objective**

Compare the measurement availability of all eight sensor streams across the complete participant-date timeline.

**Methodology**

For each sensor, its actual measurement columns were selected, excluding observation counts, validity indicators and audit columns.

A participant-date was classified as having measurements if at least one of the selected measurement columns was non-missing.

The following metrics were calculated:

- Total participant-date rows with measurements.
- Percentage of the complete participant-date timeline with measurements.
- Number of participants with at least one measurement.
- Median measurement days per participant.
- Maximum measurement days for any participant.

**Observed Results**

| Sensor | Measurement days | Timeline availability (%) | Participants with measurements | Median measurement days |
|---|---:|---:|---:|---:|
| MI | 3491 | 26.68 | 67 | 26.0 |
| SII | 2792 | 21.34 | 51 | 16.0 |
| EI | 2619 | 20.02 | 52 | 14.5 |
| PEI | 2210 | 16.89 | 65 | 11.0 |
| PRVI | 1935 | 14.79 | 59 | 9.5 |
| BRI | 700 | 5.35 | 32 | 0.0 |
| SHRI | 542 | 4.14 | 31 | 0.0 |
| SQI | 507 | 3.87 | 29 | 0.0 |

**Interpretation**

MI had the highest overall measurement availability, followed by SII and EI.

PEI and PRVI had lower timeline availability, although measurements were available for a relatively large number of participants.

BRI, SHRI and SQI had particularly sparse measurement availability. Their median measurement days per participant were zero, indicating that at least half of the participants had no measurements from these sensors.

The comparison demonstrates that sensor availability varies substantially across both participants and time.

**Preprocessing Decisions**

- Retain all eight sensor streams at this stage.
- Preserve missing values and sensor-specific availability information.
- Do not automatically remove sensors based solely on low overall availability.
- Evaluate feature usefulness and missingness handling within training folds during model preparation.

**Limitations**

This is a sensor-level availability analysis. A participant-date is considered available when at least one selected measurement column is non-missing. It does not establish that every measurement from that sensor is available or valid on that date.

The results describe recorded data availability and do not establish why measurements are missing.

In [ ]:
# Step 11L: Measurement-level missingness across all eight sensors

feature_missingness_by_sensor = []

for sensor, columns in all_sensor_measurements.items():

    for column in columns:

        missing_count = daily_source[column].isna().sum()

        feature_missingness_by_sensor.append({
            "Sensor": sensor,
            "Feature": column,
            "Available_Days": int(
                daily_source[column].notna().sum()
            ),
            "Missing_Days": int(missing_count),
            "Missing_Percentage": (
                missing_count / len(daily_source) * 100
            )
        })

feature_missingness_by_sensor = pd.DataFrame(
    feature_missingness_by_sensor
)

display(
    feature_missingness_by_sensor.sort_values(
        ["Sensor", "Missing_Percentage"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

In [ ]:
# Step 11L: Summarize feature availability within each sensor

feature_availability_summary = (
    feature_missingness_by_sensor
    .groupby("Sensor")
    .agg(
        Total_Features=("Feature", "count"),
        Best_Availability_Percentage=(
            "Missing_Percentage",
            lambda x: 100 - x.min()
        ),
        Worst_Availability_Percentage=(
            "Missing_Percentage",
            lambda x: 100 - x.max()
        ),
        Median_Availability_Percentage=(
            "Missing_Percentage",
            lambda x: 100 - x.median()
        )
    )
    .reset_index()
)

display(
    feature_availability_summary.sort_values(
        "Median_Availability_Percentage",
        ascending=False
    )
)

### Step 11L: Feature-Level Measurement Availability

**Objective**

Investigate missingness at the individual measurement-feature level rather than relying only on sensor-level availability.

**Methodology**

For each actual measurement column across all eight sensors, calculated:

- Number of participant-date rows with an available measurement.
- Number of participant-date rows with a missing measurement.
- Percentage of missing values.

The feature-level results were then grouped by sensor to calculate:

- Number of measurement features.
- Best feature availability.
- Worst feature availability.
- Median feature availability.

**Observed Results**

| Sensor | Features | Best availability (%) | Worst availability (%) | Median availability (%) |
|---|---:|---:|---:|---:|
| MI | 8 | 26.68 | 26.68 | 26.68 |
| SII | 4 | 21.34 | 21.34 | 21.34 |
| EI | 16 | 20.02 | 20.02 | 20.02 |
| PEI | 8 | 16.89 | 13.73 | 16.69 |
| PRVI | 12 | 14.79 | 13.76 | 14.79 |
| BRI | 6 | 5.35 | 5.34 | 5.35 |
| SHRI | 4 | 4.14 | 4.14 | 4.14 |
| SQI | 5 | 3.87 | 3.83 | 3.87 |

**Interpretation**

MI, SII and EI measurement features had identical availability within their respective sensor streams.

PEI and PRVI showed some variation in feature availability. In particular, PRVI nRMSSD measurements were less available than the other PRVI measurement types.

BRI, SHRI and SQI had consistently low availability across their individual features.

Identical availability does not establish identical measurement values or statistical redundancy. It only indicates that the features are available on the same or a similar set of participant-date rows.

**Preprocessing Decisions**

- Preserve individual measurement features at this stage.
- Retain feature-level missingness information.
- Do not remove features solely because they share availability patterns.
- Consider feature-level availability during training-fold feature selection and missing-data handling.

**Limitations**

Availability is calculated across the complete participant-date timeline. It describes recorded measurement presence, not clinical validity, measurement quality or predictive usefulness.

In [ ]:
# Step 11M: Detailed PEI feature availability

pei_availability = (
    feature_missingness_by_sensor[
        feature_missingness_by_sensor["Sensor"] == "PEI"
    ]
    .sort_values(
        "Missing_Percentage",
        ascending=True
    )
    .reset_index(drop=True)
)

display(pei_availability)

### Step 11M: Detailed PEI Measurement Availability

**Objective**

Investigate the availability differences between individual PEI measurement features.

**Methodology**

Inspected the eight PEI measurement columns and compared their available participant-date rows and missingness percentages.

**Observed Results**

| Measurement group | Features | Available days | Missingness |
|---|---|---:|---:|
| FEV1 / PEF ranges | FEV1 low/high, PEF low/high | 2,210 | 83.11% |
| FEV1 / PEF means | FEV1 mean, PEF mean | 2,158 | 83.51% |
| HR / SpO2 | Heart rate, oxygen saturation | 1,796 | 86.27% |

All four range features had identical availability. FEV1 and PEF mean features also had identical availability. HR and SpO2 likewise had identical availability.

**Interpretation**

PEI lung-function range measurements were more frequently available than the mean measurements. Heart rate and oxygen saturation had the lowest availability among the PEI measurements.

The results indicate that PEI measurement availability differs by measurement type, and that treating all PEI features as equally available would overlook these differences.

**Preprocessing Decisions**

- Preserve all eight PEI measurement features.
- Retain their individual missingness patterns.
- Do not impute or remove features based solely on their availability.
- Evaluate their usefulness and missing-data handling within training folds.

**Limitations**

The analysis describes measurement presence across the participant-date timeline. It does not establish why individual measurements are missing or whether missingness is clinically meaningful.

In [ ]:
# Step 11N: Detailed PRVI feature availability

prvi_availability = (
    feature_missingness_by_sensor[
        feature_missingness_by_sensor["Sensor"] == "PRVI"
    ]
    .sort_values(
        "Missing_Percentage",
        ascending=True
    )
    .reset_index(drop=True)
)

display(prvi_availability)


### Step 11N: Detailed PRVI Measurement Availability

**Objective**

Investigate the availability of individual PRVI measurement features across HR, std_RR, RMSSD and nRMSSD.

**Methodology**

Inspected all 12 PRVI measurement columns and calculated available participant-date rows and missingness percentages for each feature.

**Observed Results**

| Measurement group | Features | Available days | Missingness |
|---|---|---:|---:|
| HR | Mean, low, high | 1,935 | 85.21% |
| std_RR | Mean, low, high | 1,935 | 85.21% |
| RMSSD | Mean, low, high | 1,935 | 85.21% |
| nRMSSD | Mean, low, high | 1,801 | 86.24% |

HR, std_RR and RMSSD had identical availability across their nine features. All three nRMSSD features had 134 fewer available participant-date rows.

**Interpretation**

The results confirm that nRMSSD has slightly lower availability than the other PRVI measurement groups. This is consistent with the availability-pattern analysis performed in Step 11E.

Identical availability indicates that measurements are present on the same participant-date rows. It does not establish that their values or predictive information are identical.

**Preprocessing Decisions**

- Retain all 12 PRVI measurement features.
- Preserve individual missingness patterns.
- Do not remove nRMSSD solely because it has slightly lower availability.
- Evaluate feature usefulness and missing-data handling within training folds.

**Limitations**

The analysis describes measurement availability and does not establish the reason for missing measurements or their clinical significance.

In [ ]:
# Step 11O: Create a sensor availability and preprocessing decision summary

sensor_decisions = pd.DataFrame([
    {
        "Sensor": "MI",
        "Availability_Category": "Highest",
        "Preprocessing_Decision": "Retain all measurement features"
    },
    {
        "Sensor": "SII",
        "Availability_Category": "Moderate",
        "Preprocessing_Decision": "Retain all measurement features"
    },
    {
        "Sensor": "EI",
        "Availability_Category": "Moderate",
        "Preprocessing_Decision": "Retain all measurement features"
    },
    {
        "Sensor": "PEI",
        "Availability_Category": "Limited",
        "Preprocessing_Decision": "Retain; monitor feature-level missingness"
    },
    {
        "Sensor": "PRVI",
        "Availability_Category": "Limited",
        "Preprocessing_Decision": "Retain; monitor participant and temporal coverage"
    },
    {
        "Sensor": "BRI",
        "Availability_Category": "Very low",
        "Preprocessing_Decision": "Retain; evaluate within training folds"
    },
    {
        "Sensor": "SHRI",
        "Availability_Category": "Very low",
        "Preprocessing_Decision": "Retain; evaluate within training folds"
    },
    {
        "Sensor": "SQI",
        "Availability_Category": "Very low",
        "Preprocessing_Decision": "Retain; evaluate within training folds"
    }
])

display(sensor_decisions)

### Step 11O: Consolidated Sensor Availability and Preprocessing Decisions

**Objective**

Consolidate the sensor-level and feature-level availability findings into provisional preprocessing decisions for subsequent model preparation.

**Observed Results**

| Sensor | Availability category | Preprocessing decision |
|---|---|---|
| MI | Highest | Retain all measurement features |
| SII | Moderate | Retain all measurement features |
| EI | Moderate | Retain all measurement features |
| PEI | Limited | Retain; monitor feature-level missingness |
| PRVI | Limited | Retain; monitor participant and temporal coverage |
| BRI | Very low | Retain; evaluate within training folds |
| SHRI | Very low | Retain; evaluate within training folds |
| SQI | Very low | Retain; evaluate within training folds |

**Interpretation**

Measurement availability varies substantially across the eight sensor streams. MI has the highest overall availability, while BRI, SHRI and SQI have particularly sparse measurements.

PEI and PRVI require additional attention because their individual measurement features have different availability patterns. PRVI also exhibits substantial participant-level and temporal variation.

**Preprocessing Decisions**

- Retain all eight sensor streams at this stage.
- Preserve missing values and availability indicators.
- Do not remove features solely because of high missingness.
- Evaluate feature usefulness and missing-data handling within training folds.
- Avoid making feature-selection decisions using validation or test data.
- Treat the availability categories as descriptive rather than clinical classifications.

**Limitations**

These decisions are provisional. Measurement availability alone does not establish predictive usefulness, clinical importance or the reason for missing data. Final feature selection will require training-only evaluation.

In [ ]:
# Step 11P: Compare sensor availability across target-label groups

target_availability = known_target_model_dataset[
    ["Participant_ID", "Date", "exacerbation_within_7d"]
].copy()

target_availability["Target_Group"] = (
    target_availability["exacerbation_within_7d"]
    .map({
        0: "Negative",
        1: "Positive"
    })
)

label_sensor_summary = []

for sensor, columns in all_sensor_measurements.items():

    measurement_available = (
        daily_source[columns].notna().any(axis=1)
    )

    sensor_status = daily_source[
        ["Participant_ID", "Date"]
    ].copy()

    sensor_status["Has_Measurement"] = measurement_available

    comparison = target_availability.merge(
        sensor_status,
        on=["Participant_ID", "Date"],
        how="left",
        validate="one_to_one"
    )

    summary = (
        comparison.groupby("Target_Group")
        .agg(
            Total_Observations=("Has_Measurement", "size"),
            Observations_With_Measurements=(
                "Has_Measurement", "sum"
            )
        )
        .reset_index()
    )

    summary["Sensor"] = sensor
    summary["Availability_Percentage"] = (
        summary["Observations_With_Measurements"]
        / summary["Total_Observations"] * 100
    )

    label_sensor_summary.append(summary)

label_sensor_summary = pd.concat(
    label_sensor_summary,
    ignore_index=True
)

display(
    label_sensor_summary[
        [
            "Sensor",
            "Target_Group",
            "Total_Observations",
            "Observations_With_Measurements",
            "Availability_Percentage"
        ]
    ]
)

### Step 11P: Comparison of Sensor Availability Across Target-Label Groups

**Objective**

Investigate whether sensor measurement availability differs between observations labelled positive and negative for the seven-day exacerbation prediction target.

**Methodology**

Used the known-target model dataset, containing observations with target labels of 0 or 1.

For each sensor, a participant-date was classified as having measurements if at least one of its selected measurement columns was non-missing.

Availability was calculated separately for positive and negative observations.

**Observed Results**

| Sensor | Negative availability (%) | Positive availability (%) |
|---|---:|---:|
| BRI | 7.03 | 3.94 |
| MI | 28.54 | 26.60 |
| PEI | 22.37 | 24.63 |
| SHRI | 4.42 | 2.46 |
| SQI | 4.11 | 2.96 |
| EI | 23.57 | 24.63 |
| SII | 24.71 | 25.12 |
| PRVI | 19.02 | 13.79 |

The known-target dataset contained 4,184 negative observations and 203 positive observations.

**Interpretation**

PRVI showed the largest difference in availability, with lower availability among positive observations. BRI, SHRI and SQI also had lower availability in the positive group.

PEI, EI and SII showed slightly higher availability in the positive group, while MI showed slightly lower availability.

The differences are descriptive and do not establish a relationship between missingness and exacerbation risk.

**Preprocessing Decisions**

- Preserve sensor-level availability indicators.
- Do not use these descriptive differences to remove or retain individual features.
- Evaluate missingness handling within participant-level training folds.
- Avoid interpreting repeated participant-date observations as independent patients or events.

**Limitations**

The positive group contains only 203 observations, compared with 4,184 negative observations. Observations are clustered within participants, and multiple positive rows may relate to the same documented event.

The analysis does not establish causation or clinical significance.

In [ ]:
# Step 11Q: Participant representation across target groups

participant_target_summary = (
    known_target_model_dataset
    .groupby("Participant_ID")
    .agg(
        Total_Known_Observations=(
            "exacerbation_within_7d", "size"
        ),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

participant_target_summary["Has_Positive"] = (
    participant_target_summary["Positive_Observations"] > 0
)

participant_target_summary["Has_Negative"] = (
    participant_target_summary["Negative_Observations"] > 0
)

print("PARTICIPANT REPRESENTATION")
print(
    "Participants with known labels:",
    len(participant_target_summary)
)
print(
    "Participants with positive observations:",
    participant_target_summary["Has_Positive"].sum()
)
print(
    "Participants with negative observations:",
    participant_target_summary["Has_Negative"].sum()
)
print(
    "Participants with both positive and negative observations:",
    (
        participant_target_summary["Has_Positive"]
        & participant_target_summary["Has_Negative"]
    ).sum()
)

print("\nPOSITIVE OBSERVATIONS PER PARTICIPANT")
display(
    participant_target_summary[
        participant_target_summary["Has_Positive"]
    ]
    .sort_values(
        "Positive_Observations",
        ascending=False
    )
    .head(15)
)

### Step 11Q: Participant Representation Across Target Groups

**Objective**

Examine how positive and negative target observations are distributed across participants and determine whether positive observations are concentrated among a small number of participants.

**Methodology**

Grouped the known-target model dataset by Participant_ID and calculated:

- Total known observations per participant.
- Positive observations per participant.
- Negative observations per participant.
- Whether each participant had positive and negative observations.

**Observed Results**

- Participants with known labels: 24.
- Participants with positive observations: 19.
- Participants with negative observations: 24.
- Participants with both positive and negative observations: 19.
- Participants with negative observations but no positive observations: 5.

The highest positive-observation counts included:

| Participant | Positive observations |
|---|---:|
| GER-006 | 21 |
| SPA-015 | 16 |
| GER-022 | 14 |
| SPA-040 | 14 |
| SPA-039 | 14 |
| SPA-008 | 14 |
| GER-018 | 14 |
| SPA-017 | 14 |
| SPA-012 | 14 |
| GER-003 | 9 |

The three participants with the highest positive counts contributed 51 of the 203 positive observations, approximately 25.12%.

**Interpretation**

Positive observations are unevenly distributed across participants. Nineteen participants contribute positive observations, while five participants have only negative observations in the known-target dataset.

Repeated positive participant-date observations must not be interpreted as independent exacerbation events.

**Preprocessing and Modelling Decisions**

- Preserve Participant_ID for participant-level grouping and validation.
- Keep all observations from a participant within the same validation fold.
- Do not use random row-level train-validation splitting.
- Do not automatically classify participants without positive observations as clinically event-free.
- Evaluate model performance with awareness of the small number of participants and concentration of positive observations.

**Limitations**

The analysis describes the distribution of labelled observations, not the number of independent exacerbation events. Participants without positive observations may have unknown or unrecorded clinical events outside the available labelled observation windows.

### Step 11Q: Participant Representation Across Target Groups

**Objective:**  
Examine how known positive and negative target observations are distributed across participants.

**Method:**  
- Grouped the known-label dataset by `Participant_ID`.
- Counted each participant's total known observations, positive observations, and negative observations.
- Identified participants with at least one positive observation and participants with at least one negative observation.
- Examined the participants with the highest positive-observation counts.

**Observed results:**

| Measure | Result |
|---|---:|
| Participants with known labels | 24 |
| Participants with positive observations | 19 |
| Participants with negative observations | 24 |
| Participants with both positive and negative observations | 19 |
| Participants with negative observations only | 5 |

A total of 203 positive observations are distributed across 19 participants. The nine participants with the highest positive-observation counts contribute 135 observations (66.5% of all positive observations).

**Interpretation:**
- Positive observations are concentrated among a relatively small number of participants.
- Daily observations from the same participant are not statistically independent.
- Participant-level validation is therefore important to prevent observations from the same participant appearing in both training and validation folds.
- Participants with no positive observations in the known-label dataset cannot automatically be considered clinically event-free.

**Decision:**  
Retain the participant-level validation strategy. Do not interpret the number of daily positive observations as the number of independent exacerbation events.

In [ ]:
# Step 11R: Quantify concentration of positive observations

positive_participant_counts = (
    participant_target_summary[
        participant_target_summary["Positive_Observations"] > 0
    ]
    .sort_values(
        "Positive_Observations",
        ascending=False
    )
    .reset_index(drop=True)
)

total_positive_observations = (
    positive_participant_counts["Positive_Observations"].sum()
)

positive_concentration = []

for n in [3, 5, 10, 15, 19]:

    top_n_count = (
        positive_participant_counts
        .head(n)["Positive_Observations"]
        .sum()
    )

    positive_concentration.append({
        "Top_N_Participants": n,
        "Positive_Observations": top_n_count,
        "Percentage_of_All_Positive_Observations": (
            top_n_count / total_positive_observations * 100
        )
    })

positive_concentration = pd.DataFrame(
    positive_concentration
)

print("POSITIVE OBSERVATION CONCENTRATION")
display(positive_concentration)

print("\nDISTRIBUTION ACROSS PARTICIPANTS")
display(
    positive_participant_counts[
        ["Participant_ID", "Positive_Observations"]
    ]
)

### 
Step 11R: Quantifying Positive-Observation Concentration

**Objective**

Measure how strongly positive participant-date observations are concentrated among a small number of participants.

**Methodology**

- Selected the 19 participants with at least one positive observation.
- Sorted participants by positive-observation count in descending order.
- Calculated cumulative positive-observation counts and their percentage of all 203 positive observations for the top 3, 5, 10, 15, and all 19 participants.

**Observed Results**

| Participants | Positive observations | Percentage of all positive observations |
|---|---:|---:|
| Top 3 | 51 | 25.12% |
| Top 5 | 79 | 38.92% |
| Top 10 | 144 | 70.94% |
| Top 15 | 179 | 88.18% |
| All 19 | 203 | 100.00% |

**Interpretation**

Positive observations are concentrated among a subset of participants. The top 10 of the 19 participants with positive observations contribute approximately 70.94% of all positive participant-date observations.

This concentration may influence model evaluation, particularly if performance differs between participants.

**Modelling Implications**

- Retain participant-level validation to prevent observations from the same participant appearing across training and validation folds.
- Interpret positive-class metrics with awareness of participant concentration.
- Avoid treating repeated positive dates as independent exacerbation events.
- Examine participant-level positive rates as well as raw positive counts.

**Limitations**

This is a descriptive analysis of labelled participant-date observations. It does not establish that the participants contributing more positive observations experienced more independent exacerbation events or had greater underlying clinical risk.

In [ ]:
# Check where the Fold column exists

print("known_target_model_dataset:")
print("Fold exists:", "Fold" in known_target_model_dataset.columns)
print("Shape:", known_target_model_dataset.shape)

print("\nAvailable datasets containing Fold:")

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame) and "Fold" in obj.columns:
        print(name, obj.shape)

In [ ]:
# Step 11R: Verify participant-level fold integrity

participant_fold_check = (
    model_fold_dataset
    .groupby("Participant_ID")
    .agg(
        Number_of_Folds=("Fold", "nunique"),
        Assigned_Fold=("Fold", "first")
    )
    .reset_index()
)

print("PARTICIPANT-LEVEL FOLD INTEGRITY")

print(
    "Participants appearing in multiple folds:",
    (participant_fold_check["Number_of_Folds"] > 1).sum()
)

print(
    "Participants appearing in exactly one fold:",
    (participant_fold_check["Number_of_Folds"] == 1).sum()
)

print("\nFOLD DISTRIBUTION")

display(
    model_fold_dataset.groupby("Fold").agg(
        Participants=("Participant_ID", "nunique"),
        Observations=("Participant_ID", "size"),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    ).reset_index()
)

print("\nPARTICIPANTS APPEARING IN MULTIPLE FOLDS")

display(
    participant_fold_check[
        participant_fold_check["Number_of_Folds"] > 1
    ]
)

### Step 11R: Verification of Participant-Level Fold Integrity

**Objective:**

Verify that all observations belonging to the same participant are assigned to exactly one validation fold.

**Methodology:**

- Used `model_fold_dataset`, which contains the participant-level fold assignments.
- Grouped observations by `Participant_ID`.
- Counted the number of unique folds assigned to each participant.
- Checked whether any participant appeared in multiple folds.
- Summarized participant counts and positive/negative observations across all five folds.

**Observed Results:**

Participant-level integrity:

| Measure | Result |
|---|---:|
| Participants appearing in multiple folds | 0 |
| Participants appearing in exactly one fold | 24 |
| Total folds | 5 |

Fold distribution:

| Fold | Participants | Observations | Positive observations | Negative observations |
|---|---:|---:|---:|---:|
| 1 | 5 | 1,047 | 42 | 1,005 |
| 2 | 5 | 998 | 37 | 961 |
| 3 | 5 | 796 | 49 | 747 |
| 4 | 5 | 669 | 33 | 636 |
| 5 | 4 | 877 | 42 | 835 |
| **Total** | **24** | **4,387** | **203** | **4,184** |

**Interpretation:**

- All 24 participants are assigned to exactly one fold.
- No participant appears in more than one fold.
- The participant-level validation split successfully prevents participant overlap between folds.
- Differences in observation counts across folds reflect differences in the amount of labelled monitoring data contributed by each participant.

**Decision:**

The participant-level fold assignment passes the integrity check and is retained for subsequent model evaluation.

**Limitation:**

Passing this check establishes participant separation, but does not by itself establish model performance, generalizability, or absence of all forms of temporal or feature leakage. Any learned preprocessing, feature selection, and model fitting must still be performed using training-fold data only.

In [ ]:
# Step 11S: Calculate participant-level positive rates

participant_target_summary["Positive_Rate_Percent"] = (
    participant_target_summary["Positive_Observations"]
    / participant_target_summary["Total_Known_Observations"]
    * 100
)

print("PARTICIPANT-LEVEL POSITIVE RATES")

display(
    participant_target_summary
    .sort_values(
        "Positive_Rate_Percent",
        ascending=False
    )[
        [
            "Participant_ID",
            "Total_Known_Observations",
            "Positive_Observations",
            "Negative_Observations",
            "Positive_Rate_Percent"
        ]
    ]
)

print("\nPOSITIVE RATE SUMMARY")
display(
    participant_target_summary[
        participant_target_summary["Has_Positive"]
    ]["Positive_Rate_Percent"]
    .describe()
)

### Step 11S: Participant-Level Positive Rates

**Objective**

Calculate the proportion of positive observations within each participant's known labelled observations. This distinguishes the number of positive observations contributed by a participant from the participant's positive-observation rate.

**Methodology**

For each participant, calculated:

Positive Rate (%) = (Positive Observations / Total Known Observations) × 100

Participants were sorted by positive rate in descending order. Summary statistics were calculated for the 19 participants with at least one positive observation.

**Observed Results**

- Participants with known labels: 24.
- Participants with positive observations: 19.
- Participants with zero positive observations: 5.
- Mean positive rate among the 19 participants with positive observations: 6.39%.
- Median positive rate: 5.26%.
- Standard deviation: 4.41 percentage points.
- Minimum positive rate among participants with positive observations: 2.25%.
- Maximum positive rate: 18.92%.

The highest participant-level rates were:

| Participant | Positive observations | Total known observations | Positive rate |
|---|---:|---:|---:|
| SPA-039 | 14 | 74 | 18.92% |
| SPA-007 | 7 | 56 | 12.50% |
| SPA-040 | 14 | 114 | 12.28% |
| SPA-015 | 16 | 156 | 10.26% |
| GER-006 | 21 | 255 | 8.24% |

**Interpretation**

The participant contributing the largest number of positive observations is not necessarily the participant with the highest positive rate. For example, GER-006 contributes 21 positive observations but has an 8.24

In [ ]:
# Step 11T: Compare target distribution by country

participant_target_summary["Country"] = (
    participant_target_summary["Participant_ID"]
    .str[:3]
    .map({
        "GER": "Germany",
        "SPA": "Spain"
    })
)

country_target_summary = (
    participant_target_summary
    .groupby("Country")
    .agg(
        Participants=("Participant_ID", "nunique"),
        Participants_With_Positive=("Has_Positive", "sum"),
        Total_Known_Observations=("Total_Known_Observations", "sum"),
        Positive_Observations=("Positive_Observations", "sum"),
        Negative_Observations=("Negative_Observations", "sum")
    )
    .reset_index()
)

country_target_summary["Positive_Rate_Percent"] = (
    country_target_summary["Positive_Observations"]
    / country_target_summary["Total_Known_Observations"]
    * 100
)

country_target_summary["Participants_With_Positive_Percent"] = (
    country_target_summary["Participants_With_Positive"]
    / country_target_summary["Participants"]
    * 100
)

print("TARGET DISTRIBUTION BY COUNTRY")
display(country_target_summary)

### Step 11T: Target Distribution by Country

**Objective**

Compare the distribution of positive and negative target observations between participants from Germany and Spain.

**Methodology**

Mapped participant identifiers to their corresponding country and grouped the participant-level target summary by country.

Calculated:
- Number of participants with known labels.
- Number of participants with positive observations.
- Total known observations.
- Total positive and negative observations.
- Positive-observation rate across all known observations.
- Percentage of participants with at least one positive observation.

**Observed Results**

| Metric | Germany | Spain |
|---|---:|---:|
| Participants with known labels | 10 | 14 |
| Participants with positive observations | 7 | 12 |
| Total known observations | 1,932 | 2,455 |
| Positive observations | 77 | 126 |
| Negative observations | 1,855 | 2,329 |
| Positive-observation rate | 3.99% | 5.13% |
| Participants with positive observations (%) | 70.00% | 85.71% |

**Interpretation**

Spain has a higher observed positive-observation rate (5.13%) than Germany (3.99%), a difference of approximately 1.15 percentage points. Spain also has a higher proportion of participants with at least one positive observation.

These are descriptive differences in the available labelled dataset. They do not establish a causal relationship between country and exacerbation risk.

**Modelling Implications**

- Preserve country information for appropriate stratified descriptive evaluation.
- Be cautious when comparing model performance between countries because the participant counts are small and unequal.
- Do not infer country-specific clinical risk from these raw observation-level rates.
- Continue using participant-level validation to account for repeated observations.

**Limitations**

The countries have different numbers of participants and known observations. Repeated participant-date observations are not statistically independent. Differences may reflect participant composition, observation coverage, or event recording rather than country-level effects.

In [ ]:
# Find existing variables related to participant-level folds

fold_variables = [
    name for name in globals()
    if "fold" in name.lower()
    or "split" in name.lower()
]

print("FOLD-RELATED VARIABLES:")
print(fold_variables)

In [ ]:
# Inspect the existing model fold dataset

print("SHAPE:", model_fold_dataset.shape)

print("\nCOLUMNS:")
print(model_fold_dataset.columns.tolist())

print("\nFIRST 10 ROWS:")
display(model_fold_dataset.head(10))

In [ ]:

# Recreate participant-level fold assignments

participant_folds = (
    model_fold_dataset[
        ["Participant_ID", "Fold"]
    ]
    .drop_duplicates()
    .copy()
)

print("Participant fold assignments:", participant_folds.shape)
print("Unique participants:", participant_folds["Participant_ID"].nunique())
print("Unique folds:", participant_folds["Fold"].nunique())

display(participant_folds.head())

In [ ]:
# Identify available participant and country information

print("PARTICIPANT TARGET SUMMARY")
print(participant_target_summary.columns.tolist())

print("\nDATAFRAMES CONTAINING COUNTRY")

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        country_columns = [
            col for col in obj.columns
            if "country" in str(col).lower()
        ]

        if country_columns:
            print(
                f"{name}:",
                obj.shape,
                country_columns
            )

In [ ]:
# Find dataframes containing participant identifiers

print("DATAFRAMES WITH PARTICIPANT_ID")

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if "Participant_ID" in obj.columns:
            print(
                f"\n{name}: {obj.shape}"
            )
            print(
                "Relevant columns:",
                [
                    col for col in obj.columns
                    if col in [
                        "Participant_ID",
                        "Country",
                        "Fold"
                    ]
                ]
            )

In [ ]:

# Inspect the original feature dataset columns

print("FEATURES COLUMNS:")
print(features.columns.tolist())

print("\nPARTICIPANT COUNTRY EXAMPLES:")

display(
    features[
        ["Participant_ID"]
        + [
            col for col in features.columns
            if "country" in col.lower()
        ]
    ].drop_duplicates().head(10)
)

In [ ]:
# Step 11Q: Check participant metadata sources

print("FEATURES:")
print(
    "Country column exists:",
    "Country" in features.columns
)

print("\nPARTICIPANT FOLDS:")
print(participant_folds.head())

print("\nPARTICIPANT TARGET SUMMARY:")
print(participant_target_summary.head())

print("\nCOUNTRY TARGET SUMMARY:")
print(country_target_summary.head())

In [ ]:
# Check participant-level metadata columns

print("PARTICIPANT SUMMARY:")
print(participant_summary.columns.tolist())
display(participant_summary.head())

print("\nPARTICIPANT LABELS:")
print(participant_labels.columns.tolist())
display(participant_labels.head())

In [ ]:
# Search existing DataFrames for participant-level country information

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if (
            "Participant_ID" in obj.columns
            and "Country" in obj.columns
        ):
            print(
                f"\n{name}: {obj.shape}"
            )
            display(
                obj[
                    ["Participant_ID", "Country"]
                ]
                .drop_duplicates()
                .head(10)
            )

In [ ]:
# Inspect the clinical dataset and its columns

print("Available clinical-related DataFrames:")

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if "Country" in obj.columns:
            print(
                name,
                obj.shape,
                obj.columns.tolist()
            )

In [ ]:
from pathlib import Path

clinical_path = Path("../data/processed")

print("Files in processed data folder:")

for file in clinical_path.rglob("*"):
    if file.is_file() and (
        "clinical" in file.name.lower()
        or "participant" in file.name.lower()
    ):
        print(file)

In [ ]:
from pathlib import Path

print("Current notebook working directory:")
print(Path.cwd())

print("\nFiles and folders under data/:")
data_path = Path("../data")

if data_path.exists():
    for item in sorted(data_path.iterdir()):
        print(item.name)
else:
    print("The ../data folder does not exist.")

In [ ]:
from pathlib import Path

for folder_name in ["raw", "interim"]:
    folder = Path("../data") / folder_name

    print(f"\n--- data/{folder_name} ---")

    for file in sorted(folder.rglob("*")):
        if file.is_file():
            if any(
                word in file.name.lower()
                for word in ["clinical", "participant", "patient", "index"]
            ):
                print(file)

In [ ]:
participants_info = pd.read_csv(
    "../data/interim/standardized/participants_info.csv"
)

print("Shape:", participants_info.shape)
print("\nColumns:")
print(participants_info.columns.tolist())

print("\nFirst five rows:")
display(participants_info.head())

In [ ]:
# Combine participant-level labels, country, and fold assignment

participant_country_folds = (
    participant_summary
    .merge(
        participants_info,
        on="Participant_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        participant_labels[["Participant_ID", "Fold"]],
        on="Participant_ID",
        how="left",
        validate="one_to_one"
    )
)

# Validate the mapping
print("PARTICIPANT-COUNTRY-FOLD VALIDATION")
print("=" * 45)

print("Participants:", len(participant_country_folds))
print("Missing countries:", participant_country_folds["Country"].isna().sum())
print("Missing fold assignments:", participant_country_folds["Fold"].isna().sum())
print(
    "Duplicate participants:",
    participant_country_folds["Participant_ID"].duplicated().sum()
)

# Summarize country representation within each fold
fold_country_summary = (
    participant_country_folds
    .groupby(["Fold", "Country"])
    .agg(
        Participants=("Participant_ID", "nunique"),
        Participants_With_Positive=(
            "Has_Positive_Observation", "sum"
        ),
        Total_Known_Observations=(
            "Total_Known_Observations", "sum"
        ),
        Positive_Observations=(
            "Positive_Observations", "sum"
        ),
        Negative_Observations=(
            "Negative_Observations", "sum"
        )
    )
    .reset_index()
)

fold_country_summary["Positive_Rate_Percent"] = (
    fold_country_summary["Positive_Observations"]
    / fold_country_summary["Total_Known_Observations"]
    * 100
)

print("\nCOUNTRY REPRESENTATION ACROSS FOLDS")
display(fold_country_summary)

### Step 11Q — Country Representation Across Validation Folds

**Objective**

Evaluate whether participants from Germany and Spain are represented across the five participant-level validation folds.

**Method**

1. Loaded participant-country information from `participants_info.csv`.
2. Merged country information with participant-level target summaries and fold assignments using `Participant_ID`.
3. Validated that every participant had exactly one country and one fold assignment.
4. Summarized participant counts, positive observations, negative observations, and positive observation rates by country within each fold.

**Observed Results**

- 24 participants were included.
- Missing country mappings: 0.
- Missing fold assignments: 0.
- Duplicate participant mappings: 0.
- Germany: 10 participants.
- Spain: 14 participants.

Country distribution:

| Fold | Germany | Spain | Total |
|---|---:|---:|---:|
| 1 | 4 | 1 | 5 |
| 2 | 1 | 4 | 5 |
| 3 | 3 | 2 | 5 |
| 4 | 0 | 5 | 5 |
| 5 | 2 | 2 | 4 |

Fold 4 contains only Spanish participants. All other folds contain participants from both countries.

**Interpretation**

The existing validation split preserves participant-level separation and was stratified according to whether participants had at least one positive observation. However, country representation was not used

In [ ]:
# Step 11U: Country representation across validation folds

fold_country_summary = (
    participant_target_summary
    .merge(
        participants_info[
            ["Participant_ID", "Country"]
        ],
        on="Participant_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        participant_folds[
            ["Participant_ID", "Fold"]
        ],
        on="Participant_ID",
        how="left",
        validate="one_to_one"
    )
    .groupby(["Fold", "Country"])
    .agg(
        Participants=("Participant_ID", "nunique"),
        Positive_Participants=("Has_Positive", "sum"),
        Positive_Observations=("Positive_Observations", "sum"),
        Total_Known_Observations=("Total_Known_Observations", "sum")
    )
    .reset_index()
)

print("COUNTRY REPRESENTATION ACROSS FOLDS")

display(
    fold_country_summary.pivot(
        index="Fold",
        columns="Country",
        values="Participants"
    ).fillna(0).astype(int)
)

print("\nDETAILED COUNTRY REPRESENTATION")

display(fold_country_summary)

### Step 11U — Country Representation Across Validation Folds

**Objective**

Examine the distribution of German and Spanish participants across the five participant-level validation folds.

**Method**

Merged participant-level target summaries with country metadata (`participants_info.csv`) and fold assignments using `Participant_ID`.

Calculated:
- Number of participants per country per fold.
- Number of participants with at least one positive observation.
- Total positive observations.
- Total known observations.

**Observed Results**

A total of 24 participants were included:
- Germany: 10 participants.
- Spain: 14 participants.

| Fold | Germany | Spain | Total |
|---|---:|---:|---:|
| 1 | 4 | 1 | 5 |
| 2 | 1 | 4 | 5 |
| 3 | 3 | 2 | 5 |
| 4 | 0 | 5 | 5 |
| 5 | 2 | 2 | 4 |

All participant-country mappings and fold assignments were successfully matched.

**Interpretation**

Country representation varies across folds. Fold 4 contains only Spanish participants, while the other folds contain participants from both countries.

The existing validation split was stratified by positive-observation presence, not country. Therefore, country-specific generalization cannot be assessed consistently across all folds.

**Decision**

Retain the existing five-fold participant-level split. Do not regenerate folds solely to balance country representation, given the small number of participants with known labels.

Record country imbalance as a limitation when interpreting model evaluation results.

In [ ]:
# Step 11V: Positive observation distribution across folds and countries

print("POSITIVE OBSERVATION DISTRIBUTION")
print("=" * 55)

# Positive observations by fold
fold_positive_summary = (
    fold_country_summary
    .groupby("Fold")
    .agg(
        Participants=("Participants", "sum"),
        Positive_Participants=("Positive_Participants", "sum"),
        Positive_Observations=("Positive_Observations", "sum"),
        Total_Known_Observations=("Total_Known_Observations", "sum")
    )
    .reset_index()
)

fold_positive_summary["Positive_Rate_Percent"] = (
    fold_positive_summary["Positive_Observations"]
    / fold_positive_summary["Total_Known_Observations"]
    * 100
)

fold_positive_summary["Share_of_All_Positive_Percent"] = (
    fold_positive_summary["Positive_Observations"]
    / fold_positive_summary["Positive_Observations"].sum()
    * 100
)

print("\nPOSITIVE OBSERVATIONS BY FOLD")
display(fold_positive_summary)

# Positive observations by country
country_positive_summary = (
    fold_country_summary
    .groupby("Country")
    .agg(
        Participants=("Participants", "sum"),
        Positive_Participants=("Positive_Participants", "sum"),
        Positive_Observations=("Positive_Observations", "sum"),
        Total_Known_Observations=("Total_Known_Observations", "sum")
    )
    .reset_index()
)

country_positive_summary["Positive_Rate_Percent"] = (
    country_positive_summary["Positive_Observations"]
    / country_positive_summary["Total_Known_Observations"]
    * 100
)

country_positive_summary["Share_of_All_Positive_Percent"] = (
    country_positive_summary["Positive_Observations"]
    / country_positive_summary["Positive_Observations"].sum()
    * 100
)

print("\nPOSITIVE OBSERVATIONS BY COUNTRY")
display(country_positive_summary)

### Step 11V — Positive Observation Distribution Across Folds and Countries

**Objective**

Examine the distribution of positive observations across validation folds and countries to assess the balance of the known target observations.

**Method**

Aggregated the participant-level country and fold summary to calculate:
- Number of participants.
- Number of participants with at least one positive observation.
- Total positive observations.
- Total known observations.
- Positive observation rate.
- Percentage share of all positive observations.

**Observed Results**

**Positive observations by fold:**

| Fold | Positive observations | Positive rate (%) | Share of all positives (%) |
|---|---:|---:|---:|
| 1 | 42 | 4.01 | 20.69 |
| 2 | 37 | 3.71 | 18.23 |
| 3 | 49 | 6.16 | 24.14 |
| 4 | 33 | 4.93 | 16.26 |
| 5 | 42 | 4.79 | 20.69 |

**Positive observations by country:**

| Country | Participants | Positive participants | Positive observations | Positive rate (%) |
|---|---:|---:|---:|---:|
| Germany | 10 | 7 | 77 | 3.99 |
| Spain | 14 | 12 | 126 | 5.13 |

**Interpretation**

Positive observations are represented in all five validation folds. Fold 3 has the highest positive observation rate, while Fold 2 has the lowest.

Spain accounts for approximately 62.07% of positive observations and Germany accounts for 37.93%. Spain also contributes more participants and known observations.

These differences are descriptive and do not establish country-specific clinical risk differences.

**Limitations**

- Positive observations are repeated participant-day records and are not independent exacerbation events.
- The number of participants is small.
- Country representation is not balanced across folds.
- Positive observation rates should not be interpreted as independent event incidence rates.

**Decision**

Retain the existing participant-level validation assignments. Record positive-observation and country imbalance as considerations when interpreting model evaluation results.

In [ ]:
# Step 11W: Known-label observation concentration across participants

participant_observation_summary = (
    participant_target_summary
    .sort_values(
        "Total_Known_Observations",
        ascending=False
    )
    .reset_index(drop=True)
)

participant_observation_summary["Cumulative_Observations"] = (
    participant_observation_summary["Total_Known_Observations"].cumsum()
)

participant_observation_summary["Cumulative_Share_Percent"] = (
    participant_observation_summary["Cumulative_Observations"]
    / participant_observation_summary["Total_Known_Observations"].sum()
    * 100
)

print("KNOWN-LABEL OBSERVATION CONCENTRATION")
print("=" * 55)

print(
    "Total known observations:",
    participant_observation_summary["Total_Known_Observations"].sum()
)

print(
    "Participants:",
    participant_observation_summary["Participant_ID"].nunique()
)

print("\nTOP 10 PARTICIPANTS")
display(
    participant_observation_summary[
        [
            "Participant_ID",
            "Total_Known_Observations",
            "Positive_Observations",
            "Negative_Observations",
            "Cumulative_Share_Percent"
        ]
    ].head(10)
)

print("\nCUMULATIVE CONCENTRATION")

for n in [3, 5, 10, 15, 20, 24]:
    cumulative = participant_observation_summary[
        "Total_Known_Observations"
    ].head(n).sum()

    percentage = (
        cumulative
        / participant_observation_summary["Total_Known_Observations"].sum()
        * 100
    )

    print(
        f"Top {n:2d} participants: "
        f"{cumulative:4d} observations "
        f"({percentage:.2f}%)"
    )

### Step 11W — Known-Label Observation Concentration Across Participants

**Objective**

Assess whether known target observations are evenly distributed across participants or concentrated among participants with longer labelled monitoring periods.

**Method**

1. Grouped known target observations by `Participant_ID`.
2. Counted total, positive and negative observations per participant.
3. Sorted participants by total known observations in descending order.
4. Calculated cumulative observation counts and their percentage of all known observations.

**Observed Results**

- Total known observations: 4,387.
- Participants with known labels: 24.

| Top N participants | Known observations | Cumulative share (%) |
|---|---:|---:|
| 3 | 985 | 22.45 |
| 5 | 1,539 | 35.08 |
| 10 | 2,678 | 61.04 |
| 15 | 3,534 | 80.56 |
| 20 | 4,140 | 94.37 |
| 24 | 4,387 | 100.00 |

The ten participants with the most known observations contribute approximately 61.04% of all known observations.

**Interpretation**

Known-label observations are unevenly distributed across participants. Participants with longer periods of observable labelled data contribute disproportionately to pooled observation-level evaluation metrics.

Consequently, pooled metrics may reflect the performance of participants with greater observation coverage more strongly than that of participants with shorter coverage.

**Limitations**

- Observation counts do not represent independent samples because multiple observations belong to the same participant.
- A higher observation count does not necessarily indicate greater clinical importance or higher exacerbation risk.
- The concentration analysis is descriptive.

**Decision**

Retain participant-level validation separation. During model evaluation, consider reporting both pooled observation-level metrics and participant-level performance summaries to assess whether results are dominated by a small number of participants.

In [ ]:
# Step 11X: Positive-observation concentration by participant

positive_concentration = (
    participant_target_summary
    .copy()
    .sort_values(
        "Positive_Observations",
        ascending=False
    )
    .reset_index(drop=True)
)

positive_concentration["Participant_Positive_Rate_Percent"] = (
    positive_concentration["Positive_Observations"]
    / positive_concentration["Total_Known_Observations"]
    * 100
)

positive_concentration["Cumulative_Positive_Observations"] = (
    positive_concentration["Positive_Observations"].cumsum()
)

positive_concentration["Cumulative_Positive_Share_Percent"] = (
    positive_concentration["Cumulative_Positive_Observations"]
    / positive_concentration["Positive_Observations"].sum()
    * 100
)

print("POSITIVE OBSERVATION CONCENTRATION")
print("=" * 55)

print(
    "Total positive observations:",
    positive_concentration["Positive_Observations"].sum()
)

print(
    "Participants with positive observations:",
    (positive_concentration["Positive_Observations"] > 0).sum()
)

print("\nPARTICIPANT-LEVEL POSITIVE DISTRIBUTION")

display(
    positive_concentration[
        [
            "Participant_ID",
            "Total_Known_Observations",
            "Positive_Observations",
            "Participant_Positive_Rate_Percent",
            "Cumulative_Positive_Share_Percent"
        ]
    ]
)

print("\nCUMULATIVE POSITIVE CONCENTRATION")

for n in [3, 5, 10, 15, 19]:
    cumulative = positive_concentration[
        "Positive_Observations"
    ].head(n).sum()

    percentage = (
        cumulative
        / positive_concentration["Positive_Observations"].sum()
        * 100
    )

    print(
        f"Top {n:2d} participants: "
        f"{cumulative:3d} positive observations "
        f"({percentage:.2f}%)"
    )

### Step 11X — Positive Observation Concentration Across Participants

**Objective**

Investigate how positive target observations are distributed across participants and determine whether a small number of participants contribute disproportionately to the positive class.

**Method**

1. Grouped known target observations by participant.
2. Calculated each participant's positive observation rate as positive observations divided by total known observations.
3. Sorted participants by positive observation count in descending order.
4. Calculated cumulative positive observation counts and their share of all positive observations.

**Observed Results**

- Total positive observations: 203.
- Participants with positive observations: 19.
- Participants without positive observations: 5.

| Top N participants | Positive observations | Share of all positives (%) |
|---|---:|---:|
| 3 | 51 | 25.12 |
| 5 | 79 | 38.92 |
| 10 | 144 | 70.94 |
| 15 | 179 | 88.18 |
| 19 | 203 | 100.00 |

Examples of participant-level positive rates:

| Participant | Positive observations | Known observations | Positive rate (%) |
|---|---:|---:|---:|
| SPA-039 | 14 | 74 | 18.92 |
| SPA-040 | 14 | 114 | 12.28 |
| SPA-015 | 16 | 156 | 10.26 |
| GER-006 | 21 | 255 | 8.24 |
| SPA-001 | 5 | 222 | 2.25 |

**Interpretation**

Positive observations are concentrated among a relatively small number of participants. The ten participants with the largest positive counts contribute approximately 70.94% of all positive observations.

Participant positive rates also vary considerably. A participant contributing the largest absolute number of positive observations does not necessarily have the highest positive rate.

**Limitations**

- Positive observations represent participant-day labels, not independent exacerbation events.
- Individual positive rates are based on different numbers of known observations.
- Participants without positive observations should not automatically be considered clinically event-free because unknown labels and incomplete observation periods exist.

**Decision**

Retain participant-level validation separation. Consider both pooled observation-level and participant-level evaluation metrics during model assessment. Interpret participant positive rates descriptively rather than as clinical incidence estimates.

In [ ]:
# Step 11Y: Unknown-label distribution across participants

unknown_label_summary = (
    labels
    .groupby("Participant_ID")
    .agg(
        Total_Observations=("exacerbation_within_7d", "size"),
        Unknown_Observations=(
            "exacerbation_within_7d",
            lambda x: x.isna().sum()
        ),
        Known_Observations=(
            "exacerbation_within_7d",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

unknown_label_summary["Unknown_Rate_Percent"] = (
    unknown_label_summary["Unknown_Observations"]
    / unknown_label_summary["Total_Observations"]
    * 100
)

unknown_label_summary = unknown_label_summary.sort_values(
    "Unknown_Observations",
    ascending=False
).reset_index(drop=True)

print("UNKNOWN-LABEL DISTRIBUTION")
print("=" * 55)

print(
    "Total unknown observations:",
    unknown_label_summary["Unknown_Observations"].sum()
)

print(
    "Total participants:",
    unknown_label_summary["Participant_ID"].nunique()
)

print("\nTOP 15 PARTICIPANTS BY UNKNOWN OBSERVATIONS")

display(
    unknown_label_summary.head(15)
)

print("\nOVERALL UNKNOWN-LABEL SUMMARY")

display(
    unknown_label_summary[
        [
            "Total_Observations",
            "Unknown_Observations",
            "Known_Observations",
            "Unknown_Rate_Percent"
        ]
    ].describe()
)

### Step 11Y — Unknown-Label Distribution Across Participants

**Objective**

Investigate how unknown target observations are distributed across the 74 participants.

**Method**

1. Grouped the complete label dataset by `Participant_ID`.
2. Counted total observations, unknown observations and known observations for each participant.
3. Calculated the unknown-label percentage per participant.
4. Sorted participants by the number of unknown observations.

**Observed Results**

- Total participants: 74.
- Total participant-day observations: 13,085.
- Unknown observations: 8,698.
- Known observations: 4,387.
- Overall unknown-label percentage: approximately 66.47%.

Participant-level unknown-rate summary:

| Statistic | Unknown rate (%) |
|---|---:|
| Mean | 69.10 |
| Standard deviation | 44.95 |
| Minimum | 0.89 |
| 25th percentile | 5.81 |
| Median | 100.00 |
| 75th percentile | 100.00 |
| Maximum | 100.00 |

The first 15 participants in the unknown-observation ranking had 100% unknown labels and zero known observations.

**Interpretation**

Unknown labels are highly unevenly distributed across participants. Some participants have no usable target labels despite having feature observations.

The 50 participants without known target observations cannot contribute to supervised training or labelled validation using the current target definition.

However, their feature data may still be useful for testing the data pipeline and future inference workflows.

**Limitations**

Unknown labels do not represent negative outcomes. They arise from the target-label construction rules, including unavailable event ascertainment and incomplete follow-up.

The absence of a recorded event must not be interpreted as proof that no exacerbation occurred.

**Decision**

Retain unknown labels as missing (`NaN`). Do not convert them to zero or use them as negative training examples. Preserve the complete feature dataset separately from the known-label modelling dataset.

In [ ]:
# Step 11Z: Inspect unknown-label reason information

print("LABEL DATASET COLUMNS")
print("=" * 50)

print(labels.columns.tolist())

print("\nSAMPLE UNKNOWN-LABEL RECORDS")
print("=" * 50)

display(
    labels.loc[
        labels["exacerbation_within_7d"].isna()
    ].head(10)
)

In [ ]:
# Step 11Z: Unknown-label reasons across participants

unknown_reason_summary = (
    labels[labels["exacerbation_within_7d"].isna()]
    .pivot_table(
        index="Participant_ID",
        columns="label_status",
        values="Date",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

# Ensure both reason columns exist
reason_columns = [
    "unknown_no_event_file",
    "unknown_incomplete_followup"
]

for col in reason_columns:
    if col not in unknown_reason_summary.columns:
        unknown_reason_summary[col] = 0

unknown_reason_summary["Total_Unknown"] = (
    unknown_reason_summary[reason_columns].sum(axis=1)
)

unknown_reason_summary = unknown_reason_summary.sort_values(
    "Total_Unknown",
    ascending=False
).reset_index(drop=True)

print("UNKNOWN-LABEL REASON TOTALS")
print("=" * 60)

display(
    labels.loc[
        labels["exacerbation_within_7d"].isna(),
        "label_status"
    ].value_counts().rename_axis("Reason").reset_index(name="Observations")
)

print("\nPARTICIPANT-LEVEL UNKNOWN-LABEL REASONS")
print("=" * 60)

display(unknown_reason_summary)

print("\nVALIDATION")
print("=" * 60)

print(
    "Total unknown observations:",
    unknown_reason_summary["Total_Unknown"].sum()
)

print(
    "Participant count:",
    unknown_reason_summary["Participant_ID"].nunique()
)

print(
    "Missing reason values:",
    labels.loc[
        labels["exacerbation_within_7d"].isna(),
        "label_status"
    ].isna().sum()
)

### Step 11Z — Unknown-Label Reasons Across Participants

**Objective**

Investigate why target labels are unknown and how these reasons are distributed across participants.

**Method**

Used the `label_status` column in the label dataset to distinguish:

- `unknown_no_event_file`: No event file was available for the participant.
- `unknown_incomplete_followup`: The complete seven-day prediction horizon could not be observed.

Unknown observations were grouped by participant and reason.

**Observed Results**

A total of 8,698 observations had unknown target labels.

| Unknown-label reason | Observations | Percentage |
|---|---:|---:|
| No event file | 8,540 | 98.18% |
| Incomplete follow-up | 158 | 1.82% |
| **Total** | **8,698** | **100.00%** |

Additional validation:

- All 74 participants were represented in the unknown-label summary.
- Total participant-level unknown observations: 8,698.
- Missing `label_status` values among unknown observations: 0.

The incomplete-follow-up observations were concentrated in GER-013 and SPA-001, with two observations each.

**Interpretation**

The overwhelming majority of unknown labels result from the absence of participant event files rather than incomplete observation windows.

The absence of an event file does not establish that a participant experienced no exacerbations. Consequently, these observations cannot safely be converted into negative labels.

Incomplete follow-up is a separate source of uncertainty because the full seven-day outcome window cannot be assessed.

**Decision**

Retain both categories as unknown (`NaN`). Do not convert either category into a negative target.

The distinction between missing event ascertainment and incomplete follow-up should be preserved for transparent reporting and future model evaluation.

In [ ]:
# Step 11AA: Final participant modelling eligibility summary

eligibility_summary = participant_target_summary.copy()

# Add country information
eligibility_summary = eligibility_summary.merge(
    participants_info[["Participant_ID", "Country"]],
    on="Participant_ID",
    how="left",
    validate="one_to_one"
)

# Classify participants
eligibility_summary["Modelling_Category"] = "No known labels"

eligibility_summary.loc[
    (eligibility_summary["Total_Known_Observations"] > 0)
    & (eligibility_summary["Positive_Observations"] > 0),
    "Modelling_Category"
] = "Known labels with positive observations"

eligibility_summary.loc[
    (eligibility_summary["Total_Known_Observations"] > 0)
    & (eligibility_summary["Positive_Observations"] == 0),
    "Modelling_Category"
] = "Known labels without positive observations"

# Summarise by category
eligibility_category_summary = (
    eligibility_summary
    .groupby("Modelling_Category")
    .agg(
        Participants=("Participant_ID", "nunique"),
        Known_Observations=("Total_Known_Observations", "sum"),
        Positive_Observations=("Positive_Observations", "sum"),
        Negative_Observations=("Negative_Observations", "sum")
    )
    .reset_index()
)

eligibility_category_summary["Participant_Percentage"] = (
    eligibility_category_summary["Participants"] / 74 * 100
)

print("FINAL MODELLING ELIGIBILITY")
print("=" * 65)

display(eligibility_category_summary)

print("\nPARTICIPANT-LEVEL ELIGIBILITY")
print("=" * 65)

display(
    eligibility_summary[
        [
            "Participant_ID",
            "Country",
            "Total_Known_Observations",
            "Positive_Observations",
            "Negative_Observations",
            "Modelling_Category"
        ]
    ].sort_values(
        ["Modelling_Category", "Participant_ID"]
    ).reset_index(drop=True)
)

print("\nVALIDATION")
print("=" * 65)

print(
    "Total participants:",
    eligibility_summary["Participant_ID"].nunique()
)

print(
    "Missing country values:",
    eligibility_summary["Country"].isna().sum()
)

print(
    "Total known observations:",
    eligibility_summary["Total_Known_Observations"].sum()
)

print(
    "Total positive observations:",
    eligibility_summary["Positive_Observations"].sum()
)

print(
    "Total negative observations:",
    eligibility_summary["Negative_Observations"].sum()
)

### Step 11AA — Final Modelling Eligibility Summary

**Objective**

Consolidate participant-level label availability to determine which participants can contribute to supervised model development.

**Method**

1. Started with the participant-level target summary.
2. Merged country information using `participants_info`.
3. Classified participants according to their known and positive target observations.
4. Summarised participant counts and observation counts for each category.

**Observed Results**

| Modelling category | Participants | Known observations | Positive observations | Negative observations |
|---|---:|---:|---:|---:|
| Known labels with positive observations | 19 | 3,899 | 203 | 3,696 |
| Known labels without positive observations | 5 | 488 | 0 | 488 |
| No known labels | 50 | 0 | 0 | 0 |
| **Total** | **74** | **4,387** | **203** | **4,184** |

Validation results:

- Total participants: 74.
- Participants with known labels: 24.
- Participants with positive observations: 19.
- Participants without positive observations but with known labels: 5.
- Participants without known labels: 50.
- Missing country values: 0.
- Total known observations: 4,387.

**Interpretation**

Only 24 of the 74 participants currently have observations with known target labels.

Nineteen participants contribute both positive and negative observations, while five contribute only negative observations.

The remaining 50 participants cannot contribute to supervised training or labelled validation under the current target definition.

The five participants without positive observations should not automatically be excluded from training. Their negative observations can still contribute to model fitting, although they cannot independently support positive-class performance evaluation.

**Limitations**

The positive class consists of 203 participant-day observations from 19 participants, not 203 independent clinical exacerbation events.

The relatively small number of participants with positive observations limits the reliability of generalisation estimates.

**Decision**

Retain all 24 participants with known labels for potential supervised model development. Keep the 50 participants with unknown labels separate from supervised training and labelled validation.

Continue using participant-level validation splits to prevent observations from the same participant appearing in both training and validation sets.

In [ ]:
# Step 11AB: Fold-level modelling eligibility (corrected)

fold_dataset = (
    known_target_model_dataset[
        ["Participant_ID", "exacerbation_within_7d"]
    ]
    .merge(
        participant_folds[["Participant_ID", "Fold"]],
        on="Participant_ID",
        how="left",
        validate="many_to_one"
    )
)

# Calculate whether each participant has positive observations
participant_fold_summary = (
    fold_dataset
    .groupby(["Fold", "Participant_ID"])
    .agg(
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        )
    )
    .reset_index()
)

# Summarise observations and participants by fold
fold_eligibility = (
    fold_dataset
    .groupby("Fold")
    .agg(
        Participants=("Participant_ID", "nunique"),
        Total_Observations=("exacerbation_within_7d", "size"),
        Positive_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 1).sum()
        ),
        Negative_Observations=(
            "exacerbation_within_7d",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# Add number of participants with positive observations
positive_participant_counts = (
    participant_fold_summary
    .assign(
        Has_Positive=lambda x: x["Positive_Observations"] > 0
    )
    .groupby("Fold")["Has_Positive"]
    .sum()
    .rename("Positive_Participants")
)

fold_eligibility = fold_eligibility.merge(
    positive_participant_counts,
    on="Fold",
    how="left",
    validate="one_to_one"
)

fold_eligibility["Positive_Rate_Percent"] = (
    fold_eligibility["Positive_Observations"]
    / fold_eligibility["Total_Observations"]
    * 100
)

print("FOLD-LEVEL MODELLING ELIGIBILITY")
print("=" * 65)

display(fold_eligibility)

print("\nVALIDATION")
print("=" * 65)

print("Total participants:", fold_eligibility["Participants"].sum())
print("Total observations:", fold_eligibility["Total_Observations"].sum())
print("Total positive observations:", fold_eligibility["Positive_Observations"].sum())
print("Total negative observations:", fold_eligibility["Negative_Observations"].sum())

print(
    "Folds without positive participants:",
    (fold_eligibility["Positive_Participants"] == 0).sum()
)

### Step 11AB — Fold-Level Modelling Eligibility

**Objective**

Verify that all five participant-level validation folds contain both positive and negative target observations and participants contributing positive observations.

**Method**

1. Merged the known-label modelling dataset with the existing participant-to-fold assignments.
2. Grouped observations by fold and participant.
3. Counted positive and negative observations in each fold.
4. Counted participants with at least one positive observation in each fold.
5. Validated the totals against the complete known-label dataset.

**Observed Results**

| Fold | Participants | Positive participants | Total observations | Positive observations | Negative observations |
|---|---:|---:|---:|---:|---:|
| 1 | 5 | 4 | 1,047 | 42 | 1,005 |
| 2 | 5 | 4 | 998 | 37 | 961 |
| 3 | 5 | 4 | 796 | 49 | 747 |
| 4 | 5 | 4 | 669 | 33 | 636 |
| 5 | 4 | 3 | 877 | 42 | 835 |
| **Total** | **24** | **19** | **4,387** | **203** | **4,184** |

Validation:

- All 24 participants were assigned to folds.
- All 4,387 known observations were accounted for.
- All 203 positive and 4,184 negative observations were accounted for.
- No fold lacked participants with positive observations.

**Interpretation**

All five folds contain both target classes and at least three participants with positive observations.

The existing participant-level fold assignment therefore satisfies the basic requirement for positive-class representation in every validation fold.

However, the dataset remains small and imbalanced. The presence of positive observations in every fold does not guarantee stable or reliable performance estimates.

**Decision**

Retain the existing five-fold participant-level split for preliminary model evaluation. Do not change participant assignments solely to improve fold-level positive counts.

Continue to interpret evaluation metrics cautiously because only 19 participants contribute positive observations.

In [ ]:
# Step 11AC: Final feature and target consistency checks

print("FINAL FEATURE AND TARGET CONSISTENCY")
print("=" * 65)

# 1. Target validity
valid_targets = known_target_model_dataset[
    "exacerbation_within_7d"
].isin([0, 1])

print("\n1. TARGET VALIDATION")
print("Total observations:", len(known_target_model_dataset))
print("Valid targets:", valid_targets.sum())
print("Invalid or missing targets:", (~valid_targets).sum())

# 2. Unknown-label validation
print("\n2. UNKNOWN-LABEL VALIDATION")
print(
    "Unknown targets in modelling dataset:",
    known_target_model_dataset[
        "exacerbation_within_7d"
    ].isna().sum()
)

# 3. Participant-date uniqueness
print("\n3. PARTICIPANT-DATE UNIQUENESS")

duplicate_keys = known_target_model_dataset.duplicated(
    subset=["Participant_ID", "Date"]
)

print("Duplicate participant-date rows:", duplicate_keys.sum())

# 4. Fold assignment validation
print("\n4. FOLD ASSIGNMENT VALIDATION")

final_check = known_target_model_dataset.merge(
    participant_folds[["Participant_ID", "Fold"]],
    on="Participant_ID",
    how="left",
    validate="many_to_one"
)

print("Missing fold assignments:", final_check["Fold"].isna().sum())
print("Number of folds:", final_check["Fold"].nunique())

print("\nFINAL CHECK")
print("=" * 65)

all_checks_passed = (
    valid_targets.all()
    and known_target_model_dataset[
        "exacerbation_within_7d"
    ].isna().sum() == 0
    and duplicate_keys.sum() == 0
    and final_check["Fold"].notna().all()
    and final_check["Fold"].nunique() == 5
)

print(
    "RESULT:",
    "ALL CHECKS PASSED" if all_checks_passed else "INVESTIGATION REQUIRED"
)

### Step 11AC — Final Feature and Target Consistency Checks

**Objective**

Perform final structural validation of the known-label modelling dataset before model preparation.

**Method**

Four checks were performed:

1. Verified that all target values belong to the binary classes 0 and 1.
2. Checked for missing or unknown targets.
3. Checked uniqueness of `Participant_ID` and `Date` combinations.
4. Verified that every observation has a valid participant-level fold assignment.

**Observed Results**

| Validation | Result |
|---|---:|
| Total modelling observations | 4,387 |
| Valid binary targets | 4,387 |
| Invalid or missing targets | 0 |
| Unknown targets | 0 |
| Duplicate participant-date rows | 0 |
| Missing fold assignments | 0 |
| Number of folds | 5 |

**Interpretation**

All four structural validation checks passed.

The known-label modelling dataset contains valid binary targets, unique participant-date observations, and complete fold assignments.

No unknown target observations were found in the supervised modelling dataset.

**Decision**

The dataset passes the structural consistency checks and can proceed to the remaining model-preparation steps.

These checks establish structural integrity but do not independently prove the absence of predictor leakage or guarantee model generalisation.

In [ ]:
# Step 11AD: Verify participant-level train/validation separation

fold_separation_results = []

all_participants = set(participant_folds["Participant_ID"])

for fold in sorted(participant_folds["Fold"].unique()):

    validation_participants = set(
        participant_folds.loc[
            participant_folds["Fold"] == fold,
            "Participant_ID"
        ]
    )

    training_participants = all_participants - validation_participants

    overlap = training_participants.intersection(
        validation_participants
    )

    fold_separation_results.append({
        "Fold": fold,
        "Training_Participants": len(training_participants),
        "Validation_Participants": len(validation_participants),
        "Overlapping_Participants": len(overlap),
        "Separation_Passed": len(overlap) == 0
    })

fold_separation_summary = pd.DataFrame(fold_separation_results)

print("TRAIN / VALIDATION PARTICIPANT SEPARATION")
print("=" * 65)

display(fold_separation_summary)

print("\nFINAL VALIDATION")
print("=" * 65)

print(
    "Total folds checked:",
    len(fold_separation_summary)
)

print(
    "Folds with participant overlap:",
    (fold_separation_summary["Overlapping_Participants"] > 0).sum()
)

print(
    "All folds passed:",
    fold_separation_summary["Separation_Passed"].all()
)

### Step 11AD: Verification of Participant-Level Train/Validation Separation

**Objective:** Verify that participants assigned to each validation fold are completely excluded from that fold's training set.

**Method:**
- Identified the validation participants for each fold.
- Assigned all remaining participants to the corresponding training set.
- Compared participant IDs between the two sets.
- Checked for any overlap.

**Observed Results:**

| Fold | Training Participants | Validation Participants | Overlapping Participants |
|---|---:|---:|---:|
| 1 | 19 | 5 | 0 |
| 2 | 19 | 5 | 0 |
| 3 | 19 | 5 | 0 |
| 4 | 19 | 5 | 0 |
| 5 | 20 | 4 | 0 |

- Total folds checked: 5
- Folds with participant overlap: 0
- All folds passed: True

**Interpretation:**

All five folds maintain participant-level separation. No participant contributes observations to both the training and validation sets within the same fold.

This reduces the risk of patient-level information leakage and supports a more reliable evaluation of model generalisation to unseen participants.

**Conclusion:** Participant-level train/validation separation successfully verified.

In [ ]:
# Step 11AE: Audit preprocessing state before model preparation

print("PREPROCESSING STATE AUDIT")
print("=" * 65)

# Check the modelling dataset
print("\n1. MODELLING DATASET")
print("Shape:", known_target_model_dataset.shape)

# Check missing values in candidate features
feature_columns = [
    col for col in known_target_model_dataset.columns
    if col not in [
        "Participant_ID",
        "Date",
        "exacerbation_within_7d"
    ]
]

missing_cells = known_target_model_dataset[feature_columns].isna().sum().sum()
total_cells = (
    len(known_target_model_dataset) * len(feature_columns)
)

print("Candidate features:", len(feature_columns))
print("Missing feature cells:", missing_cells)
print("Missing feature percentage:", round(
    missing_cells / total_cells * 100, 2
))

# Check whether features contain infinite values
numeric_features = known_target_model_dataset[
    feature_columns
].select_dtypes(include="number")

infinite_cells = np.isinf(
    numeric_features.to_numpy(dtype=float)
).sum()

print("\n2. INFINITE VALUES")
print("Infinite numeric feature cells:", infinite_cells)

# Check feature scaling characteristics
print("\n3. FEATURE SCALING AUDIT")

print(
    "Numeric features:",
    numeric_features.shape[1]
)

print(
    "Features with values outside [0, 1]:",
    (
        (numeric_features.min() < 0)
        | (numeric_features.max() > 1)
    ).sum()
)

print("\n4. PREPROCESSING DECISION")
print("No global imputation or scaling will be performed.")
print("Feature selection and preprocessing must be fitted")
print("independently within each training fold.")

### Step 11AE — Preprocessing State Audit

**Objective**

Inspect the current modelling dataset before applying preprocessing transformations and establish safeguards against data leakage.

**Method**

- Counted candidate predictor features.
- Measured missing predictor cells.
- Checked numeric predictors for infinite values.
- Inspected the number of numeric features and their value ranges.
- Established that imputation, scaling and feature selection must be performed independently within each training fold.

**Observed Results**

| Metric | Result |
|---|---:|
| Modelling observations | 4,387 |
| Candidate features | 1,919 |
| Missing feature cells | 4,792,307 |
| Missing feature percentage | 56.92% |
| Infinite numeric feature cells | 0 |
| Numeric features | 1,902 |
| Numeric features with values outside [0,1] | 1,769 |

**Interpretation**

The modelling dataset contains substantial missingness, affecting approximately 56.92% of candidate predictor cells.

No infinite numeric values were detected.

Most numeric features contain values outside the [0,1] interval. This is not inherently an error because the predictors represent different measurements, statistics and derived features with different scales and units.

**Decision**

No global imputation, scaling or feature selection will be applied before cross-validation.

All learned preprocessing operations must be fitted using training participants only and then applied to the corresponding validation participants.

Scaling will be selected according to the requirements of each model family.

In [ ]:
# Step 11AF: Missingness distribution in known-label modelling data

feature_columns = [
    col for col in known_target_model_dataset.columns
    if col not in [
        "Participant_ID",
        "Date",
        "exacerbation_within_7d"
    ]
]

feature_missingness = (
    known_target_model_dataset[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .rename("Missing_Percentage")
    .reset_index()
    .rename(columns={"index": "Feature"})
)

missingness_summary = pd.DataFrame({
    "Missingness_Category": [
        "Complete (0%)",
        "Low (>0–20%)",
        "Moderate (>20–50%)",
        "High (>50–80%)",
        "Very high (>80–95%)",
        "Extremely high (>95–100%)"
    ],
    "Feature_Count": [
        (feature_missingness["Missing_Percentage"] == 0).sum(),
        (
            (feature_missingness["Missing_Percentage"] > 0)
            & (feature_missingness["Missing_Percentage"] <= 20)
        ).sum(),
        (
            (feature_missingness["Missing_Percentage"] > 20)
            & (feature_missingness["Missing_Percentage"] <= 50)
        ).sum(),
        (
            (feature_missingness["Missing_Percentage"] > 50)
            & (feature_missingness["Missing_Percentage"] <= 80)
        ).sum(),
        (
            (feature_missingness["Missing_Percentage"] > 80)
            & (feature_missingness["Missing_Percentage"] <= 95)
        ).sum(),
        (
            (feature_missingness["Missing_Percentage"] > 95)
            & (feature_missingness["Missing_Percentage"] <= 100)
        ).sum()
    ]
})

print("FEATURE MISSINGNESS DISTRIBUTION")
print("=" * 65)

display(missingness_summary)

print("\nVALIDATION")
print("=" * 65)

print("Total features:", len(feature_missingness))
print("Total features across categories:", missingness_summary["Feature_Count"].sum())

### Step 11AF — Feature Missingness Distribution

**Objective**

Examine the distribution of missing values across candidate predictors in the known-label modelling dataset.

**Method**

Calculated the percentage of missing observations independently for each of the 1,919 candidate features and grouped the features into six descriptive missingness categories.

**Observed Results**

| Missingness Category | Feature Count | Percentage |
|---|---:|---:|
| Complete (0%) | 270 | 14.07% |
| Low (>0–20%) | 148 | 7.71% |
| Moderate (>20–50%) | 138 | 7.19% |
| High (>50–80%) | 922 | 48.04% |
| Very high (>80–95%) | 376 | 19.59% |
| Extremely high (>95–100%) | 65 | 3.39% |
| **Total** | **1,919** | **100%** |

Additional observations:

- 556 features have no more than 50% missingness.
- 1,363 features have more than 50% missingness.
- 441 features have more than 80% missingness.

**Interpretation**

The modelling dataset has substantial feature-level missingness. Most candidate predictors have missing values in more than half of the known-label observations.

This reflects the uneven availability of sensor measurements and derived features. High missingness alone does not establish that a feature is uninformative.

**Decision**

Do not remove features globally based on missingness calculated across the complete dataset.

Feature availability thresholds must be evaluated within training folds only. A provisional 95% missingness threshold may be investigated, but it should be assessed alongside predictive usefulness and participant-level availability.

Imputation and any missingness-indicator generation must also be fitted within training folds to prevent information leakage.

In [ ]:
# Step 11AG: Feature availability across training folds

feature_columns = [
    col for col in known_target_model_dataset.columns
    if col not in [
        "Participant_ID",
        "Date",
        "exacerbation_within_7d"
    ]
]

missingness_thresholds = [50, 70, 80, 90, 95, 99]

fold_feature_availability = []

all_participants = set(participant_folds["Participant_ID"])

for fold in sorted(participant_folds["Fold"].unique()):

    validation_participants = set(
        participant_folds.loc[
            participant_folds["Fold"] == fold,
            "Participant_ID"
        ]
    )

    training_participants = (
        all_participants - validation_participants
    )

    training_data = known_target_model_dataset[
        known_target_model_dataset["Participant_ID"].isin(
            training_participants
        )
    ]

    feature_missingness = (
        training_data[feature_columns]
        .isna()
        .mean()
        .mul(100)
    )

    result = {
        "Fold": fold,
        "Training_Participants": len(training_participants),
        "Training_Observations": len(training_data)
    }

    for threshold in missingness_thresholds:
        result[f"Features_Missing_At_Most_{threshold}pct"] = (
            feature_missingness <= threshold
        ).sum()

    fold_feature_availability.append(result)

fold_feature_availability = pd.DataFrame(
    fold_feature_availability
)

print("TRAINING-FOLD FEATURE AVAILABILITY")
print("=" * 75)

display(fold_feature_availability)

print("\nSUMMARY")
print("=" * 75)

availability_columns = [
    f"Features_Missing_At_Most_{threshold}pct"
    for threshold in missingness_thresholds
]

display(
    fold_feature_availability[availability_columns].agg(
        ["min", "mean", "max"]
    ).T
)

### Step 11AG — Feature Availability Across Training Folds

**Objective**

Evaluate the stability of feature availability across the five participant-level training folds and investigate potential missingness thresholds for feature selection.

**Method**

For each fold:

1. Identified the validation participants.
2. Excluded those participants from the training dataset.
3. Calculated missingness independently for every candidate feature using training observations only.
4. Counted features meeting six missingness thresholds: 50%, 70%, 80%, 90%, 95% and 99%.

**Observed Results**

| Missingness Threshold | Minimum Features | Mean Features | Maximum Features |
|---|---:|---:|---:|
| At most 50% | 542 | 557.4 | 576 |
| At most 70% | 857 | 914.4 | 1,053 |
| At most 80% | 1,389 | 1,480.4 | 1,550 |
| At most 90% | 1,628 | 1,670 | 1,700 |
| At most 95% | 1,842 | 1,852.2 | 1,858 |
| At most 99% | 1,912 | 1,913.4 | 1,914 |

**Interpretation**

Feature availability is relatively consistent across folds at the 95% missingness threshold, with only 16 features separating the minimum and maximum counts.

A stricter 50% threshold would retain considerably fewer features, potentially excluding useful information from sensors with limited availability.

However, retaining a feature based only on its missingness percentage does not establish that it is informative or suitable for modelling.

**Decision**

Adopt 95% missingness as a provisional training-fold feature availability threshold for further investigation.

Final feature eligibility must also consider participant-level coverage, constant and near-constant features, and the suitability of imputation.

No global feature removal is performed at this stage. All feature-selection decisions must remain confined to the corresponding training fold.

In [ ]:
# Step 11AH: Participant-level feature availability

feature_columns = [
    col for col in known_target_model_dataset.columns
    if col not in [
        "Participant_ID",
        "Date",
        "exacerbation_within_7d"
    ]
]

all_participants = set(participant_folds["Participant_ID"])

participant_availability_results = []
fold_participant_availability = []

for fold in sorted(participant_folds["Fold"].unique()):

    validation_participants = set(
        participant_folds.loc[
            participant_folds["Fold"] == fold,
            "Participant_ID"
        ]
    )

    training_participants = (
        all_participants - validation_participants
    )

    training_data = known_target_model_dataset[
        known_target_model_dataset["Participant_ID"].isin(
            training_participants
        )
    ].copy()

    # Identify features passing the provisional 95% threshold
    missingness = training_data[feature_columns].isna().mean()

    eligible_features = missingness[
        missingness <= 0.95
    ].index.tolist()

    # Determine whether each participant has any measurement
    # for each eligible feature
    participant_has_data = (
        training_data
        .groupby("Participant_ID")[eligible_features]
        .agg(lambda x: x.notna().any())
    )

    # Count participants with at least one measurement per feature
    participants_with_data = participant_has_data.sum(axis=0)

    participant_coverage = (
        participants_with_data / len(training_participants) * 100
    )

    for feature in eligible_features:
        participant_availability_results.append({
            "Fold": fold,
            "Feature": feature,
            "Participants_With_Data": participants_with_data[feature],
            "Training_Participants": len(training_participants),
            "Participant_Coverage_Percent": participant_coverage[feature]
        })

    fold_participant_availability.append({
        "Fold": fold,
        "Training_Participants": len(training_participants),
        "Eligible_Features": len(eligible_features),
        "Available_In_At_Least_50pct_Participants": (
            participant_coverage >= 50
        ).sum(),
        "Available_In_At_Least_75pct_Participants": (
            participant_coverage >= 75
        ).sum(),
        "Available_In_All_Participants": (
            participant_coverage == 100
        ).sum()
    })

participant_availability_details = pd.DataFrame(
    participant_availability_results
)

fold_participant_availability = pd.DataFrame(
    fold_participant_availability
)

print("PARTICIPANT-LEVEL FEATURE AVAILABILITY")
print("=" * 75)

display(fold_participant_availability)

print("\nPARTICIPANT COVERAGE DISTRIBUTION")
print("=" * 75)

display(
    participant_availability_details
    .groupby("Fold")["Participant_Coverage_Percent"]
    .agg(["min", "median", "mean", "max"])
    .round(2)
)

print("\nVALIDATION")
print("=" * 75)

print(
    "Total fold-feature records:",
    len(participant_availability_details)
)

print(
    "Features passing the 95% threshold in every fold:",
    participant_availability_details
    .groupby("Feature")["Fold"]
    .nunique()
    .eq(5)
    .sum()
)

### Step 11AH — Participant-Level Feature Availability

**Objective**

Investigate whether features passing the provisional 95% missingness threshold are available across a sufficiently broad range of training participants.

**Method**

For each validation fold:

1. Excluded validation participants from the training dataset.
2. Identified features with no more than 95% missingness in training observations.
3. Grouped training observations by participant.
4. Counted the number of participants with at least one non-missing measurement for each eligible feature.
5. Summarised participant coverage across features.

**Observed Results**

| Fold | Eligible Features | Available in at Least 50% of Participants | Available in at Least 75% | Available in All Participants |
|---|---:|---:|---:|---:|
| 1 | 1,858 | 1,707 | 1,417 | 381 |
| 2 | 1,854 | 1,545 | 1,509 | 656 |
| 3 | 1,853 | 1,671 | 1,488 | 381 |
| 4 | 1,854 | 1,539 | 1,501 | 381 |
| 5 | 1,842 | 1,545 | 1,489 | 381 |

Participant coverage distribution:

| Fold | Minimum | Median | Mean | Maximum |
|---|---:|---:|---:|---:|
| 1 | 36.84% | 78.95% | 80.64% | 100% |
| 2 | 31.58% | 89.47% | 82.11% | 100% |
| 3 | 36.84% | 84.21% | 80.41% | 100% |
| 4 | 31.58% | 84.21% | 81.51% | 100% |
| 5 | 30.00% | 80.00% | 77.98% | 100% |

A total of 1,841 features passed the 95% missingness threshold in all five folds.

**Interpretation**

The majority of eligible features have measurements available across a substantial proportion of training participants.

However, passing an observation-level missingness threshold does not guarantee that a feature is available for every participant.

Participant-level availability also varies across folds, reflecting uneven sensor coverage.

**Decision**

Retain the provisional 95% training-fold missingness threshold for further evaluation.

Do not introduce an additional hard participant-coverage threshold at this stage. Instead, consider participant coverage, feature variability, imputation suitability and predictive contribution during fold-specific preprocessing and model evaluation.

All eligibility and preprocessing decisions must be learned from training participants only.

In [ ]:
# Step 11AI: Final preprocessing strategy audit

feature_columns = [
    col for col in known_target_model_dataset.columns
    if col not in [
        "Participant_ID",
        "Date",
        "exacerbation_within_7d"
    ]
]

numeric_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["number"])
    .columns.tolist()
)

boolean_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["bool", "boolean"])
    .columns.tolist()
)

other_columns = [
    col for col in feature_columns
    if col not in numeric_columns + boolean_columns
]

print("FINAL PREPROCESSING AUDIT")
print("=" * 65)

print("Total candidate features:", len(feature_columns))
print("Numeric features:", len(numeric_columns))
print("Boolean features:", len(boolean_columns))
print("Other feature types:", len(other_columns))

print("\nMISSINGNESS BY FEATURE TYPE")
print("=" * 65)

print(
    "Numeric missing cells:",
    known_target_model_dataset[numeric_columns].isna().sum().sum()
)

print(
    "Boolean missing cells:",
    known_target_model_dataset[boolean_columns].isna().sum().sum()
)

print(
    "Other feature columns:",
    other_columns
)

print("\nPROPOSED PREPROCESSING")
print("=" * 65)

print("Numeric: median imputation")
print("Boolean: most-frequent imputation")
print("Missingness indicators: evaluate within training folds")
print("Constant features: remove within training folds")
print("Scaling: StandardScaler for scale-sensitive models")
print("Tree-based models: no scaling required")
print("All learned transformations: training data only")

In [ ]:
# Step 11AI: Correct predictor-column selection

non_predictor_columns = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

feature_columns = [
    col
    for col in known_target_model_dataset.columns
    if col not in non_predictor_columns
]

numeric_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["number"])
    .columns.tolist()
)

boolean_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["bool", "boolean"])
    .columns.tolist()
)

other_columns = [
    col for col in feature_columns
    if col not in numeric_columns + boolean_columns
]

print("CORRECTED PREDICTOR AUDIT")
print("=" * 65)

print("Total dataset columns:", len(known_target_model_dataset.columns))
print("Excluded non-predictor columns:", len(non_predictor_columns))
print("Candidate predictor features:", len(feature_columns))
print("Numeric features:", len(numeric_columns))
print("Boolean features:", len(boolean_columns))
print("Other feature types:", len(other_columns))

print("\nEXCLUDED COLUMNS")
print(non_predictor_columns)

print("\nUNEXPECTED FEATURE TYPES")
print(other_columns)

print("\nVALIDATION")
print("=" * 65)

print(
    "Expected predictor count (1916):",
    len(feature_columns) == 1916
)

print(
    "No label metadata included:",
    not any(col in feature_columns for col in [
        "label_status",
        "matched_event_date",
        "matched_event_severity"
    ])
)

In [ ]:
# Corrected missingness and availability audit
# Uses only the 1,916 verified predictor features.

import pandas as pd

missingness_thresholds = [50, 70, 80, 90, 95, 99]

overall_missingness = (
    known_target_model_dataset[feature_columns]
    .isna()
    .mean()
    .mul(100)
)

# 1. Overall missingness distribution
missingness_categories = {
    "Complete (0%)": overall_missingness == 0,
    "Low (>0-20%)": (
        (overall_missingness > 0) & (overall_missingness <= 20)
    ),
    "Moderate (>20-50%)": (
        (overall_missingness > 20) & (overall_missingness <= 50)
    ),
    "High (>50-80%)": (
        (overall_missingness > 50) & (overall_missingness <= 80)
    ),
    "Very high (>80-95%)": (
        (overall_missingness > 80) & (overall_missingness <= 95)
    ),
    "Extremely high (>95-100%)": (
        (overall_missingness > 95) & (overall_missingness <= 100)
    )
}

overall_missingness_summary = pd.DataFrame([
    {
        "Missingness_Category": category,
        "Feature_Count": mask.sum()
    }
    for category, mask in missingness_categories.items()
])

# 2. Fold-level and participant-level availability
all_participants = set(participant_folds["Participant_ID"])

fold_results = []
participant_results = []

for fold in sorted(participant_folds["Fold"].unique()):

    validation_participants = set(
        participant_folds.loc[
            participant_folds["Fold"] == fold,
            "Participant_ID"
        ]
    )

    training_participants = all_participants - validation_participants

    training_data = known_target_model_dataset[
        known_target_model_dataset["Participant_ID"].isin(
            training_participants
        )
    ]

    fold_missingness = (
        training_data[feature_columns].isna().mean() * 100
    )

    eligible_features = fold_missingness[
        fold_missingness <= 95
    ].index.tolist()

    fold_result = {
        "Fold": fold,
        "Training_Participants": len(training_participants),
        "Training_Observations": len(training_data),
        "Eligible_Features_95pct": len(eligible_features)
    }

    for threshold in missingness_thresholds:
        fold_result[f"At_Most_{threshold}pct"] = (
            fold_missingness <= threshold
        ).sum()

    fold_results.append(fold_result)

    # Participant-level coverage for eligible features
    participant_has_data = (
        training_data
        .groupby("Participant_ID")[eligible_features]
        .agg(lambda x: x.notna().any())
    )

    coverage = (
        participant_has_data.sum(axis=0)
        / len(training_participants) * 100
    )

    participant_results.append({
        "Fold": fold,
        "Eligible_Features": len(eligible_features),
        "Available_In_At_Least_50pct": (coverage >= 50).sum(),
        "Available_In_At_Least_75pct": (coverage >= 75).sum(),
        "Available_In_All_Participants": (coverage == 100).sum(),
        "Median_Participant_Coverage": coverage.median()
    })

fold_availability_summary = pd.DataFrame(fold_results)
participant_availability_summary = pd.DataFrame(participant_results)

print("1. CORRECTED OVERALL MISSINGNESS")
print("=" * 65)
display(overall_missingness_summary)

print("\n2. CORRECTED TRAINING-FOLD AVAILABILITY")
print("=" * 65)
display(fold_availability_summary)

print("\n3. CORRECTED PARTICIPANT-LEVEL AVAILABILITY")
print("=" * 65)
display(participant_availability_summary)

print("\n4. VALIDATION")
print("=" * 65)
print("Total predictors:", len(feature_columns))
print("Missingness categories total:",
      overall_missingness_summary["Feature_Count"].sum())
print("Fold-feature records:",
      fold_availability_summary.shape[0])
print("Participant coverage records:",
      participant_availability_summary.shape[0])

### Steps 11AF–11AH — Corrected Missingness and Participant Availability Audit

**Objective**

Reassess feature missingness and availability using only the verified 1,916 predictor features, excluding identifiers, target variables and label-related metadata.

**Method**

Three analyses were performed:

1. Calculated overall feature missingness across the known-label dataset.
2. Evaluated feature availability at different missingness thresholds separately within each training fold.
3. Measured the proportion of training participants with at least one available measurement for each feature passing the provisional 95% threshold.

All fold-specific calculations excluded validation participants.

**Observed Results**

Overall missingness:

| Category | Features |
|---|---:|
| Complete (0%) | 269 |
| Low (>0–20%) | 148 |
| Moderate (>20–50%) | 138 |
| High (>50–80%) | 922 |
| Very high (>80–95%) | 376 |
| Extremely high (>95–100%) | 63 |
| **Total** | **1,916** |

Training-fold eligibility at the 95% threshold:

| Fold | Eligible Features |
|---|---:|
| 1 | 1,857 |
| 2 | 1,853 |
| 3 | 1,852 |
| 4 | 1,853 |
| 5 | 1,841 |

Participant-level availability:

| Fold | Available in at least 50% of participants | Available in at least 75% | Available in all participants |
|---|---:|---:|---:|
| 1 | 1,706 | 1,416 | 380 |
| 2 | 1,544 | 1,508 | 655 |
| 3 | 1,670 | 1,487 | 380 |
| 4 | 1,538 | 1,500 | 380 |
| 5 | 1,544 | 1,488 | 380 |

Median participant coverage ranged from 78.95% to 89.47%.

**Interpretation**

Feature eligibility under the provisional 95% threshold was relatively consistent across folds.

However, observation-level availability does not guarantee availability across all participants. Some features are missing for particular participants despite passing the overall training-fold threshold.

**Decision**

Use the verified 1,916 predictor features as the starting feature set.

Retain 95% training-fold missingness as a provisional eligibility threshold. Do not impose an additional universal participant-coverage threshold at this stage.

All feature filtering and preprocessing must be performed using training data only.

**Correction note:** Earlier exploratory missingness and availability calculations included three label-related metadata columns: `label_status`, `matched_event_date` and `matched_event_severity`. Those columns were excluded from the corrected analysis and must not be used as model predictors.

### Step 11AJ-1 — Identify Predictor Types and Prepare a Validation Fold

**Objective**

Prepare the verified predictor columns for preprocessing and establish a participant-separated training and validation dataset.

**Method**

- Use the corrected list of 1,916 predictor features.
- Separate numeric and Boolean predictors based on their pandas data types.
- Select Fold 1 participants as the validation group.
- Use all remaining known-label participants for training.

**Important**

The validation participants must remain completely separate from the training data. All preprocessing parameters will be learned from the training observations only.

In [ ]:
# Step 11AJ-1: Prepare predictor columns and Fold 1 data

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

# 1. Identify numeric and Boolean predictors
numeric_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["number"])
    .columns.tolist()
)

boolean_columns = (
    known_target_model_dataset[feature_columns]
    .select_dtypes(include=["bool", "boolean"])
    .columns.tolist()
)

# 2. Validate predictor separation
print("PREDICTOR TYPE VALIDATION")
print("=" * 60)
print("Total predictors:", len(feature_columns))
print("Numeric predictors:", len(numeric_columns))
print("Boolean predictors:", len(boolean_columns))
print("Total identified:", len(numeric_columns) + len(boolean_columns))
print("No overlap:", set(numeric_columns).isdisjoint(boolean_columns))
print("All predictors classified:",
      len(numeric_columns) + len(boolean_columns) == len(feature_columns))

# 3. Prepare Fold 1 participant split
validation_participants = set(
    participant_folds.loc[
        participant_folds["Fold"] == 1,
        "Participant_ID"
    ]
)

all_participants = set(
    known_target_model_dataset["Participant_ID"]
)

training_participants = all_participants - validation_participants

training_data = known_target_model_dataset[
    known_target_model_dataset["Participant_ID"].isin(training_participants)
].copy()

validation_data = known_target_model_dataset[
    known_target_model_dataset["Participant_ID"].isin(validation_participants)
].copy()

# 4. Separate predictors and target
X_train = training_data[feature_columns].copy()
y_train = training_data["exacerbation_within_7d"].copy()

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["exacerbation_within_7d"].copy()

# 5. Report split details
print("\nFOLD 1 DATA SPLIT")
print("=" * 60)
print("Training participants:", training_data["Participant_ID"].nunique())
print("Validation participants:", validation_data["Participant_ID"].nunique())
print("Training observations:", len(X_train))
print("Validation observations:", len(X_validation))
print("Training target positives:", int(y_train.sum()))
print("Validation target positives:", int(y_validation.sum()))
print("Participant overlap:",
      training_participants.intersection(validation_participants))

print("\nSCIKIT-LEARN VERSION")
print(sklearn.__version__)

### Step 11AJ-2 — Construct the Preprocessing Pipeline

**Objective**

Create separate preprocessing pipelines for numeric and Boolean predictors.

**Numeric predictors**
- Impute missing values using the median calculated from the training data.
- Generate missingness indicators for features that contain missing values in the training data.

**Boolean predictors**
- Convert nullable Boolean values into numerical representations (`True` = 1, `False` = 0, missing = NaN).
- Impute missing values using the most frequent value in the training data.
- Generate missingness indicators for features that contain missing values in the training data.

**Combined pipeline**
- Combine both transformations using `ColumnTransformer`.
- Remove zero-variance features using `VarianceThreshold`, fitted only on the training data.
- Retain the option to add numeric scaling later for models that require it.

The validation data will only be transformed using parameters learned from the training data.

In [ ]:
# Step 11AJ-2: Build the preprocessing pipeline

# 1. Convert nullable Boolean predictors to numerical values.
# True -> 1, False -> 0, missing -> NaN

X_train_prepared = X_train.copy()
X_validation_prepared = X_validation.copy()

for column in boolean_columns:
    X_train_prepared[column] = (
        X_train_prepared[column].astype("float64")
    )

    X_validation_prepared[column] = (
        X_validation_prepared[column].astype("float64")
    )


# 2. Numeric preprocessing
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])


# 3. Boolean preprocessing
boolean_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])


# 4. Combine numeric and Boolean transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("boolean", boolean_pipeline, boolean_columns)
    ],
    remainder="drop",
    verbose_feature_names_out=True
)


# 5. Remove constant features after imputation
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("remove_constant", VarianceThreshold(threshold=0))
])


# 6. Fit using training data only
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train_prepared
)

# 7. Transform validation data without fitting again
X_validation_transformed = preprocessing_pipeline.transform(
    X_validation_prepared
)


# 8. Report results
print("PREPROCESSING PIPELINE RESULTS")
print("=" * 65)

print("Original numeric predictors:", len(numeric_columns))
print("Original Boolean predictors:", len(boolean_columns))
print("Original total predictors:", len(feature_columns))

print("\nTransformed training shape:", X_train_transformed.shape)
print("Transformed validation shape:", X_validation_transformed.shape)

print("\nConstant features removed:",
      len(preprocessing_pipeline
          .named_steps["preprocessor"]
          .get_feature_names_out())
      - X_train_transformed.shape[1])

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_transformed).sum())
print("Validation:", np.isnan(X_validation_transformed).sum())

print("\nInfinite values after preprocessing:")
print("Training:", np.isinf(X_train_transformed).sum())
print("Validation:", np.isinf(X_validation_transformed).sum())

print("\nTraining and validation feature counts match:",
      X_train_transformed.shape[1] == X_validation_transformed.shape[1])

### Step 11AJ-3 — Apply Training-Only Feature Availability Filtering

**Objective**

Exclude predictors with more than 95% missing values in the training data before imputation.

**Method**

- Calculate the missingness percentage of each predictor using training observations only.
- Retain features with missingness less than or equal to 95%.
- Apply the same selected feature list to the validation dataset.
- Separate the retained numeric and Boolean predictors.
- Rebuild and fit the preprocessing pipeline using only the retained training predictors.

**Leakage prevention**

The validation dataset is not used to decide which features to retain. This ensures that feature availability filtering does not use information from held-out participants.

In [ ]:
# Step 11AJ-3: Training-only missingness filtering

# 1. Calculate missingness using training data only
training_missingness = X_train.isna().mean()

# 2. Retain predictors with at most 95% missingness
missingness_threshold = 0.95

eligible_features = training_missingness[
    training_missingness <= missingness_threshold
].index.tolist()

excluded_features = [
    column for column in feature_columns
    if column not in eligible_features
]

# 3. Separate eligible numeric and Boolean predictors
eligible_numeric_columns = [
    column for column in numeric_columns
    if column in eligible_features
]

eligible_boolean_columns = [
    column for column in boolean_columns
    if column in eligible_features
]

# 4. Apply exactly the same feature selection to both datasets
X_train_eligible = X_train_prepared[eligible_features].copy()
X_validation_eligible = X_validation_prepared[eligible_features].copy()

# 5. Rebuild numeric pipeline
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])

# 6. Rebuild Boolean pipeline
boolean_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])

# 7. Combine eligible predictors
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, eligible_numeric_columns),
        ("boolean", boolean_pipeline, eligible_boolean_columns)
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# 8. Remove constant features using training data only
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("remove_constant", VarianceThreshold(threshold=0))
])

# 9. Fit on eligible training data and transform validation data
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train_eligible
)

X_validation_transformed = preprocessing_pipeline.transform(
    X_validation_eligible
)

# 10. Report results
print("TRAINING-ONLY FEATURE FILTERING")
print("=" * 65)

print("Original predictors:", len(feature_columns))
print("Eligible predictors:", len(eligible_features))
print("Excluded predictors:", len(excluded_features))

print("\nEligible numeric predictors:", len(eligible_numeric_columns))
print("Eligible Boolean predictors:", len(eligible_boolean_columns))

print("\nMissingness among excluded features:")
if excluded_features:
    print(
        (training_missingness[excluded_features] * 100)
        .describe()
        .round(2)
    )
else:
    print("No features excluded.")

print("\nFINAL PREPROCESSING OUTPUT")
print("=" * 65)

print("Training shape:", X_train_transformed.shape)
print("Validation shape:", X_validation_transformed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_transformed).sum())
print("Validation:", np.isnan(X_validation_transformed).sum())

print("\nInfinite values after preprocessing:")
print("Training:", np.isinf(X_train_transformed).sum())
print("Validation:", np.isinf(X_validation_transformed).sum())

print("\nValidation checks:")
print(
    "All eligible features have <=95% training missingness:",
    training_missingness[eligible_features].max() <= 0.95
)
print(
    "No excluded features in eligible list:",
    set(eligible_features).isdisjoint(excluded_features)
)
print(
    "Training and validation feature counts match:",
    X_train_transformed.shape[1] == X_validation_transformed.shape[1]
)

In [ ]:
# Step 11AJ-3: Training-only missingness filtering

# 1. Calculate missingness using training data only
training_missingness = X_train.isna().mean()

# 2. Retain predictors with at most 95% missingness
missingness_threshold = 0.95

eligible_features = training_missingness[
    training_missingness <= missingness_threshold
].index.tolist()

excluded_features = [
    column for column in feature_columns
    if column not in eligible_features
]

# 3. Separate eligible numeric and Boolean predictors
eligible_numeric_columns = [
    column for column in numeric_columns
    if column in eligible_features
]

eligible_boolean_columns = [
    column for column in boolean_columns
    if column in eligible_features
]

# 4. Apply exactly the same feature selection to both datasets
X_train_eligible = X_train_prepared[eligible_features].copy()
X_validation_eligible = X_validation_prepared[eligible_features].copy()

# 5. Rebuild numeric pipeline
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])

# 6. Rebuild Boolean pipeline
boolean_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent",
            add_indicator=True,
            keep_empty_features=True
        )
    )
])

# 7. Combine eligible predictors
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, eligible_numeric_columns),
        ("boolean", boolean_pipeline, eligible_boolean_columns)
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# 8. Remove constant features using training data only
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("remove_constant", VarianceThreshold(threshold=0))
])

# 9. Fit on eligible training data and transform validation data
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train_eligible
)

X_validation_transformed = preprocessing_pipeline.transform(
    X_validation_eligible
)

# 10. Report results
print("TRAINING-ONLY FEATURE FILTERING")
print("=" * 65)

print("Original predictors:", len(feature_columns))
print("Eligible predictors:", len(eligible_features))
print("Excluded predictors:", len(excluded_features))

print("\nEligible numeric predictors:", len(eligible_numeric_columns))
print("Eligible Boolean predictors:", len(eligible_boolean_columns))

print("\nMissingness among excluded features:")
if excluded_features:
    print(
        (training_missingness[excluded_features] * 100)
        .describe()
        .round(2)
    )
else:
    print("No features excluded.")

print("\nFINAL PREPROCESSING OUTPUT")
print("=" * 65)

print("Training shape:", X_train_transformed.shape)
print("Validation shape:", X_validation_transformed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_transformed).sum())
print("Validation:", np.isnan(X_validation_transformed).sum())

print("\nInfinite values after preprocessing:")
print("Training:", np.isinf(X_train_transformed).sum())
print("Validation:", np.isinf(X_validation_transformed).sum())

print("\nValidation checks:")
print(
    "All eligible features have <=95% training missingness:",
    training_missingness[eligible_features].max() <= 0.95
)
print(
    "No excluded features in eligible list:",
    set(eligible_features).isdisjoint(excluded_features)
)
print(
    "Training and validation feature counts match:",
    X_train_transformed.shape[1] == X_validation_transformed.shape[1]
)

**Observed Results**

The training-only 95% missingness filter produced the following results for Fold 1:

| Measure | Result |
|---|---:|
| Original predictors | 1,916 |
| Eligible predictors | 1,857 |
| Excluded predictors | 59 |
| Eligible numeric predictors | 1,843 |
| Eligible Boolean predictors | 14 |
| Final transformed columns | 3,429 |

The excluded features had training missingness ranging from 95.27% to 99.76%, with a mean of 96.23%.

Both training and validation datasets had zero missing and infinite values after preprocessing. Their transformed feature counts matched.

**Interpretation**

The training-only filter reduced the original predictor set by 59 features. The remaining features were imputed, with missingness indicators added where applicable. Constant features were subsequently removed using training data only.

**Decision**

Retain the 95% threshold provisionally. The threshold will be applied independently within each training fold during cross-validation.

### Step 11AJ-4 — Audit Missingness Indicators and Constant-Feature Removal

**Objective**

Inspect the fitted Fold 1 preprocessing pipeline to understand the composition of its transformed feature matrix.

**Method**

- Count the missingness indicators generated for numeric and Boolean predictors.
- Identify features removed by `VarianceThreshold`.
- Record the number of transformed features before and after constant-feature removal.
- Verify that the final feature names correspond to the transformed matrix.

All inspection uses the already fitted training-only pipeline.

In [ ]:
# Step 11AJ-4: Inspect transformed feature composition

# 1. Get feature names before constant-feature removal
preprocessed_feature_names = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# 2. Inspect fitted imputers
fitted_preprocessor = preprocessing_pipeline.named_steps["preprocessor"]

numeric_imputer = (
    fitted_preprocessor
    .named_transformers_["numeric"]
    .named_steps["imputer"]
)

boolean_imputer = (
    fitted_preprocessor
    .named_transformers_["boolean"]
    .named_steps["imputer"]
)

# 3. Count missingness indicators
numeric_indicator_count = len(numeric_imputer.indicator_.features_)
boolean_indicator_count = len(boolean_imputer.indicator_.features_)

total_indicator_count = (
    numeric_indicator_count + boolean_indicator_count
)

# 4. Identify features removed by VarianceThreshold
variance_filter = preprocessing_pipeline.named_steps["remove_constant"]

variance_support = variance_filter.get_support()

removed_constant_features = preprocessed_feature_names[
    ~variance_support
].tolist()

retained_transformed_features = preprocessed_feature_names[
    variance_support
].tolist()

# 5. Report results
print("TRANSFORMED FEATURE COMPOSITION")
print("=" * 65)

print("Eligible original predictors:", len(eligible_features))
print("Numeric missingness indicators:", numeric_indicator_count)
print("Boolean missingness indicators:", boolean_indicator_count)
print("Total missingness indicators:", total_indicator_count)

print("\nBEFORE CONSTANT-FEATURE REMOVAL")
print("=" * 65)
print("Columns:", len(preprocessed_feature_names))

print("\nCONSTANT-FEATURE REMOVAL")
print("=" * 65)
print("Constant columns removed:", len(removed_constant_features))

print("\nFINAL TRANSFORMED FEATURES")
print("=" * 65)
print("Columns:", len(retained_transformed_features))
print("Training matrix columns:", X_train_transformed.shape[1])
print("Validation matrix columns:", X_validation_transformed.shape[1])

print("\nVALIDATION")
print("=" * 65)
print(
    "Feature-name count matches training matrix:",
    len(retained_transformed_features) == X_train_transformed.shape[1]
)
print(
    "Feature-name count matches validation matrix:",
    len(retained_transformed_features) == X_validation_transformed.shape[1]
)

print("\nEXAMPLES OF REMOVED CONSTANT FEATURES")
print(removed_constant_features[:20])

**Observed Results**

The Fold 1 preprocessing pipeline generated:

| Component | Columns |
|---|---:|
| Eligible original predictors | 1,857 |
| Numeric missingness indicators | 1,574 |
| Boolean missingness indicators | 14 |
| Total before constant removal | 3,445 |
| Constant columns removed | 16 |
| Final transformed columns | 3,429 |

All 14 Boolean predictors had missing values in the training data. A total of 1,574 numeric predictors also received missingness indicators.

The 16 constant columns included observation-count features, rolling features and validity-conflict flags.

**Interpretation**

A feature that is constant within a training fold cannot distinguish between observations in that fold. Removing it reduces unnecessary dimensions.

This does not establish that the feature is universally uninformative. Its variation may differ in another training fold.

**Validation**

The final feature-name count matched both transformed matrices: 3,429 columns.

---

### Step 11AJ-5 — Create a reusable fold-specific preprocessing function

**Objective**

Convert the preprocessing procedure into a reusable function that independently prepares any participant-level fold.

The function will:

1. Identify numeric and Boolean predictors.
2. Separate training and validation participants.
3. Apply the 95% missingness threshold using training data only.
4. Impute numeric and Boolean values using training statistics.
5. Add missingness indicators.
6. Remove constant transformed columns using training data only.
7. Transform validation data using the fitted training pipeline.

This function will allow us to apply the same preprocessing logic consistently across all five folds.

In [ ]:
# Step 11AJ-5: Reusable fold-specific preprocessing function

def prepare_fold_preprocessing(
    dataset,
    participant_folds,
    feature_columns,
    fold_number,
    target_column="exacerbation_within_7d",
    missingness_threshold=0.95
):

    # 1. Identify predictor types
    numeric_columns = (
        dataset[feature_columns]
        .select_dtypes(include=["number"])
        .columns.tolist()
    )

    boolean_columns = (
        dataset[feature_columns]
        .select_dtypes(include=["bool", "boolean"])
        .columns.tolist()
    )

    # 2. Identify training and validation participants
    validation_participants = set(
        participant_folds.loc[
            participant_folds["Fold"] == fold_number,
            "Participant_ID"
        ]
    )

    all_participants = set(dataset["Participant_ID"])

    training_participants = all_participants - validation_participants

    # 3. Split observations
    training_data = dataset[
        dataset["Participant_ID"].isin(training_participants)
    ].copy()

    validation_data = dataset[
        dataset["Participant_ID"].isin(validation_participants)
    ].copy()

    X_train = training_data[feature_columns].copy()
    X_validation = validation_data[feature_columns].copy()

    y_train = training_data[target_column].copy()
    y_validation = validation_data[target_column].copy()

    # 4. Calculate missingness using training data only
    training_missingness = X_train.isna().mean()

    eligible_features = training_missingness[
        training_missingness <= missingness_threshold
    ].index.tolist()

    excluded_features = [
        column for column in feature_columns
        if column not in eligible_features
    ]

    # 5. Separate eligible predictor types
    eligible_numeric = [
        column for column in numeric_columns
        if column in eligible_features
    ]

    eligible_boolean = [
        column for column in boolean_columns
        if column in eligible_features
    ]

    # 6. Convert Boolean predictors to numerical representation
    for column in eligible_boolean:
        X_train[column] = X_train[column].astype("float64")
        X_validation[column] = X_validation[column].astype("float64")

    X_train = X_train[eligible_features]
    X_validation = X_validation[eligible_features]

    # 7. Build numeric pipeline
    numeric_pipeline = Pipeline(steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        )
    ])

    # 8. Build Boolean pipeline
    boolean_pipeline = Pipeline(steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
                keep_empty_features=True
            )
        )
    ])

    # 9. Combine transformations
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, eligible_numeric),
            ("boolean", boolean_pipeline, eligible_boolean)
        ],
        remainder="drop",
        verbose_feature_names_out=True
    )

    # 10. Remove constant features
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("remove_constant", VarianceThreshold(threshold=0))
    ])

    # 11. Fit on training data only
    X_train_transformed = pipeline.fit_transform(X_train)

    # 12. Transform validation data
    X_validation_transformed = pipeline.transform(X_validation)

    # 13. Extract transformed feature names
    all_transformed_names = (
        pipeline.named_steps["preprocessor"]
        .get_feature_names_out()
    )

    retained_mask = (
        pipeline.named_steps["remove_constant"]
        .get_support()
    )

    final_feature_names = all_transformed_names[retained_mask]

    # 14. Return all relevant outputs
    return {
        "pipeline": pipeline,
        "X_train": X_train_transformed,
        "X_validation": X_validation_transformed,
        "y_train": y_train,
        "y_validation": y_validation,
        "training_data": training_data,
        "validation_data": validation_data,
        "eligible_features": eligible_features,
        "excluded_features": excluded_features,
        "numeric_columns": eligible_numeric,
        "boolean_columns": eligible_boolean,
        "feature_names": final_feature_names,
        "training_missingness": training_missingness,
        "training_participants": training_participants,
        "validation_participants": validation_participants
    }


# 15. Test the reusable function on Fold 1
fold1_result = prepare_fold_preprocessing(
    dataset=known_target_model_dataset,
    participant_folds=participant_folds,
    feature_columns=feature_columns,
    fold_number=1
)

print("REUSABLE PIPELINE — FOLD 1")
print("=" * 65)

print("Training participants:",
      len(fold1_result["training_participants"]))

print("Validation participants:",
      len(fold1_result["validation_participants"]))

print("Eligible predictors:",
      len(fold1_result["eligible_features"]))

print("Excluded predictors:",
      len(fold1_result["excluded_features"]))

print("Transformed training shape:",
      fold1_result["X_train"].shape)

print("Transformed validation shape:",
      fold1_result["X_validation"].shape)

print("Final feature names:",
      len(fold1_result["feature_names"]))

print("\nVALIDATION")
print("=" * 65)

print("Participant overlap:",
      fold1_result["training_participants"].intersection(
          fold1_result["validation_participants"]
      ))

print("Missing values in training:",
      np.isnan(fold1_result["X_train"]).sum())

print("Missing values in validation:",
      np.isnan(fold1_result["X_validation"]).sum())

print("Infinite values in training:",
      np.isinf(fold1_result["X_train"]).sum())

print("Infinite values in validation:",
      np.isinf(fold1_result["X_validation"]).sum())

print("Feature names match training:",
      len(fold1_result["feature_names"]) ==
      fold1_result["X_train"].shape[1])

print("Feature names match validation:",
      len(fold1_result["feature_names"]) ==
      fold1_result["X_validation"].shape[1])

**Observed Results**

The reusable preprocessing function was tested on Fold 1.

| Measure | Result |
|---|---:|
| Training participants | 19 |
| Validation participants | 5 |
| Eligible predictors | 1,857 |
| Excluded predictors | 59 |
| Transformed training columns | 3,429 |
| Transformed validation columns | 3,429 |

All validation checks passed:
- No participant overlap.
- Zero missing values after preprocessing.
- Zero infinite values.
- Feature-name counts matched both transformed matrices.

**Interpretation**

The reusable function successfully reproduced the manually constructed Fold 1 preprocessing results.

**Decision**

Use this function for the remaining participant-level folds so that feature selection, imputation, missingness indicators and constant-feature removal are independently fitted within each training fold.

### Step 11AJ-6 — Apply Preprocessing Across All Five Folds

**Objective**

Apply the reusable preprocessing function independently to each of the five participant-level folds.

**Method**

For each fold:
- Use its assigned participants as validation participants.
- Use all remaining participants for training.
- Recalculate missingness and eligible features using that fold's training data.
- Fit imputation and constant-feature removal using training data only.
- Transform validation data using the fitted pipeline.

**Purpose**

Compare feature eligibility and transformed dimensions across folds and confirm that the preprocessing procedure consistently preserves participant separation and produces valid matrices.

In [ ]:
# Step 11AJ-6: Apply preprocessing to all five folds

fold_preprocessing_results = {}

fold_summary = []

for fold_number in sorted(participant_folds["Fold"].unique()):

    result = prepare_fold_preprocessing(
        dataset=known_target_model_dataset,
        participant_folds=participant_folds,
        feature_columns=feature_columns,
        fold_number=fold_number
    )

    fold_preprocessing_results[fold_number] = result

    fold_summary.append({
        "Fold": fold_number,
        "Training_Participants": len(result["training_participants"]),
        "Validation_Participants": len(result["validation_participants"]),
        "Training_Observations": result["X_train"].shape[0],
        "Validation_Observations": result["X_validation"].shape[0],
        "Eligible_Predictors": len(result["eligible_features"]),
        "Excluded_Predictors": len(result["excluded_features"]),
        "Transformed_Features": result["X_train"].shape[1],
        "Training_Positive_Rows": int(result["y_train"].sum()),
        "Validation_Positive_Rows": int(result["y_validation"].sum()),
        "Participant_Overlap": len(
            result["training_participants"].intersection(
                result["validation_participants"]
            )
        ),
        "Training_Missing_Values": int(
            np.isnan(result["X_train"]).sum()
        ),
        "Validation_Missing_Values": int(
            np.isnan(result["X_validation"]).sum()
        )
    })

fold_preprocessing_summary = pd.DataFrame(fold_summary)

print("FIVE-FOLD PREPROCESSING SUMMARY")
print("=" * 100)
display(fold_preprocessing_summary)

print("\nOVERALL VALIDATION")
print("=" * 100)

print(
    "All folds have zero participant overlap:",
    (fold_preprocessing_summary["Participant_Overlap"] == 0).all()
)

print(
    "All training matrices have zero missing values:",
    (fold_preprocessing_summary["Training_Missing_Values"] == 0).all()
)

print(
    "All validation matrices have zero missing values:",
    (fold_preprocessing_summary["Validation_Missing_Values"] == 0).all()
)

print(
    "Training and validation feature dimensions match in every fold:",
    all(
        result["X_train"].shape[1] == result["X_validation"].shape[1]
        for result in fold_preprocessing_results.values()
    )
)

print(
    "All folds produced non-empty feature matrices:",
    all(
        result["X_train"].shape[1] > 0
        for result in fold_preprocessing_results.values()
    )
)

**Observed Results**

The reusable preprocessing function was successfully applied to all five participant-level folds.

| Fold | Training Participants | Validation Participants | Eligible Predictors | Excluded Predictors | Transformed Features |
|---|---:|---:|---:|---:|---:|
| 1 | 19 | 5 | 1,857 | 59 | 3,429 |
| 2 | 19 | 5 | 1,853 | 63 | 3,420 |
| 3 | 19 | 5 | 1,852 | 64 | 3,402 |
| 4 | 19 | 5 | 1,853 | 63 | 3,421 |
| 5 | 20 | 4 | 1,841 | 75 | 3,393 |

All five folds passed the following checks:

- No participant overlap between training and validation.
- No missing values after preprocessing.
- No infinite values.
- Matching training and validation feature dimensions.
- Non-empty transformed feature matrices.

**Interpretation**

Small differences in eligible predictors and transformed dimensions are expected because each fold has a different training participant group. Missingness indicators and constant-feature removal are also learned independently within each fold.

**Decision**

Retain separate fitted preprocessing pipelines and transformed matrices for each fold. Do not concatenate transformed matrices across folds or assume their feature positions are interchangeable.



### Step 11AJ-7 — Compare Transformed Feature Sets Across Folds

**Objective**

Assess the consistency of transformed feature names across the five folds.

**Method**

- Extract the final transformed feature names from each fitted fold pipeline.
- Calculate the number of features shared by all folds.
- Calculate pairwise feature-set overlap using the Jaccard similarity coefficient.
- Identify features that are not consistently retained across all folds.

**Interpretation**

This audit helps establish whether feature availability and constant-feature removal lead to substantial differences in the feature representations used by each fold.

The transformed matrices remain fold-specific and will not be combined.

In [ ]:
# Step 11AJ-7: Compare transformed feature sets across folds

from itertools import combinations

# 1. Build a feature-name set for each fold
fold_feature_sets = {
    fold: set(result["feature_names"])
    for fold, result in fold_preprocessing_results.items()
}

# 2. Find features shared by every fold
common_features = set.intersection(
    *fold_feature_sets.values()
)

all_features_union = set.union(
    *fold_feature_sets.values()
)

# 3. Calculate pairwise Jaccard similarity
pairwise_results = []

for fold_a, fold_b in combinations(fold_feature_sets.keys(), 2):

    set_a = fold_feature_sets[fold_a]
    set_b = fold_feature_sets[fold_b]

    intersection_count = len(set_a.intersection(set_b))
    union_count = len(set_a.union(set_b))

    jaccard_similarity = intersection_count / union_count

    pairwise_results.append({
        "Fold_A": fold_a,
        "Fold_B": fold_b,
        "Shared_Features": intersection_count,
        "Combined_Unique_Features": union_count,
        "Jaccard_Similarity": jaccard_similarity
    })

pairwise_feature_similarity = pd.DataFrame(pairwise_results)

# 4. Summarize feature-set consistency
print("FEATURE SET CONSISTENCY")
print("=" * 70)

print("Features shared by all five folds:", len(common_features))
print("Total distinct features across folds:", len(all_features_union))

print(
    "Features not shared by all folds:",
    len(all_features_union - common_features)
)

print("\nPAIRWISE FEATURE OVERLAP")
print("=" * 70)
display(pairwise_feature_similarity.round(4))

print("\nOVERALL SUMMARY")
print("=" * 70)

print(
    "Mean pairwise Jaccard similarity:",
    round(pairwise_feature_similarity["Jaccard_Similarity"].mean(), 4)
)

print(
    "Minimum pairwise Jaccard similarity:",
    round(pairwise_feature_similarity["Jaccard_Similarity"].min(), 4)
)

print(
    "Maximum pairwise Jaccard similarity:",
    round(pairwise_feature_similarity["Jaccard_Similarity"].max(), 4)
)

# 5. Check whether each fold's feature names match its matrix
print("\nFEATURE NAME VALIDATION")
print("=" * 70)

for fold, result in fold_preprocessing_results.items():
    print(
        f"Fold {fold}:",
        len(result["feature_names"]) == result["X_train"].shape[1]
        == result["X_validation"].shape[1]
    )

**Observed Results**

The transformed feature sets were compared across all five folds.

| Measure | Result |
|---|---:|
| Features shared by all five folds | 3,373 |
| Total distinct features across folds | 3,429 |
| Features not shared by all folds | 56 |
| Mean pairwise Jaccard similarity | 0.9935 |
| Minimum pairwise Jaccard similarity | 0.9863 |
| Maximum pairwise Jaccard similarity | 0.9997 |

All five folds passed feature-name and matrix-dimension validation.

**Interpretation**

The transformed feature sets are highly consistent across folds. Most differences arise from training-fold-specific missingness indicators and constant-feature removal.

This indicates stability of the preprocessing representation across these splits. It does not establish predictive value, clinical relevance or generalisation to new patients.

**Decision**

Retain independent fold-specific preprocessing pipelines. Do not combine transformed matrices across folds.

---

### Step 11AJ-8 — Scaling decision

**Decision: Do not apply scaling yet.**

The current preprocessing pipeline produces imputed, finite feature matrices without scaling.

Scaling is model-dependent:

- Tree-based models generally do not require standardisation.
- Logistic regression and other scale-sensitive models may benefit from it.

If scaling is needed, it will be incorporated into the relevant model pipeline and fitted using training data only. We will avoid scaling Boolean indicators indiscriminately when preparing a numeric-only scaling strategy.